# In silico spatial perturbation in Perturb-FISH

This notebook applies a trained **SpiderNet** model to the human melanoma **Perturb-FISH** dataset to examine how 11 NF-κB-pathway perturbations alter melanoma-to-T-cell communication. It reconstructs baseline expression and meta-interaction (MI) activity, replaces control melanoma profiles with perturbed melanoma profiles, compares predicted and observed T-cell log-fold changes, and characterizes MI-17-associated ligand–receptor (LR), sender-regulator, and receiver-target programs. The final section benchmarks MI-17 against NMF-LR, COMMOT, and ScCChain using sender- and receiver-side GO-program standardized mean differences (SMDs).

**Major inputs**

- A local SpiderNet installation and the processed Perturb-FISH bundle produced by `spidernet_dataloading_MIdimselection_PerturbFISH.ipynb`.
- The trained 23-dimensional SpiderNet checkpoint and Part 0 MI loading exports.
- Optional precomputed COMMOT and ScCChain edge scores; regenerating ScCChain scores requires the external Julia workflow described in Section 16.

**Major outputs**

- Predicted and observed T-cell response tables and per-perturbation correlations.
- Melanoma-to-T-cell and T-cell-to-melanoma MI-change tables and heatmaps.
- MI-17 edge, LR, target-gene, differential-expression, enrichment, and optional LR-perturbation summaries.
- Cross-method edge-score, axis-matching, GO-program SMD, and sender/receiver benchmark outputs.

The MI-change and original-slice interpretation sections correspond most closely to the Perturb-FISH analysis in **Fig. 5c–e** and the associated Supplementary Notes. The model loaded here is a single `V1` checkpoint; leave-one-perturbation training and the Celcomen/LinearModel comparisons reported for Fig. 5b are implemented in the dedicated leave-one-out pipeline rather than here. The NMF-LR/COMMOT/ScCChain GO-program SMD benchmark in Section 16 is an additional analysis not explicitly reported in the current manuscript. Run the notebook from this directory after configuring the data and output paths below.

## 1. Imports and device setup

In [ ]:
import json
import os
from pathlib import Path

import gseapy as gp
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch

from SpiderNet.config import PathConfig, PreprocessConfig, TrainingConfig
from SpiderNet.io import load_processed_data
from SpiderNet.api import build_model

from SpiderNet.analysis import (
    sanity_check_processed,
    load_part0_outputs,
    run_insilico_spatial_perturbation,
    compute_mi_change_tables,
    select_top_features_for_mi,
    select_perturb_genes_for_mi,
    build_edge_group_masks_for_selected_genes,
    build_tcell_subset_from_edge_groups,
)
from SpiderNet.visualization import (
    apply_publication_style,
    plot_gene_correlation_barplot,
    plot_mi_change_heatmap,
    plot_group_distribution,
    plot_lr_positive_proportion_bar,
    plot_marker_tertile_boxplot,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## 2. Configure input, output, and model parameters

In [ ]:
from run_benchmarks import WORKSPACE_ROOT, DATA_ROOT, UPSTREAM_ROOT, PROCESSED_ROOT, RESULTS_DIR, OUTPUT_DIR
OUTPUT_ROOT = UPSTREAM_ROOT
PROCESSED_DATA_DIR = PROCESSED_ROOT

# Species used for the ligand–receptor databases.
SPECIES = "human"  # "mouse" or "human"

# These preprocessing parameters are recorded for reference.
# The processed bundle is expected to have already been generated by
# `spidernet_dataloading_MIdimselection_PerturbFISH`.
N_HVG = 1000
N_HVG_LR = 2000
NUM_NEIGHBORS = 10

# Training parameters
DIM_ENVIR = 23  # number of latent MI dimensions
N_JOBS = 10  # number of parallel CPU cores for initialization and training
MAX_EPOCH = 20000  # maximum number of training epochs

VERSION = "V1"

paths = PathConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    version=VERSION,
    species=SPECIES,
)

preprocess_cfg = PreprocessConfig(
    n_hvg=N_HVG,
    n_hvg_lr=N_HVG_LR,
    num_neighbors=NUM_NEIGHBORS,
)

train_cfg = TrainingConfig(
    version=VERSION,
    dim_envir=DIM_ENVIR,
    max_epoch=MAX_EPOCH,
    n_jobs=N_JOBS,
)

processed_data_dir = Path(PROCESSED_DATA_DIR)

base_run_dir = OUTPUT_DIR
base_run_dir.mkdir(parents=True, exist_ok=True)
base_model_dir = RESULTS_DIR / "Model"
base_run_dirs = {"run_dir": base_run_dir, "model_dir": base_model_dir}

required_processed_files = [
    "adata_all.h5ad",
    "adata_list.pkl",
    "SpiderNet_data_pyg_list.pkl",
    "LR_list.pkl",
    "LR_list_cellchatdb.pkl",
    "LR_meta_cellchatdb.pkl",
    "batch_cell.pkl",
    "batch_cell_unique.pkl",
    "genenames_train.pkl",
]
missing_processed_files = [
    file_name for file_name in required_processed_files
    if not (processed_data_dir / file_name).exists()
]
if missing_processed_files:
    raise FileNotFoundError(
        "The processed PerturbFISH bundle is incomplete under "
        f"{processed_data_dir}. Missing files: {missing_processed_files}"
    )

print("Processed data directory:", processed_data_dir)
print("Base run directory:", base_run_dir)
print("Model directory:", base_model_dir)


## 3. Load the processed dataset, trained model, and exported meta-interaction outputs

In [ ]:
processed = load_processed_data(processed_data_dir)

check_df, mismatch = sanity_check_processed(processed)
if mismatch.any():
    print("Potential processed-data mismatch detected:")
    print(check_df.loc[mismatch])
else:
    print("Sanity check passed.")
    print(check_df)

# Load the trained SpiderNet model.
model = build_model(
    processed=processed,
    train_cfg=train_cfg,
    device=device,
)
model.load_state_dict(
    torch.load(
        base_model_dir / f"model_epoch{train_cfg.max_epoch - 1}.pth",
        map_location=device,
    )
)
model = model.to(device)
model.eval()

# Load exported meta-interaction outputs from Part 0.
part0_outputs = load_part0_outputs(RESULTS_DIR)
loading_receiver_use_df = part0_outputs["loading_receiver_use_df"]
loading_sender_use_df = part0_outputs["loading_sender_use_df"]
loading_LR_use = part0_outputs["loading_LR_use"]
Factor_envir_use = part0_outputs["Factor_envir_use"]

print("Processed data source:", processed_data_dir)
print("Processed batches:", len(processed.spidernet_data))
print("LR pairs:", len(processed.lr_list))
print("Training genes:", len(processed.genenames_train))
print("Loading receiver matrix:", loading_receiver_use_df.shape)
print("Loading sender matrix:", loading_sender_use_df.shape)
print("Loading LR matrix:", loading_LR_use.shape)
print("Factor_envir_use:", Factor_envir_use.shape)


## 4. Define the perturbation-gene set used in the PerturbFISH analysis

In [ ]:
perturb_gene_OI = [
    "CHUK",
    "IRAK1",
    "TRAM1",
    "LBP",
    "IRAK4",
    "PELI1",
    "TAB2",
    "MAP2K2",
    "MAP2K6",
    "IRF7",
    "MYD88",
]


## 5. Run the main in silico spatial perturbation analysis

This step:
1. reconstructs the baseline expression and MI environment from the trained SpiderNet model;
2. defines T cells near vs. not near perturbed cancer cells in the observed data;
3. performs in silico replacement of nearby control cancer cells with perturbed cancer-cell profiles;
4. predicts the post-perturbation response in T cells; and
5. compares predicted T-cell log-fold change with the observed perturbation effect.


In [ ]:
core_results = run_insilico_spatial_perturbation(
    model=model,
    processed=processed,
    LR_list=processed.lr_list,
    perturb_gene_OI=perturb_gene_OI,
    device=device,
    save_dir=base_run_dir,
    random_seed=42,
)

summary_df = core_results["summary_df"]
summary_path = base_run_dir / "Part1_insilico_spatial_perturbation_summary.csv"
summary_df.to_csv(summary_path)

print(f"Saved summary to: {summary_path}")
summary_df


## 6. Visualize the per-gene prediction accuracy

In [ ]:
plot_gene_correlation_barplot(
    summary_df=summary_df,
    save_prefix=base_run_dir / "Barplot_Part1_Tcell_perturbation_correlation",
)

summary_df


## 7. Quantify MI changes before and after in-silico perturbation

In [ ]:
mi_change_pc2t, mi_change_t2pc = compute_mi_change_tables(
    Factor_envir=core_results["Factor_envir"],
    Factor_envir_dict=core_results["Factor_envir_dict"],
    perturb_gene_OI=perturb_gene_OI,
    perturbcancerTOTcell_edgerelindex_dict=core_results["perturbcancerTOTcell_edgerelindex_dict"],
    TcellTOTperturbcancer_edgerelindex_dict=core_results["TcellTOTperturbcancer_edgerelindex_dict"],
)

mi_change_pc2t.to_csv(base_run_dir / "MI_change_perturbcancerTOTcell_df.csv")
mi_change_t2pc.to_csv(base_run_dir / "MI_change_TcellTOTperturbcancer_df.csv")



In [ ]:
plot_mi_change_heatmap(
    mi_change_df=mi_change_pc2t,
    save_prefix=base_run_dir / "Heatmap_MI_change_perturbcancer_to_Tcell",
    xlabel="Perturbed gene in cancer cells",
    ylabel="Meta-interaction",
    annotate_threshold=0.4,
    figsize=(5, 6.6),
    ifshow = True,
)
mi_change_pc2t.head()


In [ ]:
# Summarize MI-17 activity changes across the 11 perturbations.
MI17_COLUMN = "MI-17"

if MI17_COLUMN not in mi_change_pc2t.columns:
    raise KeyError(f"{MI17_COLUMN} is not present in mi_change_pc2t.")

mi17_change_table = mi_change_pc2t.copy()
mi17_change_table.index = mi17_change_table.index.astype(str)
perturbation_order = [str(gene) for gene in perturb_gene_OI]
missing_perturbations = [
    gene for gene in perturbation_order if gene not in mi17_change_table.index
]
if missing_perturbations:
    print("Warning: perturbations missing from mi_change_pc2t:", missing_perturbations)

mi17_change_by_perturbation = (
    pd.to_numeric(
        mi17_change_table.reindex(perturbation_order)[MI17_COLUMN],
        errors="coerce",
    )
    .rename("MI-17_activity_change")
    .rename_axis("Perturbation")
    .reset_index()
)
valid_mi17_changes = mi17_change_by_perturbation.dropna(
    subset=["MI-17_activity_change"]
)

mean_mi17_change = valid_mi17_changes["MI-17_activity_change"].mean()
n_mi17_increased = int(
    (valid_mi17_changes["MI-17_activity_change"] > 0).sum()
)

print(
    f"Mean MI-17 activity change across "
    f"{len(valid_mi17_changes)} perturbations: {mean_mi17_change:.6f}"
)
print(
    f"Perturbations with increased MI-17 activity: "
    f"{n_mi17_increased} / {len(valid_mi17_changes)}"
)
display(mi17_change_by_perturbation)


## 8. Choose an MI of interest and inspect its top LR, receiver, and sender programs

In [ ]:
# # MIOI = "MI10"
# MIOI = "MI8"

MIOI = "MI17"
# MIOI = "MI13"

feature_results = select_top_features_for_mi(
    MIOI=MIOI,
    loading_receiver_use_df=loading_receiver_use_df,
    loading_sender_use_df=loading_sender_use_df,
    loading_LR_use=loading_LR_use,
    LR_list=processed.lr_list,
    top_n=5,
)

LR_index_top = feature_results["LR_index_top"]
LR_list_top = feature_results["LR_list_top"]
targetgene_top = feature_results["targetgene_top"]
regulatorgene_top = feature_results["regulatorgene_top"]

print("Top LR indices:", LR_index_top)
print("Top LR pairs:", LR_list_top)
print("Top target genes:", targetgene_top)
print("Top regulator genes:", regulatorgene_top)


## 9. Build perturb-edge vs. control-edge groups for the selected MI

The perturb-edge group consists of cancer→T-cell edges in which the cancer sender carries one of the MI-associated perturbations.
The control-edge group consists of cancer→T-cell edges from cancer cells without those perturbations.


In [ ]:
perturb_gene_OI_choose = perturb_gene_OI
# perturb_gene_OI_choose = select_perturb_genes_for_mi(
#     MIOI=MIOI,
#     mi_change_pc2t=mi_change_pc2t,
# )

edge_control_index_fromcancer_to_Tcell, edge_perturb_index_fromcancer_to_Tcell = build_edge_group_masks_for_selected_genes(
        adata_all=core_results["adata_all"],
        edge_index_all=core_results["edge_index_all"],
        perturb_annotation=core_results["perturb_annotation"],
        perturb_gene_OI_choose=perturb_gene_OI_choose,
    )

print("Selected perturbation genes for this MI:", perturb_gene_OI_choose)
print("Control edges:", len(edge_control_index_fromcancer_to_Tcell))
print("Perturb edges:", len(edge_perturb_index_fromcancer_to_Tcell))


## 10. Compare the MI strength between perturb edges and control edges

In [ ]:
Factor_envir_df = pd.DataFrame(
    core_results["Factor_envir"],
    columns=[f"MI{i + 1}" for i in range(core_results["Factor_envir"].shape[1])],
)

x_control = Factor_envir_df.loc[
    edge_control_index_fromcancer_to_Tcell,
    MIOI,
].to_numpy(dtype=float)

x_perturb = Factor_envir_df.loc[
    edge_perturb_index_fromcancer_to_Tcell,
    MIOI,
].to_numpy(dtype=float)

plot_group_distribution(
    values_control=x_control,
    values_perturb=x_perturb,
    y_label=f"{MIOI} strength",
    save_prefix=base_run_dir / f"Violin_{MIOI}_strength_perturb_vs_control_edge",
    plot_kind="violin",
    ylim_top=0.9 if MIOI == "MI10" else None,
    ifshow=True,
)


In [ ]:
# ============================================================
# Spatial plots for MIOI-specific melanoma/cancer -> T-cell edges
# ------------------------------------------------------------
# It draws two spatial panels:
#   1. perturbed melanoma/cancer cells -> T cells
#   2. unperturbed/control melanoma/cancer cells -> T cells
#
# Edge color = MIOI strength from core_results["Factor_envir"].
#
# Display settings:
#   - SHOW_EDGES_PER_PANEL randomly samples edges instead of selecting top edges
#   - shared MI-strength color scale across two panels
#   - smaller/thinner edges for better visibility
#   - all cells are shown in light gray
#   - the colorbar midpoint is set by COLORBAR_CENTER
# ============================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch


# -----------------------------
# Plot settings
# -----------------------------
SPATIAL_EDGE_OUTDIR = Path(base_run_dir) / f"Spatial_{MIOI}_perturb_vs_control_cancer_to_Tcell_edges"
SPATIAL_EDGE_OUTDIR.mkdir(parents=True, exist_ok=True)

# Randomly sample edges for each panel.
# Set to None if you want to draw all edges.
SHOW_EDGES_PER_PANEL = 5000
RANDOM_SEED = 123

BACKGROUND_CELL_SIZE = 0.2
CANCER_CELL_SIZE = 0.2
TCELL_CELL_SIZE = 0.2

# Smaller edges / arrowheads
EDGE_LINEWIDTH = 0.14 * 2
EDGE_ALPHA = 1.0
EDGE_MUTATION_SCALE = 2.2

FIGSIZE = (13, 6)

# All cells in light gray
CELL_GRAY_COLOR = "#D9D9D9"
BACKGROUND_COLOR = CELL_GRAY_COLOR
CANCER_COLOR = CELL_GRAY_COLOR
TCELL_COLOR = CELL_GRAY_COLOR

# Same colormap for both panels, same MI-strength scale
EDGE_CMAP = plt.get_cmap("coolwarm")

# Fixed midpoint for colorbar
# COLORBAR_CENTER = 0.1
COLORBAR_CENTER = 0.05


# -----------------------------
# Helper functions
# -----------------------------
def _as_numpy_spatial(x):
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _edge_index_to_2col_spatial(edge_index):
    edge_index = _as_numpy_spatial(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[0] == 2 and edge_index.shape[1] != 2:
        edge_index = edge_index.T

    if edge_index.shape[1] != 2:
        raise ValueError(
            f"edge_index must have two columns after conversion, got shape {edge_index.shape}"
        )

    return edge_index.astype(int, copy=False)


def _get_spatial_xy_from_adata(adata):
    if "spatial" in adata.obsm:
        spatial = np.asarray(adata.obsm["spatial"])
        if spatial.ndim == 2 and spatial.shape[1] >= 2:
            return spatial[:, :2].astype(float)

    candidate_pairs = [
        ("x", "y"),
        ("X", "Y"),
        ("x_centroid", "y_centroid"),
        ("X_centroid", "Y_centroid"),
        ("center_x", "center_y"),
        ("CenterX_global_px", "CenterY_global_px"),
        ("global_x", "global_y"),
        ("spatial_x", "spatial_y"),
    ]

    for x_col, y_col in candidate_pairs:
        if x_col in adata.obs.columns and y_col in adata.obs.columns:
            return adata.obs[[x_col, y_col]].to_numpy(dtype=float)

    raise KeyError(
        "Cannot find spatial coordinates. Expected adata.obsm['spatial'] "
        "or x/y-like columns in adata.obs."
    )


def _safe_filename_spatial(x):
    x = str(x)
    x = re.sub(r"[^\w\-.]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")


def _format_gene_list_for_title(gene_list, max_show=5):
    if isinstance(gene_list, str):
        gene_list = [gene_list]
    gene_list = list(gene_list)

    if len(gene_list) <= max_show:
        return ", ".join(map(str, gene_list))

    return ", ".join(map(str, gene_list[:max_show])) + f", ... ({len(gene_list)} genes)"


def _subset_and_random_sample_edges(
    edge_indices,
    factor_envir_df,
    mi_col,
    show_edges=None,
    random_seed=123,
):
    """
    Build edge table and randomly sample edges if needed.
    This does NOT take top edges by MI strength.
    """
    edge_indices = np.asarray(edge_indices, dtype=int)
    edge_indices = edge_indices[
        (edge_indices >= 0) & (edge_indices < factor_envir_df.shape[0])
    ]

    if len(edge_indices) == 0:
        return pd.DataFrame(columns=["edge_idx", mi_col])

    edge_df = pd.DataFrame({
        "edge_idx": edge_indices,
        mi_col: factor_envir_df.loc[edge_indices, mi_col].to_numpy(dtype=float),
    })

    edge_df = edge_df.replace([np.inf, -np.inf], np.nan)
    edge_df = edge_df.dropna(subset=[mi_col]).copy()

    if show_edges is not None and edge_df.shape[0] > show_edges:
        edge_df = edge_df.sample(
            n=show_edges,
            replace=False,
            random_state=random_seed,
        ).copy()

    # Draw lower-strength edges first and higher-strength edges later.
    # This keeps high-MI edges visually on top, but sampling itself is random.
    edge_df = edge_df.sort_values(mi_col, ascending=True).reset_index(drop=True)

    return edge_df


def _draw_edge_panel(
    ax,
    spatial,
    edge_index_all,
    edge_df,
    mi_col,
    title,
    cmap,
    norm,
    cancer_mask,
    tcell_mask,
):
    # All cells as light gray background
    ax.scatter(
        spatial[:, 0],
        spatial[:, 1],
        s=BACKGROUND_CELL_SIZE,
        c=BACKGROUND_COLOR,
        alpha=0.45,
        edgecolors="none",
        rasterized=True,
        zorder=1,
    )

    # Cancer / melanoma cells, also light gray
    ax.scatter(
        spatial[cancer_mask, 0],
        spatial[cancer_mask, 1],
        s=CANCER_CELL_SIZE,
        c=CANCER_COLOR,
        alpha=0.45,
        edgecolors="none",
        rasterized=True,
        zorder=2,
    )

    # T cells, also light gray
    ax.scatter(
        spatial[tcell_mask, 0],
        spatial[tcell_mask, 1],
        s=TCELL_CELL_SIZE,
        c=TCELL_COLOR,
        alpha=0.45,
        edgecolors="none",
        rasterized=True,
        zorder=2,
    )

    # Directional edges
    for _, row in edge_df.iterrows():
        edge_idx = int(row["edge_idx"])
        mi_value = float(row[mi_col])

        sender = int(edge_index_all[edge_idx, 0])
        receiver = int(edge_index_all[edge_idx, 1])

        x0, y0 = spatial[sender, 0], spatial[sender, 1]
        x1, y1 = spatial[receiver, 0], spatial[receiver, 1]

        arrow = FancyArrowPatch(
            (x0, y0),
            (x1, y1),
            arrowstyle="-|>",
            mutation_scale=EDGE_MUTATION_SCALE,
            linewidth=EDGE_LINEWIDTH,
            color=cmap(norm(mi_value)),
            alpha=EDGE_ALPHA,
            shrinkA=0.0,
            shrinkB=0.0,
            zorder=4,
        )
        ax.add_patch(arrow)

    ax.set_title(title, fontsize=11, pad=8)
    ax.set_aspect("equal", adjustable="box")
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)


# -----------------------------
# Prepare core data
# -----------------------------


In [ ]:
adata_all_spatial = core_results["adata_all"]
edge_index_all_spatial = _edge_index_to_2col_spatial(core_results["edge_index_all"])
spatial_xy = _get_spatial_xy_from_adata(adata_all_spatial)

if MIOI not in Factor_envir_df.columns:
    # Fallback in case MIOI is written as MI-17 but Factor_envir_df uses MI17.
    MIOI_alt = MIOI.replace("MI-", "MI")
    if MIOI_alt in Factor_envir_df.columns:
        MIOI_plot = MIOI_alt
    else:
        raise KeyError(
            f"{MIOI} not found in Factor_envir_df.columns. "
            f"Available examples: {Factor_envir_df.columns[:5].tolist()}"
        )
else:
    MIOI_plot = MIOI

if edge_index_all_spatial.shape[0] != Factor_envir_df.shape[0]:
    raise ValueError(
        f"edge_index_all rows ({edge_index_all_spatial.shape[0]}) do not match "
        f"Factor_envir rows ({Factor_envir_df.shape[0]})."
    )

if "celltype2" not in adata_all_spatial.obs.columns:
    raise KeyError("Expected adata_all.obs['celltype2'] for cancer/T-cell annotation.")

celltype2 = adata_all_spatial.obs["celltype2"].astype(str).to_numpy()
cancer_mask = celltype2 == "cancer"
tcell_mask = celltype2 == "T cells"

print("adata_all:", adata_all_spatial.shape)
print("edge_index_all:", edge_index_all_spatial.shape)
print("MIOI used for plotting:", MIOI_plot)
print("Cancer cells:", int(cancer_mask.sum()))
print("T cells:", int(tcell_mask.sum()))


# -----------------------------
# Build randomly sampled edge tables
# -----------------------------
perturb_edge_df = _subset_and_random_sample_edges(
    edge_indices=edge_perturb_index_fromcancer_to_Tcell,
    factor_envir_df=Factor_envir_df,
    mi_col=MIOI_plot,
    show_edges=SHOW_EDGES_PER_PANEL,
    random_seed=RANDOM_SEED,
)

control_edge_df = _subset_and_random_sample_edges(
    edge_indices=edge_control_index_fromcancer_to_Tcell,
    factor_envir_df=Factor_envir_df,
    mi_col=MIOI_plot,
    show_edges=SHOW_EDGES_PER_PANEL,
    random_seed=RANDOM_SEED + 1,
)

print(f"Perturb edges plotted randomly: {perturb_edge_df.shape[0]}")
print(f"Control edges plotted randomly: {control_edge_df.shape[0]}")

perturb_edge_df.to_csv(
    SPATIAL_EDGE_OUTDIR / f"{MIOI_plot}_perturbed_cancer_to_Tcell_edges_randomly_plotted.csv",
    index=False,
)

control_edge_df.to_csv(
    SPATIAL_EDGE_OUTDIR / f"{MIOI_plot}_control_cancer_to_Tcell_edges_randomly_plotted.csv",
    index=False,
)


# -----------------------------
# Shared MI-strength color scale for both panels
# Use the configured midpoint for both panels.
# -----------------------------


In [ ]:
all_plot_values = np.concatenate([
    perturb_edge_df[MIOI_plot].to_numpy(dtype=float),
    control_edge_df[MIOI_plot].to_numpy(dtype=float),
])

all_plot_values = all_plot_values[np.isfinite(all_plot_values)]

if len(all_plot_values) == 0:
    raise ValueError("No finite MIOI edge values found for spatial plotting.")

data_min = float(np.nanmin(all_plot_values))
data_max = float(np.nanmax(all_plot_values))

# Ensure TwoSlopeNorm condition: vmin < vcenter < vmax
vmin = min(0.0, data_min, COLORBAR_CENTER - 1e-6)
vmax = max(data_max, COLORBAR_CENTER + 1e-6)

edge_norm = TwoSlopeNorm(
    vmin=vmin,
    vcenter=COLORBAR_CENTER,
    vmax=vmax,
)

print(
    f"Shared MI-strength color scale: "
    f"vmin={vmin:.4f}, vcenter={COLORBAR_CENTER:.4f}, vmax={vmax:.4f}"
)


# -----------------------------
# Plot two spatial panels
# -----------------------------
plt.close("all")
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "Arial"

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
fig.patch.set_facecolor("white")

perturb_gene_title = _format_gene_list_for_title(perturb_gene_OI_choose)

_draw_edge_panel(
    ax=axes[0],
    spatial=spatial_xy,
    edge_index_all=edge_index_all_spatial,
    edge_df=perturb_edge_df,
    mi_col=MIOI_plot,
    title=(
        f"Perturbed melanoma/cancer → T cells\n"
        f"{perturb_gene_title}; random n={perturb_edge_df.shape[0]}"
    ),
    cmap=EDGE_CMAP,
    norm=edge_norm,
    cancer_mask=cancer_mask,
    tcell_mask=tcell_mask,
)

_draw_edge_panel(
    ax=axes[1],
    spatial=spatial_xy,
    edge_index_all=edge_index_all_spatial,
    edge_df=control_edge_df,
    mi_col=MIOI_plot,
    title=(
        f"Control melanoma/cancer → T cells\n"
        f"unperturbed sender cells; random n={control_edge_df.shape[0]}"
    ),
    cmap=EDGE_CMAP,
    norm=edge_norm,
    cancer_mask=cancer_mask,
    tcell_mask=tcell_mask,
)

# Shared colorbar for MIOI edge strength
sm = ScalarMappable(norm=edge_norm, cmap=EDGE_CMAP)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=axes.ravel().tolist(),
    fraction=0.025,
    pad=0.02,
)

cbar.set_label(f"{MIOI_plot} edge strength", fontsize=10)
cbar.ax.tick_params(labelsize=9)

# Add explicit ticks including midpoint
cbar.set_ticks([vmin, COLORBAR_CENTER, vmax])
cbar.set_ticklabels([
    f"{vmin:.2f}",
    f"{COLORBAR_CENTER:.2f}",
    f"{vmax:.2f}",
])

# Legend
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor=CELL_GRAY_COLOR,
        markeredgecolor="none",
        markersize=5,
        alpha=0.6,
        label="Cells",
    ),
    Line2D(
        [0], [0],
        color=EDGE_CMAP(edge_norm(max(COLORBAR_CENTER, vmax * 0.85))),
        lw=1.0,
        alpha=EDGE_ALPHA,
        label=f"{MIOI_plot} edge",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=2,
    frameon=False,
    fontsize=9,
)

fig.suptitle(
    f"Spatial distribution of {MIOI_plot} melanoma/cancer → T-cell edges",
    fontsize=13,
    y=0.99,
)

plt.tight_layout(rect=[0, 0.06, 0.94, 0.94])

save_base = SPATIAL_EDGE_OUTDIR / (
    f"Spatial_{MIOI_plot}_perturbed_vs_control_cancer_to_Tcell_edges_"
    f"random{SHOW_EDGES_PER_PANEL}_"
    f"center{COLORBAR_CENTER}_"
    f"{_safe_filename_spatial(perturb_gene_title)}"
)

fig.savefig(str(save_base) + ".png", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(str(save_base) + ".pdf", dpi=300, bbox_inches="tight", facecolor="white")

plt.show()
plt.close(fig)

print(f"Saved spatial edge plots to:\n{save_base}.png\n{save_base}.pdf")

## 11. Compare LR coexpression between perturb edges and control edges for the top LR pairs

In [ ]:
LR_index_top_use = np.array(LR_index_top)[[0,1,6,7]].tolist()
LR_coexpression = core_results["LR_coexpression"]

lr_positive_records = []

for LR_index_top_cur in LR_index_top_use:
    LR_top_cur = processed.lr_list[LR_index_top_cur]
    LR_top_cur_merge = "+".join(LR_top_cur[0]) + "->" + "+".join(LR_top_cur[1])
    LR_top_cur_merge1 = "+".join(LR_top_cur[0]) + "-" + "+".join(LR_top_cur[1])

    x_control = LR_coexpression[edge_control_index_fromcancer_to_Tcell, LR_index_top_cur].ravel()
    x_perturb = LR_coexpression[edge_perturb_index_fromcancer_to_Tcell, LR_index_top_cur].ravel()

    x_control = x_control[np.isfinite(x_control)]
    x_perturb = x_perturb[np.isfinite(x_perturb)]

    lr_positive_records.append(
        {
            "LR_pair": LR_top_cur_merge,
            "Control_edge_positive_prop": np.mean(x_control > 0),
            "Perturb_edge_positive_prop": np.mean(x_perturb > 0),
        }
    )

    plot_group_distribution(
        values_control=x_control,
        values_perturb=x_perturb,
        y_label=f"{LR_top_cur_merge} strength",
        save_prefix=base_run_dir / f"Boxplot_{MIOI}_{LR_top_cur_merge1}_strength_perturb_vs_control_edge",
        plot_kind="box",
        ylim_top=None,
    )

LR_positive_prop = pd.DataFrame(lr_positive_records)
LR_positive_prop.to_csv(base_run_dir / f"LR_positive_proportion_{MIOI}.csv", index=False)

plot_lr_positive_proportion_bar(
    df=LR_positive_prop,
    save_prefix=base_run_dir / f"Barplot_LR_positive_proportion_{MIOI}",
    ifshow=True,
)

LR_positive_prop


## 12. Compare target-gene expression between perturbation-edge and control-edge T cells

In [ ]:
adata_all_subset = build_tcell_subset_from_edge_groups(
    adata_all=core_results["adata_all"],
    edge_index_all=core_results["edge_index_all"],
    edge_control_index_fromcancer_to_Tcell=edge_control_index_fromcancer_to_Tcell,
    edge_perturb_index_fromcancer_to_Tcell=edge_perturb_index_fromcancer_to_Tcell,
)

adata_all_subset


In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


def _looks_like_counts(X, sample_n=2000):
    if sp.issparse(X):
        vals = X.data
        if vals.size == 0:
            return False
        vals = vals[: min(vals.size, sample_n)]
    else:
        X = np.asarray(X)
        flat = X.ravel()
        if flat.size == 0:
            return False
        vals = flat[np.random.choice(flat.size, size=min(sample_n, flat.size), replace=False)]

    nonneg = np.all(vals >= 0)
    near_int = np.mean(np.isclose(vals, np.round(vals)))
    vmax = float(np.max(vals)) if vals.size > 0 else 0.0
    return nonneg and near_int > 0.9 and vmax > 20


def prepare_target_gene_group_expression_analysis(
    adata,
    genes,
    group_key="Tcell_group",
    group_order=("InPerturb_edge", "InControl_edge"),
    layer_key="log1p_norm",
    alternative="greater",   # test whether InPerturb_edge > InControl_edge
    stats_csv_path=None,
    palette=None,
):
    if palette is None:
        palette = {
            "InControl_edge": "#F0F3FF",
            "InPerturb_edge": "#836FFF",
        }

    if group_key not in adata.obs.columns:
        raise KeyError(f"'{group_key}' not found in adata.obs")

    adata_use = adata[adata.obs[group_key].isin(group_order)].copy()
    adata_use.obs[group_key] = adata_use.obs[group_key].astype(str)

    present_groups = adata_use.obs[group_key].unique().tolist()
    missing_groups = [g for g in group_order if g not in present_groups]
    if len(missing_groups) > 0:
        raise ValueError(
            f"Missing groups in adata.obs['{group_key}']: {missing_groups}. "
            f"Present groups: {present_groups}"
        )

    genes_in = [g for g in genes if g in adata_use.var_names]
    genes_missing = [g for g in genes if g not in adata_use.var_names]

    if len(genes_in) == 0:
        raise ValueError("None of the requested genes are found in adata.var_names")

    # create layer if needed
    if layer_key not in adata_use.layers:
        if _looks_like_counts(adata_use.X):
            tmp = adata_use.copy()
            sc.pp.normalize_total(tmp, target_sum=1e4)
            sc.pp.log1p(tmp)
            adata_use.layers[layer_key] = tmp.X.copy()
            del tmp
        else:
            adata_use.layers[layer_key] = adata_use.X.copy()

    expr_df = sc.get.obs_df(adata_use, keys=genes_in, layer=layer_key)
    expr_df[group_key] = adata_use.obs[group_key].values

    pert = group_order[0]
    ctrl = group_order[1]

    records = []
    for gene in genes_in:
        x_pert = expr_df.loc[expr_df[group_key] == pert, gene].to_numpy(dtype=float)
        x_ctrl = expr_df.loc[expr_df[group_key] == ctrl, gene].to_numpy(dtype=float)

        x_pert = x_pert[np.isfinite(x_pert)]
        x_ctrl = x_ctrl[np.isfinite(x_ctrl)]

        if x_pert.size < 2 or x_ctrl.size < 2:
            stat, p = np.nan, np.nan
        else:
            stat, p = mannwhitneyu(x_pert, x_ctrl, alternative=alternative)

        records.append(
            {
                "gene": gene,
                "group_perturb": pert,
                "group_control": ctrl,
                "n_perturb": x_pert.size,
                "n_control": x_ctrl.size,
                "U": stat,
                "p_one_sided": p,
                "median_perturb": np.nanmedian(x_pert) if x_pert.size else np.nan,
                "median_control": np.nanmedian(x_ctrl) if x_ctrl.size else np.nan,
                "median_diff_perturb_minus_control": (
                    np.nanmedian(x_pert) - np.nanmedian(x_ctrl)
                    if (x_pert.size and x_ctrl.size) else np.nan
                ),
            }
        )

    stats_df = pd.DataFrame(records)

    valid = stats_df["p_one_sided"].notna().values
    q = np.full(stats_df.shape[0], np.nan, dtype=float)
    if valid.sum() > 0:
        q[valid] = multipletests(stats_df.loc[valid, "p_one_sided"].values, method="fdr_bh")[1]
    stats_df["q_fdr_bh"] = q

    if stats_csv_path is not None:
        stats_csv_path = Path(stats_csv_path)
        stats_csv_path.parent.mkdir(parents=True, exist_ok=True)
        stats_df.to_csv(stats_csv_path, index=False)

    return {
        "adata_use": adata_use,
        "expr_df": expr_df,
        "stats_df": stats_df,
        "genes_in": genes_in,
        "genes_missing": genes_missing,
        "group_order": list(group_order),
        "palette": palette,
        "layer_key": layer_key,
    }


def _p_to_star(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def plot_target_gene_group_violins(
    expr_df,
    genes,
    group_key,
    group_order,
    palette,
    stats_df,
    save_prefix,
    genes_per_page=12,
    ncols=3,
    figsize_per_ax=(1.9, 2.9),
    violin_inner=None,
    violin_cut=0,
    violin_scale="width",
    show=True,
):
    genes = list(genes)
    if len(genes) == 0:
        return

    stat_map_p = dict(zip(stats_df["gene"], stats_df["p_one_sided"]))
    stat_map_q = dict(zip(stats_df["gene"], stats_df["q_fdr_bh"]))

    group_label_map = {
        "InPerturb_edge": "Perturb edge",
        "InControl_edge": "Control edge",
    }
    
    display_order = [group_label_map.get(g, g) for g in group_order]

    expr_plot = expr_df.copy()
    expr_plot[group_key] = expr_plot[group_key].map(group_label_map).fillna(expr_plot[group_key])

    n_pages = math.ceil(len(genes) / genes_per_page)

    for page in range(n_pages):
        genes_page = genes[page * genes_per_page : (page + 1) * genes_per_page]
        nrows = math.ceil(len(genes_page) / ncols)

        fig, axes = plt.subplots(
            nrows=nrows,
            ncols=ncols,
            figsize=(figsize_per_ax[0] * ncols, figsize_per_ax[1] * nrows),
            squeeze=False,
        )
        axes = axes.flatten()

        for ax_i, gene in enumerate(genes_page):
            ax = axes[ax_i]
            df_gene = expr_plot[[gene, group_key]].copy()
            df_gene.columns = ["expression", "group"]

            sns.violinplot(
                data=df_gene,
                x="group",
                y="expression",
                order=display_order,
                palette=[palette[group_order[0]], palette[group_order[1]]],
                inner=violin_inner,
                cut=violin_cut,
                scale=violin_scale,
                linewidth=1,
                ax=ax,
            )

            medians = (
                df_gene.groupby("group")["expression"]
                .median()
                .reindex(display_order)
            )
            for j, med in enumerate(medians):
                if pd.notna(med):
                    ax.hlines(
                        y=med,
                        xmin=j - 0.18,
                        xmax=j + 0.18,
                        colors="black",
                        linewidth=1.2,
                        zorder=5,
                    )

            p = stat_map_p.get(gene, np.nan)
            q = stat_map_q.get(gene, np.nan)

            ymax = np.nanmax(df_gene["expression"].values) if len(df_gene) else 1.0
            ymin = np.nanmin(df_gene["expression"].values) if len(df_gene) else 0.0
            yrng = max(ymax - ymin, 1e-6)

            line_y = ymax + 0.10 * yrng
            text_y = ymax + 0.16 * yrng

            ax.plot([0, 0, 1, 1], [line_y, line_y + 0.02 * yrng, line_y + 0.02 * yrng, line_y],
                    lw=1.0, c="black")
            ax.text(
                0.5,
                text_y,
                f"{_p_to_star(q)}\nq={q:.2e}" if pd.notna(q) else "NA",
                ha="center",
                va="bottom",
                fontsize=8,
            )

            ax.set_title(gene, fontsize=10, pad=6)
            ax.set_xlabel("")
            ax.set_ylabel("log1p normalized expression", fontsize=8)
            ax.tick_params(axis="x", labelrotation=20, labelsize=8)
            ax.tick_params(axis="y", labelsize=8)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        for j in range(len(genes_page), len(axes)):
            axes[j].axis("off")

        plt.tight_layout()

        save_prefix = Path(save_prefix)
        save_prefix.parent.mkdir(parents=True, exist_ok=True)

        if n_pages == 1:
            png_path = save_prefix.with_suffix(".png")
            pdf_path = save_prefix.with_suffix(".pdf")
        else:
            png_path = save_prefix.parent / f"{save_prefix.name}_page{page + 1}.png"
            pdf_path = save_prefix.parent / f"{save_prefix.name}_page{page + 1}.pdf"

        plt.savefig(png_path, dpi=300, bbox_inches="tight")
        plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
        if show:
            plt.show()
        plt.close()

In [ ]:
# Use the T-cell subset created in the preceding section.
# adata_all_subset contains only:
#   - InControl_edge T cells
#   - InPerturb_edge T cells

target_genes_to_plot = targetgene_top.copy()

target_gene_results = prepare_target_gene_group_expression_analysis(
    adata=adata_all_subset,
    genes=target_genes_to_plot,
    group_key="Tcell_group",
    group_order=("InPerturb_edge", "InControl_edge"),
    layer_key="log1p_norm",
    alternative="greater",   # InPerturb_edge > InControl_edge
    stats_csv_path=base_run_dir / f"{MIOI}_target_gene_Tcell_group_stats.csv",
    palette={
        "InPerturb_edge": "#cd1d69",
        "InControl_edge": "#f2f3fa",
    },
)

print("Genes found:", target_gene_results["genes_in"])
print("Genes missing:", target_gene_results["genes_missing"])

display(
    target_gene_results["stats_df"].sort_values(
        ["q_fdr_bh", "median_diff_perturb_minus_control"],
        ascending=[True, False],
        na_position="last",
    )
)

plot_target_gene_group_violins(
    expr_df=target_gene_results["expr_df"],
    genes=target_gene_results["genes_in"],
    group_key="Tcell_group",
    group_order=target_gene_results["group_order"],
    palette=target_gene_results["palette"],
    stats_df=target_gene_results["stats_df"],
    save_prefix=base_run_dir / f"Violin_{MIOI}_target_genes_Tcell_InPerturb_vs_InControl",
    genes_per_page=12,
    ncols=3,
    figsize_per_ax=(1.9, 2.9),
    violin_inner=None,
    violin_cut=0,
    violin_scale="width",
    show=True,
)

In [ ]:
# ============================================================
# Combined one-row figure:
#   Panel 1: MIOI edge strength, Perturb edge vs Control edge
#   Panel 2: T-cell expression of VEGFA / TBX3 / PLOD2
#   Panel 3: LR positive proportion for selected top LR pairs
#
# Layout: one row with MI-strength, target-gene, and LR-positivity panels.
# The implemented width ratio is 1:5:4, x-axis baselines are aligned, and
# violin and bar borders use #bcbec0.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu

try:
    from statsmodels.stats.multitest import multipletests
except Exception:
    multipletests = None


# -----------------------------
# Settings
# -----------------------------
COMBINED_OUTDIR = Path(base_run_dir) / f"Combined_{MIOI}_MI_targetgene_LR_panels"
COMBINED_OUTDIR.mkdir(parents=True, exist_ok=True)

TARGET_GENES_COMBINED = ["VEGFA", "TBX3", "PLOD2"]



GROUP_ORDER_EDGE = ["Perturb edge", "Control edge"]
GROUP_ORDER_TCELL = ["InPerturb_edge", "InControl_edge"]

GROUP_LABEL_MAP = {
    "InPerturb_edge": "Perturb edge",
    "InControl_edge": "Control edge",
}

PALETTE_EDGE = {
    "Perturb edge": "#cd1d69",
    "Control edge": "#f2f3fa",
}

PALETTE_TCELL = {
    "InPerturb_edge": "#cd1d69",
    "InControl_edge": "#f2f3fa",
}

BORDER_COLOR = "#bcbec0"
BAR_WIDTH = 0.36

FIGSIZE = (12.0 * 0.44, 2.9)
SHOW = True


# -----------------------------
# Helper functions
# -----------------------------
def _p_to_star_local(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "n.s."


def _format_p_local(p):
    if pd.isna(p):
        return "P = NA"
    if p < 1e-4:
        return "P < 1e-4"
    if p < 0.001:
        return f"P = {p:.1e}"
    return f"P = {p:.3f}"


def _clean_values(x):
    x = np.asarray(x, dtype=float)
    return x[np.isfinite(x)]


def _get_y_for_sig(ax, values, frac=0.08):
    values = _clean_values(values)
    if values.size == 0:
        ymin, ymax = 0.0, 1.0
    else:
        ymin, ymax = float(np.nanmin(values)), float(np.nanmax(values))
        if np.isclose(ymin, ymax):
            ymin = min(0.0, ymin)
            ymax = ymax + 1.0
    yrange = max(ymax - ymin, 1e-6)
    y = ymax + frac * yrange
    h = 0.03 * yrange
    return y, h, ymin, ymax, yrange


def _add_sig_bracket(ax, x1, x2, y, h, text, fontsize=7):
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        lw=0.8,
        c="black",
        clip_on=False,
    )
    ax.text(
        (x1 + x2) / 2,
        y + h * 1.15,
        text,
        ha="center",
        va="bottom",
        fontsize=fontsize,
        clip_on=False,
    )


def _make_lr_pair_name(lr_pair, arrow=True):
    ligand, receptor = lr_pair
    ligand = "+".join([str(x) for x in ligand])
    receptor = "+".join([str(x) for x in receptor])
    return f"{ligand}→{receptor}" if arrow else f"{ligand}-{receptor}"


def _style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)
    ax.tick_params(axis="both", width=0.7, length=3)
    ax.spines["left"].set_linewidth(0.7)
    ax.spines["bottom"].set_linewidth(0.7)


def _set_violin_edges(ax, edge_color="#bcbec0", linewidth=0.8):
    """
    Set seaborn violin body borders.
    Works by updating PolyCollections drawn by sns.violinplot.
    """
    for coll in ax.collections:
        if hasattr(coll, "set_edgecolor"):
            coll.set_edgecolor(edge_color)
        if hasattr(coll, "set_linewidth"):
            coll.set_linewidth(linewidth)


# -----------------------------
# Panel 1 data: MIOI strength
# -----------------------------


In [ ]:
LR_INDEX_TOP_USE = np.array(LR_index_top)[[0, 1, 6, 7]].tolist()
Factor_envir_df = pd.DataFrame(
    core_results["Factor_envir"],
    columns=[f"MI{i + 1}" for i in range(core_results["Factor_envir"].shape[1])],
)

if MIOI not in Factor_envir_df.columns:
    MIOI_alt = MIOI.replace("MI-", "MI")
    if MIOI_alt in Factor_envir_df.columns:
        MIOI_plot = MIOI_alt
    else:
        raise KeyError(
            f"{MIOI} not found in Factor_envir_df.columns. "
            f"Available examples: {Factor_envir_df.columns[:5].tolist()}"
        )
else:
    MIOI_plot = MIOI

x_control = _clean_values(
    Factor_envir_df.loc[
        edge_control_index_fromcancer_to_Tcell,
        MIOI_plot,
    ].to_numpy(dtype=float)
)

x_perturb = _clean_values(
    Factor_envir_df.loc[
        edge_perturb_index_fromcancer_to_Tcell,
        MIOI_plot,
    ].to_numpy(dtype=float)
)

mi_strength_plot_df = pd.DataFrame({
    "Feature": [f"{MIOI_plot} strength"] * (len(x_perturb) + len(x_control)),
    "Group": (["Perturb edge"] * len(x_perturb)) + (["Control edge"] * len(x_control)),
    "Value": np.concatenate([x_perturb, x_control]),
})

if len(x_perturb) >= 2 and len(x_control) >= 2:
    _, p_mi_strength = mannwhitneyu(x_perturb, x_control, alternative="two-sided")
else:
    p_mi_strength = np.nan


# -----------------------------
# Panel 2 data: target gene expression
# -----------------------------
target_gene_results_combined = prepare_target_gene_group_expression_analysis(
    adata=adata_all_subset,
    genes=TARGET_GENES_COMBINED,
    group_key="Tcell_group",
    group_order=("InPerturb_edge", "InControl_edge"),
    layer_key="log1p_norm",
    alternative="greater",
    stats_csv_path=COMBINED_OUTDIR / f"{MIOI_plot}_VEGFA_TBX3_PLOD2_Tcell_group_stats.csv",
    palette=PALETTE_TCELL,
)

genes_in_combined = target_gene_results_combined["genes_in"]
genes_missing_combined = target_gene_results_combined["genes_missing"]

print("Target genes found:", genes_in_combined)
print("Target genes missing:", genes_missing_combined)

expr_plot_df = target_gene_results_combined["expr_df"].copy()
expr_long_df = expr_plot_df.melt(
    id_vars="Tcell_group",
    value_vars=genes_in_combined,
    var_name="Gene",
    value_name="Expression",
)

expr_long_df["Group"] = expr_long_df["Tcell_group"].map(GROUP_LABEL_MAP)
expr_long_df["Gene"] = pd.Categorical(
    expr_long_df["Gene"],
    categories=[g for g in TARGET_GENES_COMBINED if g in genes_in_combined],
    ordered=True,
)
expr_long_df["Group"] = pd.Categorical(
    expr_long_df["Group"],
    categories=["Perturb edge", "Control edge"],
    ordered=True,
)

target_stats_df = target_gene_results_combined["stats_df"].copy()
target_stats_df["star"] = target_stats_df["q_fdr_bh"].apply(_p_to_star_local)


# -----------------------------
# Panel 3 data: LR positive proportion
# -----------------------------
LR_coexpression = core_results["LR_coexpression"]

lr_positive_records = []

for LR_index_top_cur in LR_INDEX_TOP_USE:
    LR_top_cur = processed.lr_list[LR_index_top_cur]
    LR_pair_label = _make_lr_pair_name(LR_top_cur, arrow=True)
    LR_pair_safe = _make_lr_pair_name(LR_top_cur, arrow=False)

    x_control_lr = LR_coexpression[
        edge_control_index_fromcancer_to_Tcell,
        LR_index_top_cur,
    ].ravel()

    x_perturb_lr = LR_coexpression[
        edge_perturb_index_fromcancer_to_Tcell,
        LR_index_top_cur,
    ].ravel()

    x_control_lr = _clean_values(x_control_lr)
    x_perturb_lr = _clean_values(x_perturb_lr)

    lr_positive_records.append({
        "LR_index": LR_index_top_cur,
        "LR_pair": LR_pair_label,
        "LR_pair_safe": LR_pair_safe,
        "Control_edge_positive_prop": float(np.mean(x_control_lr > 0)) if len(x_control_lr) else np.nan,
        "Perturb_edge_positive_prop": float(np.mean(x_perturb_lr > 0)) if len(x_perturb_lr) else np.nan,
        "n_control": len(x_control_lr),
        "n_perturb": len(x_perturb_lr),
    })

LR_positive_prop_combined = pd.DataFrame(lr_positive_records)
LR_positive_prop_combined.to_csv(
    COMBINED_OUTDIR / f"LR_positive_proportion_{MIOI_plot}_selected.csv",
    index=False,
)

display(LR_positive_prop_combined)


# -----------------------------
# Save source tables
# -----------------------------
mi_strength_plot_df.to_csv(
    COMBINED_OUTDIR / f"{MIOI_plot}_MI_strength_plot_data.csv",
    index=False,
)

expr_long_df.to_csv(
    COMBINED_OUTDIR / f"{MIOI_plot}_VEGFA_TBX3_PLOD2_expression_plot_data.csv",
    index=False,
)


# -----------------------------
# Plot: one row, three panels
# -----------------------------


In [ ]:
plt.close("all")
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 8

sns.set_style("white")

fig, axes = plt.subplots(
    1,
    3,
    figsize=FIGSIZE,
    gridspec_kw={"width_ratios": [0.5, 2.5, 2]},
    sharex=False,
    sharey=False,
)

ax0, ax1, ax2 = axes


# ============================================================
# Panel 1: MIOI strength
# ============================================================
sns.violinplot(
    data=mi_strength_plot_df,
    x="Feature",
    y="Value",
    hue="Group",
    hue_order=GROUP_ORDER_EDGE,
    palette=PALETTE_EDGE,
    inner=None,
    cut=0,
    scale="width",
    linewidth=0.8,
    width=0.78,
    ax=ax0,
)
_set_violin_edges(ax0, edge_color=BORDER_COLOR, linewidth=0.8)

# Add medians
for gi, group in enumerate(GROUP_ORDER_EDGE):
    vals = mi_strength_plot_df.loc[
        mi_strength_plot_df["Group"] == group,
        "Value",
    ].to_numpy(dtype=float)
    vals = _clean_values(vals)
    if len(vals):
        x_pos = -0.20 if group == "Perturb edge" else 0.20
        ax0.hlines(
            np.nanmedian(vals),
            x_pos - 0.09,
            x_pos + 0.09,
            color="black",
            linewidth=1.1,
            zorder=5,
        )

y, h, ymin, ymax, yrng = _get_y_for_sig(ax0, mi_strength_plot_df["Value"].values)
_add_sig_bracket(
    ax0,
    -0.20,
    0.20,
    y,
    h,
    _format_p_local(p_mi_strength),
    fontsize=7,
)

ax0.set_title(f"{MIOI_plot} edge strength", fontsize=9, pad=6)
ax0.set_xlabel("")
ax0.set_ylabel(f"{MIOI_plot} strength", fontsize=8)
ax0.set_xticklabels([MIOI_plot], rotation=0)
ax0.set_ylim(ymin - 0.03 * yrng, y + 4.0 * h)
if ax0.get_legend() is not None:
    ax0.get_legend().remove()
_style_axis(ax0)


# ============================================================
# Panel 2: T-cell target-gene expression
# ============================================================
sns.violinplot(
    data=expr_long_df,
    x="Gene",
    y="Expression",
    hue="Group",
    hue_order=GROUP_ORDER_EDGE,
    palette=PALETTE_EDGE,
    inner=None,
    cut=0,
    scale="width",
    linewidth=0.8,
    width=0.78,
    ax=ax1,
)
_set_violin_edges(ax1, edge_color=BORDER_COLOR, linewidth=0.8)

# Add median bars and q-value stars
gene_order = [g for g in TARGET_GENES_COMBINED if g in genes_in_combined]

for gene_i, gene in enumerate(gene_order):
    for group in GROUP_ORDER_EDGE:
        vals = expr_long_df.loc[
            (expr_long_df["Gene"].astype(str) == gene)
            & (expr_long_df["Group"].astype(str) == group),
            "Expression",
        ].to_numpy(dtype=float)
        vals = _clean_values(vals)

        if len(vals):
            offset = -0.20 if group == "Perturb edge" else 0.20
            ax1.hlines(
                np.nanmedian(vals),
                gene_i + offset - 0.08,
                gene_i + offset + 0.08,
                color="black",
                linewidth=1.0,
                zorder=5,
            )

    q_val = np.nan
    if gene in target_stats_df["gene"].values:
        q_val = float(target_stats_df.loc[target_stats_df["gene"] == gene, "q_fdr_bh"].iloc[0])

    vals_gene = expr_long_df.loc[
        expr_long_df["Gene"].astype(str) == gene,
        "Expression",
    ].to_numpy(dtype=float)

    y_gene, h_gene, _, _, _ = _get_y_for_sig(ax1, vals_gene, frac=0.10)
    _add_sig_bracket(
        ax1,
        gene_i - 0.20,
        gene_i + 0.20,
        y_gene,
        h_gene,
        _p_to_star_local(q_val),
        fontsize=7,
    )

expr_vals_all = _clean_values(expr_long_df["Expression"].to_numpy(dtype=float))
if len(expr_vals_all):
    expr_ymin = float(np.nanmin(expr_vals_all))
    expr_ymax = float(np.nanmax(expr_vals_all))
    expr_yrng = max(expr_ymax - expr_ymin, 1e-6)
    ax1.set_ylim(expr_ymin - 0.03 * expr_yrng, expr_ymax + 0.22 * expr_yrng)

ax1.set_title("T-cell target genes", fontsize=9, pad=6)
ax1.set_xlabel("")
ax1.set_ylabel("log1p normalized expression", fontsize=8)
ax1.tick_params(axis="x", labelrotation=0)
if ax1.get_legend() is not None:
    ax1.get_legend().remove()
_style_axis(ax1)


# ============================================================
# Panel 3: LR positive proportion
# ============================================================
lr_plot_df = LR_positive_prop_combined.copy()
x = np.arange(lr_plot_df.shape[0])

ax2.bar(
    x - BAR_WIDTH / 2,
    lr_plot_df["Perturb_edge_positive_prop"].to_numpy(dtype=float),
    width=BAR_WIDTH,
    label="Perturb edge",
    color=PALETTE_EDGE["Perturb edge"],
    edgecolor=BORDER_COLOR,
    linewidth=0.8,
)

ax2.bar(
    x + BAR_WIDTH / 2,
    lr_plot_df["Control_edge_positive_prop"].to_numpy(dtype=float),
    width=BAR_WIDTH,
    label="Control edge",
    color=PALETTE_EDGE["Control edge"],
    edgecolor=BORDER_COLOR,
    linewidth=0.8,
)

ax2.set_xticks(x)
ax2.set_xticklabels(
    lr_plot_df["LR_pair"].tolist(),
    rotation=35,
    ha="right",
    fontsize=7,
)
ax2.set_title("LR-pair positivity", fontsize=9, pad=6)
ax2.set_xlabel("")
ax2.set_ylabel("Positive proportion", fontsize=8)

lr_ymax = np.nanmax(
    lr_plot_df[
        ["Perturb_edge_positive_prop", "Control_edge_positive_prop"]
    ].to_numpy(dtype=float)
)
ax2.set_ylim(0, max(0.05, lr_ymax * 1.18))
_style_axis(ax2)


# -----------------------------
# Shared legend and alignment
# -----------------------------
handles = [
    plt.Line2D(
        [0],
        [0],
        marker="s",
        color="none",
        markerfacecolor=PALETTE_EDGE["Perturb edge"],
        markeredgecolor=BORDER_COLOR,
        markeredgewidth=0.8,
        markersize=7,
        label="Perturb edge",
    ),
    plt.Line2D(
        [0],
        [0],
        marker="s",
        color="none",
        markerfacecolor=PALETTE_EDGE["Control edge"],
        markeredgecolor=BORDER_COLOR,
        markeredgewidth=0.8,
        markersize=7,
        label="Control edge",
    ),
]

fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=2,
    frameon=False,
    fontsize=8,
)

fig.suptitle(
    f"{MIOI_plot}: MI strength, T-cell response genes, and LR-pair positivity",
    fontsize=10,
    y=1.02,
)

# Keep all x-axis baselines aligned by fixing the same subplot bottom.
fig.subplots_adjust(
    left=0.07,
    right=0.995,
    bottom=0.32,
    top=0.82,
    wspace=0.42,
)

fig.align_xlabels(axes)
fig.align_ylabels(axes)

save_base = COMBINED_OUTDIR / f"Combined_one_row_{MIOI_plot}_MI_targetgenes_LR_positive"

fig.savefig(str(save_base) + ".png", dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(str(save_base) + ".pdf", bbox_inches="tight", facecolor="white")

if SHOW:
    plt.show()

plt.close(fig)

print(f"Saved combined one-row figure to:\n{save_base}.png\n{save_base}.pdf")

In [ ]:
# ============================================================
# Perturbation-specific T-cell differential-expression analysis.
# For each perturbation gene in perturb_gene_OI:
#   1. use build_edge_group_masks_for_selected_genes() to get:
#        - cancer -> T-cell edges from melanoma/cancer cells carrying this perturbation
#        - cancer -> T-cell control edges from unperturbed/control melanoma/cancer cells
#   2. split T cells into:
#        - T cells receiving edges from this perturb-gene melanoma/cancer cells
#        - T cells receiving control edges, excluding the first group
#   3. perform T-cell DEG analysis:
#        log2FC and Wilcoxon rank-sum p-value
#
# Edge groups are obtained from the shared helper to preserve its cell-type definitions.
# ============================================================

import re
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import ranksums
from pathlib import Path

Tcell_DEG_LOG2FC_THRESHOLD = 0.4
Tcell_DEG_PVALUE_THRESHOLD = 0.05
Tcell_DEG_MIN_CELLS_PER_GROUP = 5
Tcell_DEG_EPS = 1e-8

tcell_deg_outdir = Path(base_run_dir) / f"{MIOI}_Tcell_DEG_neighboring_perturbed_melanoma"
tcell_deg_outdir.mkdir(parents=True, exist_ok=True)


def _as_numpy_local(x):
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _edge_index_to_2col_local(edge_index):
    edge_index = _as_numpy_local(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[0] == 2 and edge_index.shape[1] != 2:
        edge_index = edge_index.T

    if edge_index.shape[1] != 2:
        raise ValueError(f"edge_index must have two columns after conversion, got shape {edge_index.shape}")

    return edge_index.astype(int, copy=False)


def _get_expression_matrix_for_deg(adata, preferred_layer="log1p_norm"):
    if preferred_layer is not None and preferred_layer in adata.layers:
        print(f"Using adata.layers['{preferred_layer}'] for T-cell DEG.")
        return adata.layers[preferred_layer]

    print("Using adata.X for T-cell DEG.")
    return adata.X


def _subset_to_dense(X, idx):
    idx = np.asarray(idx, dtype=int)
    X_sub = X[idx, :]

    if sp.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)

    return X_sub.astype(float, copy=False)


def _wilcox_rank_sum_deg(
    X_case,
    X_control,
    gene_names,
    perturb_gene,
    eps=1e-8,
):
    mean_case = np.nanmean(X_case, axis=0)
    mean_control = np.nanmean(X_control, axis=0)

    log2fc = np.log2((mean_case + eps) / (mean_control + eps))

    pvals = np.full(len(gene_names), np.nan, dtype=float)
    stats = np.full(len(gene_names), np.nan, dtype=float)

    for j in range(len(gene_names)):
        x_case = X_case[:, j]
        x_ctrl = X_control[:, j]

        x_case = x_case[np.isfinite(x_case)]
        x_ctrl = x_ctrl[np.isfinite(x_ctrl)]

        if len(x_case) < 2 or len(x_ctrl) < 2:
            continue

        if (
            np.nanstd(x_case) == 0
            and np.nanstd(x_ctrl) == 0
            and np.nanmean(x_case) == np.nanmean(x_ctrl)
        ):
            stats[j] = 0.0
            pvals[j] = 1.0
        else:
            stat, pval = ranksums(x_case, x_ctrl)
            stats[j] = stat
            pvals[j] = pval

    deg_df = pd.DataFrame({
        "PerturbGene": perturb_gene,
        "Gene": gene_names,
        "mean_Tcell_near_perturbed_melanoma": mean_case,
        "mean_Tcell_near_control_melanoma": mean_control,
        "log2FC": log2fc,
        "wilcox_rank_sum_stat": stats,
        "pvalue": pvals,
    })

    return deg_df


def _safe_filename_local(x):
    x = str(x)
    x = re.sub(r"[^\w\-.]+", "_", x)
    return x.strip("_")


# ------------------------------------------------------------
# Prepare global objects
# ------------------------------------------------------------
adata_all_for_deg = core_results["adata_all"]
edge_index_all_for_deg = _edge_index_to_2col_local(core_results["edge_index_all"])
perturb_annotation_for_deg = core_results["perturb_annotation"]

if not isinstance(perturb_annotation_for_deg, pd.DataFrame):
    raise TypeError(
        "core_results['perturb_annotation'] should be a pandas DataFrame "
        "with shape cells x perturbation genes."
    )

if perturb_annotation_for_deg.shape[0] != adata_all_for_deg.n_obs:
    raise ValueError(
        f"perturb_annotation rows ({perturb_annotation_for_deg.shape[0]}) "
        f"do not match adata_all cells ({adata_all_for_deg.n_obs})."
    )

X_deg = _get_expression_matrix_for_deg(adata_all_for_deg, preferred_layer="log1p_norm")
gene_names_deg = np.asarray(adata_all_for_deg.var_names.astype(str))

print("adata_all_for_deg:", adata_all_for_deg.shape)
print("edge_index_all_for_deg:", edge_index_all_for_deg.shape)
print("perturb_annotation_for_deg:", perturb_annotation_for_deg.shape)
print("Number of genes for DEG:", len(gene_names_deg))


# ------------------------------------------------------------
# Run DEG for each perturb_gene
# ------------------------------------------------------------
tcell_deg_records = []
tcell_deg_group_records = []

for perturb_gene_cur in perturb_gene_OI:
    if perturb_gene_cur not in perturb_annotation_for_deg.columns:
        print(f"Skip {perturb_gene_cur}: not found in perturb_annotation.")
        continue

    # This function already knows how to identify cancer -> T-cell edges.
    edge_control_cur, edge_perturb_cur = build_edge_group_masks_for_selected_genes(
        adata_all=adata_all_for_deg,
        edge_index_all=edge_index_all_for_deg,
        perturb_annotation=perturb_annotation_for_deg,
        perturb_gene_OI_choose=[perturb_gene_cur],
    )

    edge_control_cur = np.asarray(edge_control_cur, dtype=int)
    edge_perturb_cur = np.asarray(edge_perturb_cur, dtype=int)

    edge_control_cur = edge_control_cur[
        (edge_control_cur >= 0) & (edge_control_cur < edge_index_all_for_deg.shape[0])
    ]
    edge_perturb_cur = edge_perturb_cur[
        (edge_perturb_cur >= 0) & (edge_perturb_cur < edge_index_all_for_deg.shape[0])
    ]

    # For cancer -> T-cell edges:
    #   sender = melanoma/cancer cell
    #   receiver = T cell
    perturb_sender_melanoma_idx = np.unique(edge_index_all_for_deg[edge_perturb_cur, 0]).astype(int)
    control_sender_melanoma_idx = np.unique(edge_index_all_for_deg[edge_control_cur, 0]).astype(int)

    tcell_near_perturb_idx = np.unique(edge_index_all_for_deg[edge_perturb_cur, 1]).astype(int)
    tcell_near_control_idx = np.unique(edge_index_all_for_deg[edge_control_cur, 1]).astype(int)

    # Strict control:
    # T cells near control melanoma/cancer cells but not near this perturb-gene melanoma/cancer cells.
    tcell_control_idx = np.setdiff1d(
        tcell_near_control_idx,
        tcell_near_perturb_idx,
        assume_unique=False,
    )

    control_definition = "Tcells_near_control_melanoma_not_near_this_perturbed_melanoma"

    # Fallback if strict control is too small.
    # This still uses T cells from control cancer->T-cell edges, but allows overlap.
    if len(tcell_control_idx) < Tcell_DEG_MIN_CELLS_PER_GROUP:
        tcell_control_idx = tcell_near_control_idx.copy()
        control_definition = "Tcells_near_control_melanoma_allow_overlap"

    n_pert_melanoma = len(perturb_sender_melanoma_idx)
    n_ctrl_melanoma = len(control_sender_melanoma_idx)
    n_pert_edges = len(edge_perturb_cur)
    n_ctrl_edges = len(edge_control_cur)
    n_tcell_case = len(tcell_near_perturb_idx)
    n_tcell_control = len(tcell_control_idx)

    print(
        f"{perturb_gene_cur}: "
        f"perturb_edges={n_pert_edges}, control_edges={n_ctrl_edges}, "
        f"perturbed melanoma/cancer={n_pert_melanoma}, "
        f"control melanoma/cancer={n_ctrl_melanoma}, "
        f"T cells near perturb={n_tcell_case}, "
        f"T cells control={n_tcell_control}, "
        f"control={control_definition}"
    )

    tcell_deg_group_records.append({
        "PerturbGene": perturb_gene_cur,
        "n_perturb_edges_cancer_to_Tcell": n_pert_edges,
        "n_control_edges_cancer_to_Tcell": n_ctrl_edges,
        "n_perturbed_melanoma_sender_cells": n_pert_melanoma,
        "n_control_melanoma_sender_cells": n_ctrl_melanoma,
        "n_Tcell_near_perturbed_melanoma": n_tcell_case,
        "n_Tcell_control": n_tcell_control,
        "control_definition": control_definition,
    })

    if (
        n_tcell_case < Tcell_DEG_MIN_CELLS_PER_GROUP
        or n_tcell_control < Tcell_DEG_MIN_CELLS_PER_GROUP
    ):
        print(f"Skip DEG for {perturb_gene_cur}: too few T cells in one group.")
        continue

    X_case = _subset_to_dense(X_deg, tcell_near_perturb_idx)
    X_control = _subset_to_dense(X_deg, tcell_control_idx)

    deg_df_cur = _wilcox_rank_sum_deg(
        X_case=X_case,
        X_control=X_control,
        gene_names=gene_names_deg,
        perturb_gene=perturb_gene_cur,
        eps=Tcell_DEG_EPS,
    )

    deg_df_cur["n_Tcell_near_perturbed_melanoma"] = n_tcell_case
    deg_df_cur["n_Tcell_control"] = n_tcell_control
    deg_df_cur["n_perturb_edges_cancer_to_Tcell"] = n_pert_edges
    deg_df_cur["n_control_edges_cancer_to_Tcell"] = n_ctrl_edges

    deg_df_cur["Significant_up"] = (
        (deg_df_cur["log2FC"] > Tcell_DEG_LOG2FC_THRESHOLD)
        & (deg_df_cur["pvalue"] < Tcell_DEG_PVALUE_THRESHOLD)
    )

    deg_df_cur.to_csv(
        tcell_deg_outdir / f"Tcell_DEG_{_safe_filename_local(perturb_gene_cur)}.csv",
        index=False,
    )

    tcell_deg_records.append(deg_df_cur)


if len(tcell_deg_records) == 0:
    raise RuntimeError("No T-cell DEG results were generated.")

tcell_deg_all_df = pd.concat(tcell_deg_records, axis=0, ignore_index=True)
tcell_deg_group_summary_df = pd.DataFrame(tcell_deg_group_records)

tcell_deg_sig_df = tcell_deg_all_df.loc[
    tcell_deg_all_df["Significant_up"]
].copy()

sig_tcell_deg_gene_set_union = sorted(
    tcell_deg_sig_df["Gene"].astype(str).unique()
)

tcell_deg_summary_by_gene = (
    tcell_deg_sig_df
    .groupby("Gene")
    .agg(
        n_perturb_genes_significant=("PerturbGene", "nunique"),
        perturb_genes_significant=("PerturbGene", lambda x: ";".join(sorted(map(str, set(x))))),
        max_log2FC=("log2FC", "max"),
        min_pvalue=("pvalue", "min"),
    )
    .reset_index()
    .sort_values(
        ["n_perturb_genes_significant", "max_log2FC"],
        ascending=[False, False],
    )
)

tcell_deg_all_df.to_csv(
    tcell_deg_outdir / f"{MIOI}_Tcell_DEG_all_perturb_genes.csv",
    index=False,
)

tcell_deg_sig_df.to_csv(
    tcell_deg_outdir / (
        f"{MIOI}_Tcell_DEG_significant_up_"
        f"log2FC{Tcell_DEG_LOG2FC_THRESHOLD}_p{Tcell_DEG_PVALUE_THRESHOLD}.csv"
    ),
    index=False,
)

tcell_deg_group_summary_df.to_csv(
    tcell_deg_outdir / f"{MIOI}_Tcell_DEG_group_summary.csv",
    index=False,
)

tcell_deg_summary_by_gene.to_csv(
    tcell_deg_outdir / f"{MIOI}_Tcell_DEG_significant_up_gene_frequency.csv",
    index=False,
)

print(f"Significant upregulated T-cell DEG union: {len(sig_tcell_deg_gene_set_union)} genes")
print(sig_tcell_deg_gene_set_union)

display(tcell_deg_group_summary_df)
display(tcell_deg_summary_by_gene.head(30))

In [ ]:
# ============================================================
# Differential-expression dot plot for T-cell response genes.
#
# Rows:
#   perturb genes in perturb_gene_OI
#
# Columns:
#   union of significant T-cell DEG genes across perturb genes
#
# Dot size:
#   -log10 Wilcoxon rank-sum P value
#   capped at 5, so values > 5 are shown as 5
#
# Dot color:
#   log2FC, using coolwarm
#
# Yellow boxes:
#   perturb-gene-specific response genes passing:
#       log2FC > Tcell_DEG_LOG2FC_THRESHOLD
#       pvalue < Tcell_DEG_PVALUE_THRESHOLD
# ============================================================

from pathlib import Path
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import matplotlib as mpl

# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------
DOTPLOT_MAX_GENES = None
# Set DOTPLOT_MAX_GENES = 60 if the figure becomes too crowded.

DOTPLOT_MIN_NEGLOG10P_FOR_SIZE = 0.0
DOTPLOT_MAX_NEGLOG10P_FOR_SIZE = 5.0   # values > 5 are capped at 5 for dot size

DOTPLOT_MIN_DOT_SIZE = 8
DOTPLOT_MAX_DOT_SIZE = 120

DOTPLOT_YELLOW_BOX_COLOR = "#FFD92F"
DOTPLOT_YELLOW_BOX_LINEWIDTH = 1.2

deg_dotplot_outdir = Path(tcell_deg_outdir) / "DEG_dotplot"
deg_dotplot_outdir.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Check input tables
# ------------------------------------------------------------
required_cols = ["PerturbGene", "Gene", "log2FC", "pvalue"]
missing_cols = [c for c in required_cols if c not in tcell_deg_all_df.columns]

if len(missing_cols) > 0:
    raise KeyError(f"tcell_deg_all_df is missing required columns: {missing_cols}")

deg_dot_df = tcell_deg_all_df.copy()

if "Significant_up" not in deg_dot_df.columns:
    deg_dot_df["Significant_up"] = (
        (deg_dot_df["log2FC"] > Tcell_DEG_LOG2FC_THRESHOLD)
        & (deg_dot_df["pvalue"] < Tcell_DEG_PVALUE_THRESHOLD)
    )

deg_dot_df["PerturbGene"] = deg_dot_df["PerturbGene"].astype(str)
deg_dot_df["Gene"] = deg_dot_df["Gene"].astype(str)
deg_dot_df["log2FC"] = pd.to_numeric(deg_dot_df["log2FC"], errors="coerce")
deg_dot_df["pvalue"] = pd.to_numeric(deg_dot_df["pvalue"], errors="coerce")
deg_dot_df["Significant_up"] = deg_dot_df["Significant_up"].astype(bool)

deg_dot_df = deg_dot_df.replace([np.inf, -np.inf], np.nan)


# ------------------------------------------------------------
# Row order: perturb_gene_OI order
# ------------------------------------------------------------
perturb_gene_order = [
    str(g) for g in perturb_gene_OI
    if str(g) in set(deg_dot_df["PerturbGene"])
]

if len(perturb_gene_order) == 0:
    raise ValueError("No perturb_gene_OI genes were found in tcell_deg_all_df['PerturbGene'].")


# ------------------------------------------------------------
# Column order: significant DEG gene union
# Sorted by:
#   1. number of perturb genes where the gene is significant
#   2. maximum log2FC
#   3. minimum pvalue
# ------------------------------------------------------------
sig_pair_df = deg_dot_df.loc[
    (deg_dot_df["log2FC"] > Tcell_DEG_LOG2FC_THRESHOLD)
    & (deg_dot_df["pvalue"] < Tcell_DEG_PVALUE_THRESHOLD)
].copy()

if sig_pair_df.shape[0] == 0:
    raise ValueError(
        "No significant T-cell response genes found under current thresholds: "
        f"log2FC > {Tcell_DEG_LOG2FC_THRESHOLD}, "
        f"pvalue < {Tcell_DEG_PVALUE_THRESHOLD}."
    )

gene_summary_for_order = (
    sig_pair_df
    .groupby("Gene", as_index=False)
    .agg(
        n_significant_perturb_genes=("PerturbGene", "nunique"),
        max_log2FC=("log2FC", "max"),
        min_pvalue=("pvalue", "min"),
        perturb_genes_significant=("PerturbGene", lambda x: ";".join(sorted(map(str, set(x))))),
    )
    .sort_values(
        ["n_significant_perturb_genes", "max_log2FC", "min_pvalue"],
        ascending=[False, False, True],
    )
)

gene_order = gene_summary_for_order["Gene"].astype(str).tolist()

if DOTPLOT_MAX_GENES is not None:
    gene_order = gene_order[:DOTPLOT_MAX_GENES]

print(f"Perturb genes shown: {len(perturb_gene_order)}")
print(f"Significant DEG genes shown: {len(gene_order)}")

gene_summary_for_order.to_csv(
    deg_dotplot_outdir / (
        f"{MIOI}_Tcell_DEG_dotplot_gene_order_"
        f"log2FC{Tcell_DEG_LOG2FC_THRESHOLD}_p{Tcell_DEG_PVALUE_THRESHOLD}.csv"
    ),
    index=False,
)


# ------------------------------------------------------------
# Build complete plotting grid
# ------------------------------------------------------------
plot_grid = pd.MultiIndex.from_product(
    [perturb_gene_order, gene_order],
    names=["PerturbGene", "Gene"],
).to_frame(index=False)

plot_df = plot_grid.merge(
    deg_dot_df[
        [
            "PerturbGene",
            "Gene",
            "log2FC",
            "pvalue",
            "Significant_up",
        ]
    ],
    on=["PerturbGene", "Gene"],
    how="left",
)

plot_df["log2FC"] = plot_df["log2FC"].fillna(0.0)
plot_df["pvalue"] = plot_df["pvalue"].fillna(1.0)
plot_df["Significant_up"] = plot_df["Significant_up"].fillna(False).astype(bool)

plot_df["neglog10p"] = -np.log10(plot_df["pvalue"].clip(lower=1e-300))

# Cap dot size at -log10(P) = 5.
# The original uncapped value is kept in "neglog10p".
plot_df["neglog10p_for_size"] = plot_df["neglog10p"].clip(
    lower=DOTPLOT_MIN_NEGLOG10P_FOR_SIZE,
    upper=DOTPLOT_MAX_NEGLOG10P_FOR_SIZE,
)

plot_df["DotSize"] = DOTPLOT_MIN_DOT_SIZE + (
    (plot_df["neglog10p_for_size"] - DOTPLOT_MIN_NEGLOG10P_FOR_SIZE)
    / (DOTPLOT_MAX_NEGLOG10P_FOR_SIZE - DOTPLOT_MIN_NEGLOG10P_FOR_SIZE)
) * (DOTPLOT_MAX_DOT_SIZE - DOTPLOT_MIN_DOT_SIZE)

plot_df["x"] = plot_df["Gene"].map({g: i for i, g in enumerate(gene_order)})
plot_df["y"] = plot_df["PerturbGene"].map({g: i for i, g in enumerate(perturb_gene_order)})

plot_df.to_csv(
    deg_dotplot_outdir / (
        f"{MIOI}_Tcell_DEG_dotplot_plotting_table_"
        f"log2FC{Tcell_DEG_LOG2FC_THRESHOLD}_p{Tcell_DEG_PVALUE_THRESHOLD}.csv"
    ),
    index=False,
)


# ------------------------------------------------------------
# Color normalization for log2FC
# ------------------------------------------------------------
log2fc_abs_max = np.nanmax(np.abs(plot_df["log2FC"].values))

if not np.isfinite(log2fc_abs_max) or log2fc_abs_max == 0:
    log2fc_abs_max = 1.0

# Avoid extremely wide color scale driven by one outlier
log2fc_abs_q = np.nanquantile(np.abs(plot_df["log2FC"].values), 0.98)

if np.isfinite(log2fc_abs_q) and log2fc_abs_q > 0:
    log2fc_color_lim = max(log2fc_abs_q, Tcell_DEG_LOG2FC_THRESHOLD)
else:
    log2fc_color_lim = log2fc_abs_max

norm = TwoSlopeNorm(
    vmin=-log2fc_color_lim,
    vcenter=0,
    vmax=log2fc_color_lim,
)

cmap = plt.get_cmap("coolwarm")


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
apply_publication_style(font_size=8)

fig_width = max(6.0, 0.28 * len(gene_order) + 1.8)
fig_height = max(3.0, 0.35 * len(perturb_gene_order) + 1.2)

fig, ax = plt.subplots(figsize=(fig_width, fig_height))

scatter = ax.scatter(
    plot_df["x"],
    plot_df["y"],
    s=plot_df["DotSize"],
    c=plot_df["log2FC"],
    cmap=cmap,
    norm=norm,
    edgecolor="none",
    alpha=0.9,
)


# ------------------------------------------------------------
# Add yellow boxes around perturb-gene-specific significant response genes
# ------------------------------------------------------------
sig_plot_df = plot_df.loc[plot_df["Significant_up"]].copy()

for _, row in sig_plot_df.iterrows():
    rect = Rectangle(
        (row["x"] - 0.48, row["y"] - 0.48),
        0.96,
        0.96,
        fill=False,
        edgecolor=DOTPLOT_YELLOW_BOX_COLOR,
        linewidth=DOTPLOT_YELLOW_BOX_LINEWIDTH,
        zorder=4,
    )
    ax.add_patch(rect)


# ------------------------------------------------------------
# Axis settings
# ------------------------------------------------------------
ax.set_xticks(np.arange(len(gene_order)))
ax.set_xticklabels(gene_order, rotation=90, ha="center", va="top")

ax.set_yticks(np.arange(len(perturb_gene_order)))
ax.set_yticklabels(perturb_gene_order)

ax.set_xlim(-0.6, len(gene_order) - 0.4)
ax.set_ylim(len(perturb_gene_order) - 0.4, -0.6)

ax.set_xlabel("Significant T-cell response genes")
ax.set_ylabel("Perturbed gene in melanoma cells")

ax.set_title(
    (
        "T-cell DEG response near perturbed melanoma cells\n"
        f"Yellow boxes: log2FC>{Tcell_DEG_LOG2FC_THRESHOLD}, "
        f"P<{Tcell_DEG_PVALUE_THRESHOLD}"
    ),
    pad=6,
)

# Light grid to emphasize dot cells
ax.set_axisbelow(True)
ax.grid(
    which="major",
    color="#E0E0E0",
    linewidth=0.5,
    linestyle="-",
)

ax.tick_params(axis="both", which="both", direction="out", length=0)

sns.despine(ax=ax, top=True, right=True, left=False, bottom=False)


# ------------------------------------------------------------
# Colorbar for log2FC
# ------------------------------------------------------------
cbar = fig.colorbar(
    scatter,
    ax=ax,
    fraction=0.025,
    pad=0.01,
)
cbar.set_label("log2FC")
cbar.outline.set_linewidth(0.6)


# ------------------------------------------------------------
# Size legend for capped -log10(P)
# Values larger than 5 are shown as >=5
# ------------------------------------------------------------
legend_values = [1, 2, 3, 4, 5]

legend_handles = []

for v in legend_values:
    size_v = DOTPLOT_MIN_DOT_SIZE + (
        (v - DOTPLOT_MIN_NEGLOG10P_FOR_SIZE)
        / (DOTPLOT_MAX_NEGLOG10P_FOR_SIZE - DOTPLOT_MIN_NEGLOG10P_FOR_SIZE)
    ) * (DOTPLOT_MAX_DOT_SIZE - DOTPLOT_MIN_DOT_SIZE)

    label_v = "≥5" if v == DOTPLOT_MAX_NEGLOG10P_FOR_SIZE else str(v)

    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="0.55",
            markeredgecolor="none",
            markersize=np.sqrt(size_v),
            label=label_v,
        )
    )

size_legend = ax.legend(
    handles=legend_handles,
    title=r"$-\log_{10}(P)$",
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.03, 1.0),
    borderaxespad=0,
    fontsize=7,
    title_fontsize=8,
)

ax.add_artist(size_legend)


# ------------------------------------------------------------
# Yellow box legend
# ------------------------------------------------------------
yellow_box_handle = Rectangle(
    (0, 0),
    1,
    1,
    fill=False,
    edgecolor=DOTPLOT_YELLOW_BOX_COLOR,
    linewidth=DOTPLOT_YELLOW_BOX_LINEWIDTH,
)

ax.legend(
    handles=[yellow_box_handle],
    labels=["Pass threshold"],
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.03, 0.68),
    borderaxespad=0,
    fontsize=7,
)

fig.tight_layout()

dotplot_prefix = deg_dotplot_outdir / (
    f"Dotplot_Tcell_DEG_log2FC_neglog10P_"
    f"{MIOI}_log2FC{Tcell_DEG_LOG2FC_THRESHOLD}_p{Tcell_DEG_PVALUE_THRESHOLD}_sizecap5"
)

fig.savefig(f"{dotplot_prefix}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"{dotplot_prefix}.pdf", bbox_inches="tight")

plt.show()
plt.close(fig)

display(gene_summary_for_order.head(30))
display(plot_df.loc[plot_df["Significant_up"]].head(50))

In [ ]:
# ============================================================
# Stem plot for MI-17 receiver-target gene loadings.
#
# y-axis:
#   MI17 receiver loading normalized per gene:
#     loading_receiver[MI17, gene] / sum_MIs loading_receiver[MI, gene]
#
# Color:
#   red  = gene is significantly upregulated in T cells near perturbed melanoma
#          for at least one perturb_gene in perturb_gene_OI
#   gray = otherwise
#
# max_log2FC and min_pvalue are computed from tcell_deg_all_df for all genes,
# not only significant DEG genes. Therefore, max_log2FC is retained
#   for non-significant genes.
# ============================================================

from scipy.stats import mannwhitneyu
from matplotlib.lines import Line2D

MI_RECEIVER_LOADING_MIOI = "MI17"
MI_RECEIVER_LOADING_INDEX = int(MI_RECEIVER_LOADING_MIOI.replace("MI-", "").replace("MI", "")) - 1
TOP_N_STEM_GENES = 30   # set to None if you want to plot all genes


def _candidate_mi_names(mi_name):
    mi_name = str(mi_name)
    mi_num = mi_name.replace("MI-", "").replace("MI", "")
    return [
        mi_name,
        f"MI{mi_num}",
        f"MI-{mi_num}",
        f"MI_{mi_num}",
        int(mi_num) if str(mi_num).isdigit() else mi_name,
        int(mi_num) - 1 if str(mi_num).isdigit() else mi_name,
    ]


def _get_sum_normalized_receiver_loading(loading_receiver_df, mi_name, mi_index, eps=1e-12):
    loading_df = loading_receiver_df.copy()

    loading_num = loading_df.apply(pd.to_numeric, errors="coerce")
    loading_num = loading_num.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    mi_candidates = _candidate_mi_names(mi_name)

    # Case 1: rows are MIs, columns are genes
    row_match = None
    for cand in mi_candidates:
        if cand in loading_num.index:
            row_match = cand
            break
        if str(cand) in loading_num.index.astype(str).tolist():
            row_match = loading_num.index[loading_num.index.astype(str) == str(cand)][0]
            break

    if row_match is not None:
        mi_loading = loading_num.loc[row_match, :].astype(float)
        denom = loading_num.sum(axis=0).astype(float)
        orientation = "rows_are_MIs_columns_are_genes"

    else:
        # Case 2: columns are MIs, rows are genes
        col_match = None
        for cand in mi_candidates:
            if cand in loading_num.columns:
                col_match = cand
                break
            if str(cand) in loading_num.columns.astype(str).tolist():
                col_match = loading_num.columns[loading_num.columns.astype(str) == str(cand)][0]
                break

        if col_match is not None:
            mi_loading = loading_num.loc[:, col_match].astype(float)
            denom = loading_num.sum(axis=1).astype(float)
            orientation = "rows_are_genes_columns_are_MIs"

        else:
            # Fallback by shape
            if loading_num.shape[0] <= loading_num.shape[1]:
                mi_loading = loading_num.iloc[mi_index, :].astype(float)
                denom = loading_num.sum(axis=0).astype(float)
                orientation = "fallback_rows_are_MIs_columns_are_genes"
            else:
                mi_loading = loading_num.iloc[:, mi_index].astype(float)
                denom = loading_num.sum(axis=1).astype(float)
                orientation = "fallback_rows_are_genes_columns_are_MIs"

    denom = denom.replace(0, np.nan)
    normalized_loading = mi_loading / (denom + eps)
    normalized_loading = normalized_loading.replace([np.inf, -np.inf], np.nan).dropna()
    normalized_loading.index = normalized_loading.index.astype(str)

    print("Receiver loading orientation:", orientation)
    print("Normalized loading vector length:", normalized_loading.shape[0])

    return normalized_loading


# ------------------------------------------------------------
# Build all-gene DEG summary.
# This avoids NaN max_log2FC for non-significant genes.
# ------------------------------------------------------------
deg_all_for_merge = tcell_deg_all_df.copy()
deg_all_for_merge["Gene"] = deg_all_for_merge["Gene"].astype(str)
deg_all_for_merge["Gene_upper"] = deg_all_for_merge["Gene"].str.upper()
deg_all_for_merge["PerturbGene"] = deg_all_for_merge["PerturbGene"].astype(str)

deg_all_for_merge["Significant_up"] = (
    (deg_all_for_merge["log2FC"] > Tcell_DEG_LOG2FC_THRESHOLD)
    & (deg_all_for_merge["pvalue"] < Tcell_DEG_PVALUE_THRESHOLD)
)

# For each target gene, find the perturbation gene giving the maximum log2FC
idx_max_log2fc = (
    deg_all_for_merge
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["log2FC"])
    .groupby("Gene_upper")["log2FC"]
    .idxmax()
)

deg_best_log2fc_df = deg_all_for_merge.loc[
    idx_max_log2fc,
    ["Gene_upper", "Gene", "PerturbGene", "log2FC", "pvalue"]
].copy()

deg_best_log2fc_df = deg_best_log2fc_df.rename(columns={
    "Gene": "Gene_from_DEG",
    "PerturbGene": "perturb_gene_with_max_log2FC",
    "log2FC": "max_log2FC",
    "pvalue": "pvalue_at_max_log2FC",
})

deg_min_pvalue_df = (
    deg_all_for_merge
    .groupby("Gene_upper", as_index=False)
    .agg(
        min_pvalue=("pvalue", "min"),
        n_perturb_genes_tested=("PerturbGene", "nunique"),
    )
)

deg_sig_summary_df = (
    deg_all_for_merge.loc[deg_all_for_merge["Significant_up"]]
    .groupby("Gene_upper", as_index=False)
    .agg(
        n_perturb_genes_significant=("PerturbGene", "nunique"),
        perturb_genes_significant=("PerturbGene", lambda x: ";".join(sorted(map(str, set(x))))),
    )
)

deg_summary_all_by_gene = (
    deg_best_log2fc_df
    .merge(deg_min_pvalue_df, on="Gene_upper", how="left")
    .merge(deg_sig_summary_df, on="Gene_upper", how="left")
)

deg_summary_all_by_gene["n_perturb_genes_significant"] = (
    deg_summary_all_by_gene["n_perturb_genes_significant"]
    .fillna(0)
    .astype(int)
)

deg_summary_all_by_gene["perturb_genes_significant"] = (
    deg_summary_all_by_gene["perturb_genes_significant"]
    .fillna("")
)

deg_summary_all_by_gene["Significant_Tcell_DEG_up"] = (
    deg_summary_all_by_gene["n_perturb_genes_significant"] > 0
)

deg_summary_all_by_gene.to_csv(
    tcell_deg_outdir / f"{MI_RECEIVER_LOADING_MIOI}_all_gene_DEG_summary_for_loading_merge.csv",
    index=False,
)

# display(deg_summary_all_by_gene.head(20))


# ------------------------------------------------------------
# Get MI17 sum-normalized receiver loading
# ------------------------------------------------------------
mi17_receiver_loading_norm = _get_sum_normalized_receiver_loading(
    loading_receiver_df=loading_receiver_use_df,
    mi_name=MI_RECEIVER_LOADING_MIOI,
    mi_index=MI_RECEIVER_LOADING_INDEX,
)

# Keep genes present in the expression matrix
expr_gene_set_upper = set(map(str.upper, adata_all_for_deg.var_names.astype(str)))
mi17_receiver_loading_norm = mi17_receiver_loading_norm.loc[
    [g for g in mi17_receiver_loading_norm.index if str(g).upper() in expr_gene_set_upper]
].copy()

loading_plot_df = pd.DataFrame({
    "Gene": mi17_receiver_loading_norm.index.astype(str),
    "MI17_receiver_loading_sum_normalized": mi17_receiver_loading_norm.values.astype(float),
})

loading_plot_df["Gene_upper"] = loading_plot_df["Gene"].astype(str).str.upper()

loading_plot_df = loading_plot_df.merge(
    deg_summary_all_by_gene[
        [
            "Gene_upper",
            "Gene_from_DEG",
            "max_log2FC",
            "pvalue_at_max_log2FC",
            "min_pvalue",
            "n_perturb_genes_tested",
            "n_perturb_genes_significant",
            "perturb_gene_with_max_log2FC",
            "perturb_genes_significant",
            "Significant_Tcell_DEG_up",
        ]
    ],
    on="Gene_upper",
    how="left",
)

# Genes in loading table should mostly be in DEG table.
# Fill remaining missing values only as safety.
loading_plot_df["max_log2FC"] = loading_plot_df["max_log2FC"].fillna(0.0)
loading_plot_df["pvalue_at_max_log2FC"] = loading_plot_df["pvalue_at_max_log2FC"].fillna(1.0)
loading_plot_df["min_pvalue"] = loading_plot_df["min_pvalue"].fillna(1.0)
loading_plot_df["n_perturb_genes_tested"] = loading_plot_df["n_perturb_genes_tested"].fillna(0).astype(int)
loading_plot_df["n_perturb_genes_significant"] = loading_plot_df["n_perturb_genes_significant"].fillna(0).astype(int)
loading_plot_df["perturb_gene_with_max_log2FC"] = loading_plot_df["perturb_gene_with_max_log2FC"].fillna("")
loading_plot_df["perturb_genes_significant"] = loading_plot_df["perturb_genes_significant"].fillna("")
loading_plot_df["Significant_Tcell_DEG_up"] = loading_plot_df["Significant_Tcell_DEG_up"].fillna(False).astype(bool)

loading_plot_df = loading_plot_df.drop(columns=["Gene_upper"])

loading_plot_df = loading_plot_df.sort_values(
    "MI17_receiver_loading_sum_normalized",
    ascending=False,
).reset_index(drop=True)

loading_plot_df.to_csv(
    tcell_deg_outdir / f"{MI_RECEIVER_LOADING_MIOI}_receiver_loading_sum_normalized_with_ALL_Tcell_DEG_stats.csv",
    index=False,
)

print("Number of NaN max_log2FC after merge:", loading_plot_df["max_log2FC"].isna().sum())
# display(loading_plot_df.head(20))


# ------------------------------------------------------------
# Test whether significant T-cell DEGs have higher MI17 receiver loading
# ------------------------------------------------------------
vals_sig = loading_plot_df.loc[
    loading_plot_df["Significant_Tcell_DEG_up"],
    "MI17_receiver_loading_sum_normalized",
].dropna().values

vals_nonsig = loading_plot_df.loc[
    ~loading_plot_df["Significant_Tcell_DEG_up"],
    "MI17_receiver_loading_sum_normalized",
].dropna().values

if len(vals_sig) > 0 and len(vals_nonsig) > 0:
    stat_loading, pval_loading = mannwhitneyu(
        vals_sig,
        vals_nonsig,
        alternative="greater",
    )
else:
    stat_loading, pval_loading = np.nan, np.nan

print(f"Significant DEG genes in loading table: {len(vals_sig)}")
print(f"Non-significant genes in loading table: {len(vals_nonsig)}")
print(f"Mann-Whitney U test, significant DEG loading > others: P = {pval_loading}")


# ------------------------------------------------------------
# Stem plot
# ------------------------------------------------------------
if TOP_N_STEM_GENES is None:
    stem_plot_df = loading_plot_df.copy()
else:
    stem_plot_df = loading_plot_df.head(TOP_N_STEM_GENES).copy()

stem_plot_df = stem_plot_df.reset_index(drop=True)

x = np.arange(stem_plot_df.shape[0])
y = stem_plot_df["MI17_receiver_loading_sum_normalized"].to_numpy(dtype=float)

colors = np.where(
    stem_plot_df["Significant_Tcell_DEG_up"].to_numpy(),
    "#D62728",
    "#9E9E9E",
)

apply_publication_style(font_size=8)

fig_width = max(8.0, 0.20 * stem_plot_df.shape[0] + 2.0)
fig, ax = plt.subplots(figsize=(fig_width, 3.0))

for xi, yi, ci in zip(x, y, colors):
    ax.vlines(
        xi,
        0,
        yi,
        color=ci,
        linewidth=1.2,
        alpha=0.95,
    )
    ax.scatter(
        xi,
        yi,
        color=ci,
        s=15,
        zorder=3,
        edgecolor="none",
    )

ax.axhline(0, color="black", linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(
    stem_plot_df["Gene"].astype(str).tolist(),
    rotation=90,
    ha="center",
)

ax.set_ylabel(f"{MI_RECEIVER_LOADING_MIOI} receiver loading\nsum-normalized per gene")
ax.set_xlabel("Receiver target genes")

title_extra = ""
if np.isfinite(pval_loading):
    title_extra = f" | MWU P={pval_loading:.2e}"

ax.set_title(
    f"{MI_RECEIVER_LOADING_MIOI} receiver loading vs T-cell DEG status{title_extra}",
    pad=5,
)

legend_handles = [
    Line2D(
        [0],
        [0],
        color="#D62728",
        marker="o",
        linestyle="-",
        linewidth=1.2,
        markersize=4,
        label=f"Up DEG: log2FC>{Tcell_DEG_LOG2FC_THRESHOLD}, P<{Tcell_DEG_PVALUE_THRESHOLD}",
    ),
    Line2D(
        [0],
        [0],
        color="#9E9E9E",
        marker="o",
        linestyle="-",
        linewidth=1.2,
        markersize=4,
        label="Other genes",
    ),
]

ax.legend(
    handles=legend_handles,
    frameon=False,
    loc="upper right",
    fontsize=7,
)

ax.tick_params(axis="both", which="both", direction="out")
sns.despine(ax=ax, top=True, right=True)

fig.tight_layout()

stem_prefix = (
    tcell_deg_outdir
    / f"Stemplot_{MI_RECEIVER_LOADING_MIOI}_receiver_loading_sum_normalized_Tcell_DEG_highlight"
)

fig.savefig(f"{stem_prefix}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"{stem_prefix}.pdf", bbox_inches="tight")

plt.show()
plt.close(fig)

display(stem_plot_df)

In [ ]:
# ============================================================
# GSEA-style enrichment of T-cell response genes by MI-17 receiver loading.
# Test whether significant T-cell up-DEG genes are enriched
# at the top of MI17 receiver loading-ranked target genes.
#
# Ranked list:
#   all genes ranked by MI17 receiver loading sum-normalized value
#
# Gene set:
#   genes with Significant_Tcell_DEG_up == True
#
# Output:
#   1. GSEA running enrichment score plot
#   2. Ranked gene table with hit information
#   3. Permutation-based ES / NES / P-value summary
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------
GSEA_N_PERMUTATIONS = 10000
GSEA_WEIGHT = 1.0
GSEA_RANDOM_SEED = 123
GSEA_MIN_GENESET_SIZE = 5

loading_value_col = "MI17_receiver_loading_sum_normalized"

gsea_outdir = Path(tcell_deg_outdir) / f"{MI_RECEIVER_LOADING_MIOI}_receiver_loading_GSEA"
gsea_outdir.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def _compute_gsea_running_es(ranked_metric, hit_mask, weight=1.0):
    """
    ranked_metric: 1D np.array, ordered from high to low.
    hit_mask: boolean np.array, same length.
    weight: GSEA weight. weight=1 uses weighted enrichment.
    """
    ranked_metric = np.asarray(ranked_metric, dtype=float)
    hit_mask = np.asarray(hit_mask, dtype=bool)

    n = ranked_metric.shape[0]
    n_hit = int(hit_mask.sum())
    n_miss = n - n_hit

    if n_hit == 0:
        raise ValueError("Gene set has no hits in ranked list.")
    if n_miss == 0:
        raise ValueError("All ranked genes are hits; GSEA is not meaningful.")

    # Use abs(metric)^weight for hit weights.
    # If all hit weights are zero, fall back to unweighted hit weights.
    hit_weights = np.abs(ranked_metric) ** weight
    hit_weights_sum = hit_weights[hit_mask].sum()

    if hit_weights_sum <= 0 or not np.isfinite(hit_weights_sum):
        hit_weights = np.ones_like(ranked_metric, dtype=float)
        hit_weights_sum = hit_weights[hit_mask].sum()

    p_hit = np.where(hit_mask, hit_weights / hit_weights_sum, 0.0)
    p_miss = np.where(~hit_mask, 1.0 / n_miss, 0.0)

    running_es = np.cumsum(p_hit - p_miss)

    max_es = np.nanmax(running_es)
    min_es = np.nanmin(running_es)

    if abs(max_es) >= abs(min_es):
        es = max_es
        es_index = int(np.nanargmax(running_es))
    else:
        es = min_es
        es_index = int(np.nanargmin(running_es))

    return running_es, es, es_index


def _permute_gsea_es(ranked_metric, n_hits, n_perm=10000, weight=1.0, seed=123):
    rng = np.random.default_rng(seed)
    n = len(ranked_metric)
    null_es = np.zeros(n_perm, dtype=float)

    for i in range(n_perm):
        hit_idx = rng.choice(n, size=n_hits, replace=False)
        hit_mask_perm = np.zeros(n, dtype=bool)
        hit_mask_perm[hit_idx] = True

        _, es_perm, _ = _compute_gsea_running_es(
            ranked_metric=ranked_metric,
            hit_mask=hit_mask_perm,
            weight=weight,
        )
        null_es[i] = es_perm

    return null_es


def _normalize_es(es, null_es):
    null_es = np.asarray(null_es, dtype=float)

    if es >= 0:
        pos_null = null_es[null_es >= 0]
        denom = np.mean(pos_null) if len(pos_null) > 0 else np.nan
    else:
        neg_null = np.abs(null_es[null_es < 0])
        denom = np.mean(neg_null) if len(neg_null) > 0 else np.nan

    if denom is None or not np.isfinite(denom) or denom == 0:
        return np.nan

    return es / denom


def _gsea_pvalue(es, null_es):
    null_es = np.asarray(null_es, dtype=float)

    if es >= 0:
        return (np.sum(null_es >= es) + 1) / (np.sum(null_es >= 0) + 1)
    else:
        return (np.sum(null_es <= es) + 1) / (np.sum(null_es < 0) + 1)


# ------------------------------------------------------------
# Build ranked table
# ------------------------------------------------------------
if "loading_plot_df" not in globals():
    raise NameError(
        "loading_plot_df is not found. Please run the previous MI17 receiver loading cell first."
    )

required_cols = [
    "Gene",
    loading_value_col,
    "Significant_Tcell_DEG_up",
]

missing_cols = [c for c in required_cols if c not in loading_plot_df.columns]
if len(missing_cols) > 0:
    raise KeyError(f"loading_plot_df is missing required columns: {missing_cols}")

gsea_rank_df = loading_plot_df.copy()
gsea_rank_df["Gene"] = gsea_rank_df["Gene"].astype(str)
gsea_rank_df[loading_value_col] = pd.to_numeric(
    gsea_rank_df[loading_value_col],
    errors="coerce",
)

gsea_rank_df = (
    gsea_rank_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=[loading_value_col])
    .copy()
)

# Collapse duplicated genes if any, keeping the largest loading value
gsea_rank_df = (
    gsea_rank_df
    .sort_values(loading_value_col, ascending=False)
    .drop_duplicates(subset=["Gene"], keep="first")
    .reset_index(drop=True)
)

# Rank from high to low loading
gsea_rank_df = gsea_rank_df.sort_values(
    loading_value_col,
    ascending=False,
).reset_index(drop=True)

gsea_rank_df["Rank"] = np.arange(1, gsea_rank_df.shape[0] + 1)

# Gene set = significant up-DEG genes
gsea_rank_df["IsHit"] = gsea_rank_df["Significant_Tcell_DEG_up"].astype(bool)

n_total_genes = gsea_rank_df.shape[0]
n_hits = int(gsea_rank_df["IsHit"].sum())

print(f"Total ranked genes: {n_total_genes}")
print(f"Significant T-cell up-DEG genes in ranked list: {n_hits}")

if n_hits < GSEA_MIN_GENESET_SIZE:
    raise ValueError(
        f"Gene set too small for GSEA-style analysis: n_hits={n_hits}. "
        f"Minimum required: {GSEA_MIN_GENESET_SIZE}."
    )

ranked_metric = gsea_rank_df[loading_value_col].to_numpy(dtype=float)
hit_mask = gsea_rank_df["IsHit"].to_numpy(dtype=bool)


# ------------------------------------------------------------
# Compute observed ES and permutation null
# ------------------------------------------------------------
running_es, observed_es, es_peak_index = _compute_gsea_running_es(
    ranked_metric=ranked_metric,
    hit_mask=hit_mask,
    weight=GSEA_WEIGHT,
)

null_es = _permute_gsea_es(
    ranked_metric=ranked_metric,
    n_hits=n_hits,
    n_perm=GSEA_N_PERMUTATIONS,
    weight=GSEA_WEIGHT,
    seed=GSEA_RANDOM_SEED,
)

nes = _normalize_es(observed_es, null_es)
pval = _gsea_pvalue(observed_es, null_es)

gsea_rank_df["RunningES"] = running_es
gsea_rank_df["LeadingEdge"] = False

if observed_es >= 0:
    gsea_rank_df.loc[:es_peak_index, "LeadingEdge"] = gsea_rank_df.loc[:es_peak_index, "IsHit"]
else:
    gsea_rank_df.loc[es_peak_index:, "LeadingEdge"] = gsea_rank_df.loc[es_peak_index:, "IsHit"]

leading_edge_genes = gsea_rank_df.loc[gsea_rank_df["LeadingEdge"], "Gene"].tolist()

gsea_summary_df = pd.DataFrame({
    "MI": [MI_RECEIVER_LOADING_MIOI],
    "RankingMetric": [loading_value_col],
    "GeneSet": ["Tcell_up_DEG_union"],
    "N_ranked_genes": [n_total_genes],
    "N_gene_set_hits": [n_hits],
    "ES": [observed_es],
    "NES": [nes],
    "Pvalue": [pval],
    "N_permutations": [GSEA_N_PERMUTATIONS],
    "Weight": [GSEA_WEIGHT],
    "PeakRank": [es_peak_index + 1],
    "N_leading_edge_genes": [len(leading_edge_genes)],
    "LeadingEdgeGenes": [";".join(leading_edge_genes)],
})

gsea_rank_df.to_csv(
    gsea_outdir / f"{MI_RECEIVER_LOADING_MIOI}_receiver_loading_ranked_GSEA_table.csv",
    index=False,
)

gsea_summary_df.to_csv(
    gsea_outdir / f"{MI_RECEIVER_LOADING_MIOI}_receiver_loading_GSEA_summary.csv",
    index=False,
)

display(gsea_summary_df)
display(gsea_rank_df.head(20))


# ------------------------------------------------------------
# Plot GSEA-style enrichment curve
# ------------------------------------------------------------


In [ ]:
apply_publication_style(font_size=8)

fig_height = 3.6
fig_width = 5.0

fig = plt.figure(figsize=(fig_width, fig_height))
gs = fig.add_gridspec(
    nrows=3,
    ncols=1,
    height_ratios=[2.2, 0.35, 0.8],
    hspace=0.08,
)

ax_es = fig.add_subplot(gs[0])
ax_hits = fig.add_subplot(gs[1], sharex=ax_es)
ax_metric = fig.add_subplot(gs[2], sharex=ax_es)

x = np.arange(1, n_total_genes + 1)

# Running ES curve
ax_es.plot(
    x,
    running_es,
    color="#D62728",
    linewidth=1.4,
)

ax_es.axhline(
    0,
    color="black",
    linewidth=0.6,
    linestyle="-",
)

ax_es.axvline(
    es_peak_index + 1,
    color="0.45",
    linewidth=0.8,
    linestyle="--",
)

ax_es.scatter(
    es_peak_index + 1,
    running_es[es_peak_index],
    color="#D62728",
    s=18,
    zorder=3,
)

ax_es.set_ylabel("Running ES")

ax_es.set_title(
    (
        f"{MI_RECEIVER_LOADING_MIOI} receiver loading GSEA\n"
        f"ES={observed_es:.3f}, NES={nes:.3f}, P={pval:.2e}, hits={n_hits}/{n_total_genes}"
    ),
    pad=5,
)

sns.despine(ax=ax_es, top=True, right=True)
ax_es.tick_params(axis="x", labelbottom=False, direction="out")
ax_es.tick_params(axis="y", direction="out")


# Hit barcode
hit_positions = x[hit_mask]

for hp in hit_positions:
    ax_hits.axvline(
        hp,
        ymin=0.05,
        ymax=0.95,
        color="#D62728",
        linewidth=0.6,
        alpha=0.85,
    )

ax_hits.set_ylim(0, 1)
ax_hits.set_yticks([])
ax_hits.set_ylabel("Hits", rotation=0, ha="right", va="center")
sns.despine(ax=ax_hits, top=True, right=True, left=True, bottom=True)
ax_hits.tick_params(axis="x", labelbottom=False, bottom=False)


# Ranked loading metric
ax_metric.fill_between(
    x,
    ranked_metric,
    0,
    color="0.65",
    alpha=0.85,
    linewidth=0,
)

ax_metric.axhline(
    0,
    color="black",
    linewidth=0.5,
)

ax_metric.set_ylabel("Loading")
ax_metric.set_xlabel("Genes ranked by MI17 receiver loading")

sns.despine(ax=ax_metric, top=True, right=True)
ax_metric.tick_params(axis="both", direction="out")

fig.tight_layout()

gsea_plot_prefix = (
    gsea_outdir
    / f"GSEA_{MI_RECEIVER_LOADING_MIOI}_receiver_loading_Tcell_upDEG_enrichment"
)

fig.savefig(f"{gsea_plot_prefix}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"{gsea_plot_prefix}.pdf", bbox_inches="tight")

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Optional: plot null ES distribution
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(3.0, 2.4))

ax.hist(
    null_es,
    bins=40,
    color="0.7",
    edgecolor="white",
    linewidth=0.4,
)

ax.axvline(
    observed_es,
    color="#D62728",
    linewidth=1.2,
    linestyle="-",
    label=f"Observed ES={observed_es:.3f}",
)

ax.set_xlabel("Null ES")
ax.set_ylabel("Frequency")
ax.set_title(f"Permutation null | P={pval:.2e}", pad=5)

ax.legend(frameon=False, fontsize=7)

ax.tick_params(axis="both", which="both", direction="out")
sns.despine(ax=ax, top=True, right=True)

fig.tight_layout()

null_plot_prefix = (
    gsea_outdir
    / f"NullES_{MI_RECEIVER_LOADING_MIOI}_receiver_loading_Tcell_upDEG_enrichment"
)

fig.savefig(f"{null_plot_prefix}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"{null_plot_prefix}.pdf", bbox_inches="tight")

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Compare MI-17 receiver-loading percentile ranks between:
#   Group 1: significant T-cell DEG-up genes
#   Group 2: other genes
#
# Rank definition:
#   LoadingRank01 = (rank - 1) / (n_genes - 1)
#   Range: 0 to 1
#   Larger rank means larger MI17 receiver loading value.
#
# Use loading_plot_df rather than stem_plot_df because the latter may contain
# only the top genes selected for display.
# ============================================================

from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# Prepare rank dataframe using all genes in loading_plot_df
# ------------------------------------------------------------
loading_rank_df = loading_plot_df.copy()

loading_value_col = "MI17_receiver_loading_sum_normalized"

loading_rank_df = loading_rank_df.replace([np.inf, -np.inf], np.nan)
loading_rank_df = loading_rank_df.dropna(subset=[loading_value_col]).copy()

n_genes_rank = loading_rank_df.shape[0]

if n_genes_rank <= 1:
    loading_rank_df["LoadingRank01"] = 1.0
else:
    # ascending=True: smallest loading gets rank 1, largest gets rank n
    raw_rank = loading_rank_df[loading_value_col].rank(
        method="average",
        ascending=True,
    )
    loading_rank_df["LoadingRank01"] = (raw_rank - 1) / (n_genes_rank - 1)

loading_rank_df["Group"] = np.where(
    loading_rank_df["Significant_Tcell_DEG_up"].astype(bool),
    "Up DEG genes",
    "Other genes",
)

group_order = [
    "Up DEG genes",
    "Other genes",
]

loading_rank_df["Group"] = pd.Categorical(
    loading_rank_df["Group"],
    categories=group_order,
    ordered=True,
)

loading_rank_df = loading_rank_df.sort_values(
    ["Group", "LoadingRank01"],
    ascending=[True, False],
).reset_index(drop=True)

loading_rank_df.to_csv(
    tcell_deg_outdir / f"{MI_RECEIVER_LOADING_MIOI}_receiver_loading_rank01_DEG_group_comparison.csv",
    index=False,
)

display(loading_rank_df.head(20))


# ------------------------------------------------------------
# Statistical comparison
# Hypothesis: significant up-DEG genes have higher loading ranks
# ------------------------------------------------------------
vals_deg = loading_rank_df.loc[
    loading_rank_df["Group"] == "Up DEG genes",
    "LoadingRank01",
].dropna().values

vals_other = loading_rank_df.loc[
    loading_rank_df["Group"] == "Other genes",
    "LoadingRank01",
].dropna().values

if len(vals_deg) > 0 and len(vals_other) > 0:
    stat_rank, pval_rank = mannwhitneyu(
        vals_deg,
        vals_other,
        alternative="greater",
    )
else:
    stat_rank, pval_rank = np.nan, np.nan

print(f"Up DEG genes n = {len(vals_deg)}")
print(f"Other genes n = {len(vals_other)}")
print(f"Mann-Whitney U statistic = {stat_rank}")
print(f"One-sided P value, Up DEG rank > Other rank = {pval_rank}")


# ------------------------------------------------------------
# Boxplot + jittered points
# ------------------------------------------------------------
apply_publication_style(font_size=8)

fig, ax = plt.subplots(figsize=(2.5, 2.8))

rank_palette = {
    "Up DEG genes": "#D62728",
    "Other genes": "#9E9E9E",
}

sns.boxplot(
    data=loading_rank_df,
    x="Group",
    y="LoadingRank01",
    order=group_order,
    palette=rank_palette,
    width=0.55,
    showfliers=False,
    linewidth=0.9,
    ax=ax,
)

sns.stripplot(
    data=loading_rank_df,
    x="Group",
    y="LoadingRank01",
    order=group_order,
    palette=rank_palette,
    size=3.2,
    jitter=0.18,
    alpha=0.75,
    edgecolor="none",
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel(f"{MI_RECEIVER_LOADING_MIOI} receiver loading rank")
ax.set_ylim(-0.03, 1.12)

ax.set_xticklabels(
    [
        f"Up DEG\nn={len(vals_deg)}",
        f"Other\nn={len(vals_other)}",
    ],
    rotation=0,
)

# ------------------------------------------------------------
# Add P-value annotation
# ------------------------------------------------------------
if np.isfinite(pval_rank):
    y0 = 1.02
    y1 = 1.06

    ax.plot(
        [0, 0, 1, 1],
        [y0, y1, y1, y0],
        color="black",
        linewidth=0.8,
    )

    ax.text(
        0.5,
        1.075,
        f"MWU P = {pval_rank:.2e}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

ax.tick_params(axis="both", which="both", direction="out")
sns.despine(ax=ax, top=True, right=True)

fig.tight_layout()

rank_boxplot_prefix = (
    tcell_deg_outdir
    / f"Boxplot_{MI_RECEIVER_LOADING_MIOI}_receiver_loading_rank01_DEG_vs_other"
)

fig.savefig(f"{rank_boxplot_prefix}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"{rank_boxplot_prefix}.pdf", bbox_inches="tight")

plt.show()
plt.close(fig)

## 13. Run GO enrichment for the selected MI's top regulator and target genes

In [ ]:
from go_enrichment import run_enrichment

# Bind GO inputs to the same selected checkpoint used for perturbation inference.
# The full workflow reuses its existing, unnormalized baseline MI activities.
go_tables = run_enrichment(
    results_dir=RESULTS_DIR,
    processed_root=PROCESSED_ROOT,
    output_dir=base_run_dir,
    model=globals().get("model"),
    processed=globals().get("processed"),
    factor_envir=globals().get("core_results", {}).get("Factor_envir"),
)
df_regulatory_enrich = go_tables["sender"]
df_target_enrich = go_tables["receiver"]
display(df_regulatory_enrich.head(10))
display(df_target_enrich.head(10))


## 14. Save a compact session manifest

This makes it easier for downstream notebooks to locate the main analysis outputs generated in this Part 1 run.


In [ ]:
manifest = {
    "processed_data_dir": str(processed_data_dir),
    "base_run_dir": str(base_run_dir),
    "model_dir": str(base_model_dir),
    "summary_path": str(base_run_dir / "Part1_insilico_spatial_perturbation_summary.csv"),
    "golden_lfc_path": str(base_run_dir / "Part1_LFC_Tcell_golden.csv"),
    "predicted_lfc_path": str(base_run_dir / "Part1_LFC_Tcell_predicted.csv"),
    "mi_change_pc2t_path": str(base_run_dir / "MI_change_perturbcancerTOTcell_df.csv"),
    "mi_change_t2pc_path": str(base_run_dir / "MI_change_TcellTOTperturbcancer_df.csv"),
}

with open(base_run_dir / "part1_output_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

manifest


## 15. Exploratory in silico MI-17 top-LR perturbation and GO-program response

This section performs an additional **in silico ligand--receptor perturbation experiment** for the selected `MI17` melanoma-cell → T-cell communication program.

The analysis:
1. selects the top MI-17 LR pairs;
2. perturbs ligand genes in melanoma sender cells and receptor genes in T-cell receivers;
3. includes knockout, 50% expression scaling, and a count-matched random-LR baseline;
4. measures the pre/post perturbation changes of the sender-enriched and receiver-enriched GO-program module scores.

In [ ]:
# ==============================================================
# MI-17 top LR-pair KO setup and helper functions
# --------------------------------------------------------------
# Sender: melanoma/cancer cells
# Receiver: T cells
# MI: MI17
#
# Perturbation modes:
#   - KO-TopMI17: selected ligand genes in sender + receptor genes in receiver -> 0
#   - 50%-TopMI17: selected ligand/receptor genes scaled to 50%
#   - KO-Random: matched random LR genes -> 0
# ==============================================================

import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, wilcoxon

MI17_KO_MIOI = "MI17"
MI17_KO_MI_INDEX = int(MI17_KO_MIOI.replace("MI", "").replace("-", "")) - 1
MI17_KO_TOP_N_LR = 5
MI17_KO_MI_STRENGTH_THRESHOLD = 0.0
MI17_KO_RANDOM_SEED = 123

MI17_KO_SENDER_MATCH_KEYWORDS = ["melanoma", "cancer", "tumor", "malignant"]
MI17_KO_RECEIVER_MATCH_KEYWORDS = ["t cell", "t_cell", "t-cell", "tcell", " t "]

mi17_lrko_outdir = Path(base_run_dir) / f"InSilico_LRKO_{MI17_KO_MIOI}_melanoma_to_Tcell_GOprograms"
mi17_lrko_outdir.mkdir(parents=True, exist_ok=True)


def _as_numpy(x):
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _edge_index_to_2col(edge_index):
    edge_index = _as_numpy(edge_index)
    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2-dimensional, got shape {edge_index.shape}")
    if edge_index.shape[0] == 2 and edge_index.shape[1] != 2:
        edge_index = edge_index.T
    if edge_index.shape[1] != 2:
        raise ValueError(f"edge_index must have two columns after conversion, got shape {edge_index.shape}")
    return edge_index.astype(int, copy=False)




def _get_data_edge_index(data_cur):
    edge_index = data_cur["edge_index"] if isinstance(data_cur, dict) else data_cur.edge_index
    return _edge_index_to_2col(edge_index)


def _align_edge_index_to_factor_rows(edge_index_cur, n_factor_rows, slice_index=None):
    """
    Return one sender/receiver edge per row of factor_cur.

    In some processed bundles, data_cur.edge_index stores LR-expanded edges
    (for example, each cell-cell edge repeated for each LR feature), whereas
    Factor_envir is stored at the cell-cell edge level. In that case, raw
    edge_index can have k times more rows than Factor_envir. This helper
    collapses the LR-expanded edge_index back to the factor-level cell-cell
    edge_index in a deterministic way.
    """
    edge_index_cur = _edge_index_to_2col(edge_index_cur)
    n_edge_rows = int(edge_index_cur.shape[0])
    n_factor_rows = int(n_factor_rows)
    prefix = f"slice {slice_index}: " if slice_index is not None else ""

    if n_edge_rows == n_factor_rows:
        return edge_index_cur

    if n_factor_rows <= 0:
        raise ValueError(f"{prefix}Factor_envir has no rows.")

    # Common case 1:
    # edge_index = [all cell-cell edges for LR1, all cell-cell edges for LR2, ...]
    # Then each block of n_factor_rows is identical; use the first block.
    if n_edge_rows % n_factor_rows == 0:
        repeat_factor = n_edge_rows // n_factor_rows
        first_block = edge_index_cur[:n_factor_rows, :]
        block_layout_ok = True
        for block_i in range(1, min(repeat_factor, 5)):
            start = block_i * n_factor_rows
            stop = start + n_factor_rows
            if not np.array_equal(first_block, edge_index_cur[start:stop, :]):
                block_layout_ok = False
                break
        if block_layout_ok:
            print(
                f"{prefix}edge_index has {n_edge_rows} rows but Factor_envir has {n_factor_rows}; "
                f"detected {repeat_factor} repeated LR blocks, using first block as cell-cell edges."
            )
            return first_block

        # Common case 2:
        # edge_index = [edge1 repeated k times, edge2 repeated k times, ...]
        # Then every contiguous group of repeat_factor rows has identical sender/receiver.
        grouped = edge_index_cur.reshape(n_factor_rows, repeat_factor, 2)
        if np.all(grouped == grouped[:, :1, :]):
            print(
                f"{prefix}edge_index has {n_edge_rows} rows but Factor_envir has {n_factor_rows}; "
                f"detected {repeat_factor} repeats per cell-cell edge, taking one edge per group."
            )
            return grouped[:, 0, :]

    # General fallback: preserve first occurrence of each sender-receiver pair.
    # This is useful when edge_index is LR-expanded but not arranged in clean blocks.
    edge_df_tmp = pd.DataFrame(edge_index_cur, columns=["sender", "receiver"])
    dedup = edge_df_tmp.drop_duplicates(keep="first").to_numpy(dtype=int)
    if dedup.shape[0] == n_factor_rows:
        print(
            f"{prefix}edge_index has {n_edge_rows} rows but Factor_envir has {n_factor_rows}; "
            "using first occurrence of each unique sender-receiver pair."
        )
        return dedup

    raise ValueError(
        f"{prefix}Cannot align edge_index rows ({n_edge_rows}) to Factor_envir rows ({n_factor_rows}). "
        "This usually means Factor_envir and edge_index come from different edge spaces. "
        "Inspect data_cur keys and factor_envir_list_for_lrko source."
    )


def _to_device_graph(data_cur, device):
    return data_cur.to(device) if hasattr(data_cur, "to") else data_cur


def _infer_celltype_column(adata_list, preferred=("celltype", "cell_type", "cell.types", "cell_class", "celltype_major", "cell_type_major")):
    candidate_scores = {}
    for col in preferred:
        score = 0
        for adata in adata_list:
            if col in adata.obs.columns:
                values = adata.obs[col].astype(str).str.lower()
                has_sender = values.str.contains("melanoma|cancer|tumor|malignant", regex=True).any()
                has_receiver = values.str.contains("t cell|t_cell|t-cell|tcell", regex=True).any()
                score += int(has_sender) + int(has_receiver)
        if score > 0:
            candidate_scores[col] = score

    if len(candidate_scores) > 0:
        return sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[0][0]

    # Last fallback: any categorical/object column that contains T cells and cancer-like cells.
    for adata in adata_list:
        for col in adata.obs.columns:
            values = adata.obs[col].astype(str).str.lower()
            if (
                values.str.contains("t cell|t_cell|t-cell|tcell", regex=True).any()
                and values.str.contains("melanoma|cancer|tumor|malignant", regex=True).any()
            ):
                return col

    raise KeyError(
        "Could not infer a cell-type column containing both melanoma/cancer-like cells and T cells. "
        "Please set `MI17_KO_CELLTYPE_COL` manually before running this cell."
    )


def _match_celltype_values(values, keywords):
    values = pd.Series(values).astype(str)
    values_low = values.str.lower()
    mask = np.zeros(len(values), dtype=bool)

    for kw in keywords:
        kw_low = str(kw).lower().strip()
        if kw_low == " t ":
            mask |= values_low.eq("t")
        elif kw_low in {"t cell", "t_cell", "t-cell", "tcell"}:
            mask |= values_low.str.contains(r"\bt[\s_-]*cell\b|^tcell$|^t_cell$|^t-cell$", regex=True)
        else:
            mask |= values_low.str.contains(re.escape(kw_low), regex=True)

    return mask


def _get_factor_envir_list_for_lrko():
    n_edges_per_slice = [
        _get_data_edge_index(data_cur).shape[0]
        for data_cur in processed.spidernet_data
    ]

    if "Factor_envir_list" in globals():
        fac_list = [np.asarray(x, dtype=float) for x in Factor_envir_list]
        if len(fac_list) == len(n_edges_per_slice):
            return fac_list

    if "core_results" in globals() and isinstance(core_results, dict) and "Factor_envir" in core_results:
        fac_all = np.asarray(core_results["Factor_envir"], dtype=float)
    elif "Factor_envir_use" in globals():
        fac_all = np.asarray(Factor_envir_use, dtype=float)
    else:
        raise KeyError("Could not find Factor_envir_list, core_results['Factor_envir'], or Factor_envir_use.")

    if fac_all.shape[0] != int(np.sum(n_edges_per_slice)):
        raise ValueError(
            f"Global Factor_envir has {fac_all.shape[0]} rows, but processed edges sum to {np.sum(n_edges_per_slice)}."
        )

    fac_list = []
    start = 0
    for n_edges in n_edges_per_slice:
        fac_list.append(fac_all[start:start + n_edges, :])
        start += n_edges
    return fac_list


def _resolve_gene_indices(gene_names, gene_list):
    gene_names = pd.Index([str(g) for g in gene_names])
    upper_to_idx = {}
    for i, g in enumerate(gene_names):
        upper_to_idx.setdefault(str(g).upper(), i)

    indices = []
    found = []
    missing = []

    for gene in gene_list:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue
        if gene in gene_names:
            idx = gene_names.get_loc(gene)
            if isinstance(idx, slice):
                idx = idx.start
            elif isinstance(idx, np.ndarray):
                idx = np.where(idx)[0][0]
            indices.append(int(idx))
            found.append(gene)
        elif gene.upper() in upper_to_idx:
            indices.append(int(upper_to_idx[gene.upper()]))
            found.append(gene)
        else:
            missing.append(gene)

    # Keep first occurrence while preserving order
    seen = set()
    indices_unique = []
    found_unique = []
    for idx, gene in zip(indices, found):
        if idx not in seen:
            seen.add(idx)
            indices_unique.append(idx)
            found_unique.append(gene)

    return np.asarray(indices_unique, dtype=int), found_unique, missing


def _flatten_lr_side(lr_side):
    genes = []
    for item in lr_side:
        if isinstance(item, (list, tuple, set)):
            genes.extend([str(x) for x in item])
        else:
            genes.extend(str(item).replace("(", "").replace(")", "").split("+"))
    genes = [g.strip() for g in genes if str(g).strip() != ""]
    return genes


def _lr_pair_to_name(lr_pair):
    lig = "+".join([str(x) for x in lr_pair[0]])
    rec = "+".join([str(x) for x in lr_pair[1]])
    return f"{lig}->{rec}"


def _select_mi17_top_lr_indices():
    if "LR_index_top" in globals() and len(LR_index_top) > 0 and str(globals().get("MIOI", MI17_KO_MIOI)) == MI17_KO_MIOI:
        top_indices = list(map(int, LR_index_top))[:MI17_KO_TOP_N_LR]
        source = "existing LR_index_top"

    else:
        loading = loading_LR_use.copy() if isinstance(loading_LR_use, pd.DataFrame) else pd.DataFrame(loading_LR_use)

        if MI17_KO_MIOI in loading.index:
            vals = pd.to_numeric(loading.loc[MI17_KO_MIOI], errors="coerce")
        elif MI17_KO_MIOI in loading.columns:
            vals = pd.to_numeric(loading[MI17_KO_MIOI], errors="coerce")
        else:
            # Fall back to zero-based MI index as row if possible.
            vals = pd.to_numeric(loading.iloc[MI17_KO_MI_INDEX], errors="coerce")

        vals = vals.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        top_indices = vals.sort_values(ascending=False).head(MI17_KO_TOP_N_LR).index.tolist()
        # Convert index labels to integer positions if needed.
        top_indices = [int(x) if str(x).isdigit() else loading.columns.get_loc(x) for x in top_indices]
        source = "loading_LR_use"

    top_indices = [idx for idx in top_indices if 0 <= idx < len(processed.lr_list)]
    if len(top_indices) == 0:
        raise ValueError("No valid MI17 top LR indices were selected.")

    return top_indices, source


def run_model_on_lrko_graph(
    graph,
    sender_idx,
    receiver_idx,
    ligand_idx,
    receptor_idx,
    ligand_scale=1.0,
    receptor_scale=1.0,
):
    """Run SpiderNet after scaling selected ligand/receptor gene expression in selected sender/receiver cells."""
    g = graph.clone()

    sender_idx = np.asarray(sender_idx, dtype=int)
    receiver_idx = np.asarray(receiver_idx, dtype=int)
    ligand_idx = np.asarray(ligand_idx, dtype=int)
    receptor_idx = np.asarray(receptor_idx, dtype=int)

    x_device = g.x.device

    with torch.no_grad():
        if sender_idx.size > 0 and ligand_idx.size > 0:
            rows = torch.as_tensor(sender_idx, dtype=torch.long, device=x_device)
            cols = torch.as_tensor(ligand_idx, dtype=torch.long, device=x_device)
            g.x[rows[:, None], cols[None, :]] = g.x[rows[:, None], cols[None, :]] * float(ligand_scale)

        if receiver_idx.size > 0 and receptor_idx.size > 0:
            rows = torch.as_tensor(receiver_idx, dtype=torch.long, device=x_device)
            cols = torch.as_tensor(receptor_idx, dtype=torch.long, device=x_device)
            g.x[rows[:, None], cols[None, :]] = g.x[rows[:, None], cols[None, :]] * float(receptor_scale)

        exp_recon, *_ = model(g)

    return exp_recon.detach().cpu().numpy().astype(np.float32)


# -----------------------------
# Select MI17 top LR genes and matched random baseline genes
# -----------------------------
adata_list = processed.adata_list
spidernet_data_list = processed.spidernet_data
factor_envir_list_for_lrko = _get_factor_envir_list_for_lrko()

MI17_KO_CELLTYPE_COL = globals().get(
    "MI17_KO_CELLTYPE_COL",
    _infer_celltype_column(adata_list),
)

gene_names_for_lrko = pd.Index([str(g) for g in adata_list[0].var_names])
mi17_top_lr_indices, mi17_top_lr_source = _select_mi17_top_lr_indices()
mi17_top_lr_pairs = [processed.lr_list[idx] for idx in mi17_top_lr_indices]
mi17_top_lr_pair_names = [_lr_pair_to_name(x) for x in mi17_top_lr_pairs]

ligand_top_genes = sorted(set(_flatten_lr_side([lr[0] for lr in mi17_top_lr_pairs])))
receptor_top_genes = sorted(set(_flatten_lr_side([lr[1] for lr in mi17_top_lr_pairs])))

ligand_top_idx, ligand_top_found, ligand_top_missing = _resolve_gene_indices(gene_names_for_lrko, ligand_top_genes)
receptor_top_idx, receptor_top_found, receptor_top_missing = _resolve_gene_indices(gene_names_for_lrko, receptor_top_genes)

all_ligand_genes = sorted(set(_flatten_lr_side([lr[0] for lr in processed.lr_list])))
all_receptor_genes = sorted(set(_flatten_lr_side([lr[1] for lr in processed.lr_list])))
all_ligand_idx, _, _ = _resolve_gene_indices(gene_names_for_lrko, all_ligand_genes)
all_receptor_idx, _, _ = _resolve_gene_indices(gene_names_for_lrko, all_receptor_genes)

rng = np.random.default_rng(MI17_KO_RANDOM_SEED)
ligand_pool = np.setdiff1d(all_ligand_idx, ligand_top_idx)
receptor_pool = np.setdiff1d(all_receptor_idx, receptor_top_idx)

if len(ligand_top_idx) == 0 or len(receptor_top_idx) == 0:
    raise ValueError(
        "Top MI17 LR genes could not be resolved in adata.var_names. "
        f"Missing ligands: {ligand_top_missing}; missing receptors: {receptor_top_missing}"
    )

ligand_random_idx = rng.choice(
    ligand_pool,
    size=len(ligand_top_idx),
    replace=len(ligand_pool) < len(ligand_top_idx),
) if len(ligand_top_idx) > 0 else np.array([], dtype=int)

receptor_random_idx = rng.choice(
    receptor_pool,
    size=len(receptor_top_idx),
    replace=len(receptor_pool) < len(receptor_top_idx),
) if len(receptor_top_idx) > 0 else np.array([], dtype=int)

mi17_top_lr_summary_df = pd.DataFrame({
    "rank": np.arange(1, len(mi17_top_lr_indices) + 1),
    "LR_index": mi17_top_lr_indices,
    "LR_pair": mi17_top_lr_pair_names,
    "source": mi17_top_lr_source,
})
mi17_top_lr_summary_df.to_csv(mi17_lrko_outdir / f"{MI17_KO_MIOI}_top_LR_pairs_for_KO.csv", index=False)

print("Cell-type column used:", MI17_KO_CELLTYPE_COL)
print("MI17 top LR source:", mi17_top_lr_source)
print("MI17 top LR pairs:")
display(mi17_top_lr_summary_df)
print("Top ligand genes found:", ligand_top_found)
print("Top ligand genes missing:", ligand_top_missing)
print("Top receptor genes found:", receptor_top_found)
print("Top receptor genes missing:", receptor_top_missing)
print("Random ligand indices:", ligand_random_idx.tolist())
print("Random receptor indices:", receptor_random_idx.tolist())

In [ ]:
# ==============================================================
# GO-program gene sets for MI-17 sender and receiver programs
# --------------------------------------------------------------
# The GO terms follow the provided MI-17 sender/receiver panel.
# Genes are loaded from Enrichr GO Biological Process libraries and
# intersected with the PerturbFISH expression matrix.
# ==============================================================

MI17_SENDER_GO_TERMS = [
    "Cellular response to cytokine stimulus",
    "Response to interleukin-1",
    "Cellular response to reactive oxygen species",
    "TGF-beta receptor signaling pathway",
    "Epithelial to mesenchymal transition",
    "Extracellular matrix organization",
]

MI17_RECEIVER_GO_TERMS = [
    "Negative regulation of apoptotic process",
    "Response to endoplasmic reticulum stress",
    "Cellular response to reactive oxygen species",
    "Integrin-mediated signaling pathway",
    "Heterotypic cell-cell adhesion",
    "Regulation of SMAD protein phosphorylation",
]



In [ ]:
GO_LIBRARY_NAME = "GO_Biological_Process_2021"
GO_ORGANISM_CANDIDATES = []
if "SPECIES" in globals():
    if str(SPECIES).lower().startswith("human"):
        GO_ORGANISM_CANDIDATES.extend(["Human", "human", "Mouse", "mouse"])
    else:
        GO_ORGANISM_CANDIDATES.extend(["Mouse", "mouse", "Human", "human"])
GO_ORGANISM_CANDIDATES.extend(["Human", "Mouse", "human", "mouse"])


def _normalize_go_term_name(x):
    x = str(x)
    x = re.sub(r"\s*\(GO:\d+\)\s*$", "", x)
    x = x.replace("–", "-").replace("—", "-")
    x = re.sub(r"\s+", " ", x).strip().lower()
    return x


def _load_go_library():
    last_error = None
    seen = set()
    for organism in GO_ORGANISM_CANDIDATES:
        if organism in seen:
            continue
        seen.add(organism)
        try:
            lib = gp.get_library(name=GO_LIBRARY_NAME, organism=organism)
            print(f"Loaded {GO_LIBRARY_NAME} for organism={organism}; n_terms={len(lib)}")
            return lib, organism
        except Exception as e:
            last_error = e
            print(f"[Warning] Failed to load {GO_LIBRARY_NAME} for organism={organism}: {e}")

    raise RuntimeError(
        "Could not load GO Biological Process gene sets via gseapy. "
        "Please check the internet connection / Enrichr access."
    ) from last_error


def _find_go_library_key(term, go_library):
    target = _normalize_go_term_name(term)

    normalized_to_key = {}
    for key in go_library.keys():
        normalized_to_key.setdefault(_normalize_go_term_name(key), key)

    if target in normalized_to_key:
        return normalized_to_key[target]

    # Common alternative naming in GO libraries.
    aliases = {
        "tgf-beta receptor signaling pathway": [
            "transforming growth factor beta receptor signaling pathway",
            "tgf beta receptor signaling pathway",
        ],
        "integrin-mediated signaling pathway": [
            "integrin mediated signaling pathway",
        ],
        "heterotypic cell-cell adhesion": [
            "heterotypic cell cell adhesion",
            "heterotypic cell-cell adhesion via plasma membrane cell adhesion molecules",
        ],
    }

    for alias in aliases.get(target, []):
        alias_norm = _normalize_go_term_name(alias)
        if alias_norm in normalized_to_key:
            return normalized_to_key[alias_norm]

    # Fuzzy contains matching.
    for key in go_library.keys():
        key_norm = _normalize_go_term_name(key)
        if target in key_norm or key_norm in target:
            return key

    return None


def _resolve_go_genes_to_data(term_list, go_library):
    records = []
    term_gene_dict = {}
    gene_columns = gene_names_for_lrko

    for term in term_list:
        key = _find_go_library_key(term, go_library)
        if key is None:
            print(f"[Warning] Could not find GO term in library: {term}")
            term_gene_dict[term] = []
            records.append({
                "GO_program": term,
                "matched_library_key": None,
                "gene": None,
                "gene_in_data": False,
            })
            continue

        genes_raw = [str(g).strip() for g in go_library[key] if str(g).strip() != ""]
        _, genes_in_data, genes_missing = _resolve_gene_indices(gene_columns, genes_raw)
        genes_in_data = sorted(set(genes_in_data))

        term_gene_dict[term] = genes_in_data
        for gene in genes_raw:
            records.append({
                "GO_program": term,
                "matched_library_key": key,
                "gene": gene,
                "gene_in_data": gene.upper() in {g.upper() for g in genes_in_data},
            })

        print(f"{term}: {len(genes_in_data)} genes in data / {len(genes_raw)} genes in GO library")

    return term_gene_dict, pd.DataFrame(records)


go_library, go_organism_used = _load_go_library()

mi17_sender_go_gene_dict, mi17_sender_go_gene_table = _resolve_go_genes_to_data(
    MI17_SENDER_GO_TERMS,
    go_library,
)
mi17_receiver_go_gene_dict, mi17_receiver_go_gene_table = _resolve_go_genes_to_data(
    MI17_RECEIVER_GO_TERMS,
    go_library,
)

mi17_sender_go_gene_table["GO_role"] = "sender"
mi17_receiver_go_gene_table["GO_role"] = "receiver"
mi17_go_gene_table = pd.concat(
    [mi17_sender_go_gene_table, mi17_receiver_go_gene_table],
    axis=0,
    ignore_index=True,
)

mi17_go_gene_table.to_csv(
    mi17_lrko_outdir / f"{MI17_KO_MIOI}_sender_receiver_GO_program_genes.csv",
    index=False,
)

display(
    mi17_go_gene_table
    .query("gene_in_data == True")
    .groupby(["GO_role", "GO_program"], as_index=False)
    .agg(n_genes=("gene", "nunique"))
)

In [ ]:
# ==============================================================
# Run MI-17 top-LR KO and reconstruct expression before/after KO
# ==============================================================

MI17_KO_PERTURBATIONS = {
    "KO-TopMI17": {
        "ligand_idx": ligand_top_idx,
        "receptor_idx": receptor_top_idx,
        "ligand_scale": 0.0,
        "receptor_scale": 0.0,
        "description": "Top MI17 ligand genes in melanoma senders and receptor genes in T-cell receivers set to zero.",
    },
    "50%-TopMI17": {
        "ligand_idx": ligand_top_idx,
        "receptor_idx": receptor_top_idx,
        "ligand_scale": 0.5,
        "receptor_scale": 0.5,
        "description": "Top MI17 ligand/receptor genes scaled to 50%.",
    },
    "KO-Random": {
        "ligand_idx": ligand_random_idx,
        "receptor_idx": receptor_random_idx,
        "ligand_scale": 0.0,
        "receptor_scale": 0.0,
        "description": "Matched random ligand/receptor genes set to zero.",
    },
}

pd.DataFrame([
    {
        "Perturbation": name,
        "n_ligand_genes": len(cfg["ligand_idx"]),
        "n_receptor_genes": len(cfg["receptor_idx"]),
        "ligand_scale": cfg["ligand_scale"],
        "receptor_scale": cfg["receptor_scale"],
        "description": cfg["description"],
    }
    for name, cfg in MI17_KO_PERTURBATIONS.items()
]).to_csv(mi17_lrko_outdir / f"{MI17_KO_MIOI}_perturbation_modes.csv", index=False)


def _make_cell_key(slice_index, barcode):
    return f"slice{int(slice_index)}::{barcode}"


def _expression_df_from_numpy(exp_array, adata_cur, slice_index, role, selected_idx):
    selected_idx = np.asarray(selected_idx, dtype=int)
    if selected_idx.size == 0:
        return pd.DataFrame(columns=list(adata_cur.var_names))

    barcodes = adata_cur.obs_names[selected_idx].astype(str).tolist()
    df = pd.DataFrame(
        exp_array[selected_idx, :],
        index=[_make_cell_key(slice_index, bc) for bc in barcodes],
        columns=adata_cur.var_names.astype(str),
    )
    df.index.name = "cell_key"
    df.attrs["barcodes"] = barcodes
    df.attrs["slice_index"] = int(slice_index)
    df.attrs["role"] = role
    return df


def _safe_model_reconstruct(graph):
    with torch.no_grad():
        exp_recon, *_ = model(graph)
    return exp_recon.detach().cpu().numpy().astype(np.float32)


sender_original_parts = []
receiver_original_parts = []
sender_perturbed_parts = {name: [] for name in MI17_KO_PERTURBATIONS}
receiver_perturbed_parts = {name: [] for name in MI17_KO_PERTURBATIONS}
selected_cell_metadata_rows = []
selected_edge_rows = []
slice_ko_summary_rows = []

model.eval()

for slice_index, (adata_cur, data_cur, factor_cur) in enumerate(
    zip(adata_list, spidernet_data_list, factor_envir_list_for_lrko)
):
    edge_index_raw_cur = _get_data_edge_index(data_cur)
    factor_cur = np.asarray(factor_cur, dtype=float)

    # Align edge_index to Factor_envir rows. Some processed data store LR-expanded
    # edge_index, while Factor_envir is cell-cell-edge-level.
    edge_index_cur = _align_edge_index_to_factor_rows(
        edge_index_raw_cur,
        n_factor_rows=factor_cur.shape[0],
        slice_index=slice_index,
    )

    if MI17_KO_MI_INDEX >= factor_cur.shape[1]:
        raise IndexError(
            f"{MI17_KO_MIOI} requires column {MI17_KO_MI_INDEX}, "
            f"but factor_envir has {factor_cur.shape[1]} columns."
        )

    celltype_values = adata_cur.obs[MI17_KO_CELLTYPE_COL].astype(str)
    sender_cell_mask = _match_celltype_values(
        celltype_values,
        MI17_KO_SENDER_MATCH_KEYWORDS,
    )
    receiver_cell_mask = _match_celltype_values(
        celltype_values,
        MI17_KO_RECEIVER_MATCH_KEYWORDS,
    )

    sender_cell_mask = np.asarray(sender_cell_mask, dtype=bool)
    receiver_cell_mask = np.asarray(receiver_cell_mask, dtype=bool)

    sender_is_melanoma = sender_cell_mask[edge_index_cur[:, 0]]
    receiver_is_tcell = receiver_cell_mask[edge_index_cur[:, 1]]
    mi17_positive = np.asarray(
        factor_cur[:, MI17_KO_MI_INDEX] > MI17_KO_MI_STRENGTH_THRESHOLD,
        dtype=bool,
    )

    if not (len(sender_is_melanoma) == len(receiver_is_tcell) == len(mi17_positive)):
        raise ValueError(
            f"slice {slice_index}: mask length mismatch after edge alignment: "
            f"sender={len(sender_is_melanoma)}, receiver={len(receiver_is_tcell)}, "
            f"mi17={len(mi17_positive)}"
        )

    edge_mask = sender_is_melanoma & receiver_is_tcell & mi17_positive
    edge_idx_selected = np.where(edge_mask)[0]

    if edge_idx_selected.size == 0:
        slice_ko_summary_rows.append({
            "slice_index": slice_index,
            "n_selected_edges": 0,
            "n_sender_cells": 0,
            "n_receiver_cells": 0,
            "status": "skip_no_MI17_melanoma_to_Tcell_edges",
        })
        continue

    selected_edges = edge_index_cur[edge_idx_selected, :]
    sender_idx = np.unique(selected_edges[:, 0]).astype(int)
    receiver_idx = np.unique(selected_edges[:, 1]).astype(int)

    for local_edge_idx, sender_i, receiver_i in zip(edge_idx_selected, selected_edges[:, 0], selected_edges[:, 1]):
        selected_edge_rows.append({
            "slice_index": int(slice_index),
            "edge_index_in_slice": int(local_edge_idx),
            "sender_index": int(sender_i),
            "receiver_index": int(receiver_i),
            "sender_barcode": str(adata_cur.obs_names[int(sender_i)]),
            "receiver_barcode": str(adata_cur.obs_names[int(receiver_i)]),
            "sender_celltype": str(celltype_values.iloc[int(sender_i)]),
            "receiver_celltype": str(celltype_values.iloc[int(receiver_i)]),
            "MI17_strength": float(factor_cur[int(local_edge_idx), MI17_KO_MI_INDEX]),
        })

    for role, idx_arr in [("sender_melanoma", sender_idx), ("receiver_Tcell", receiver_idx)]:
        for idx in idx_arr:
            selected_cell_metadata_rows.append({
                "slice_index": int(slice_index),
                "cell_key": _make_cell_key(slice_index, str(adata_cur.obs_names[int(idx)])),
                "barcode": str(adata_cur.obs_names[int(idx)]),
                "cell_index": int(idx),
                "cell_role": role,
                "celltype": str(celltype_values.iloc[int(idx)]),
            })

    graph_device = _to_device_graph(data_cur, device)

    # Original reconstruction.
    exp_original = _safe_model_reconstruct(graph_device)
    sender_original_parts.append(
        _expression_df_from_numpy(exp_original, adata_cur, slice_index, "sender_melanoma", sender_idx)
    )
    receiver_original_parts.append(
        _expression_df_from_numpy(exp_original, adata_cur, slice_index, "receiver_Tcell", receiver_idx)
    )

    # Perturbed reconstructions.
    for perturb_name, perturb_cfg in MI17_KO_PERTURBATIONS.items():
        exp_perturb = run_model_on_lrko_graph(
            graph_device,
            sender_idx=sender_idx,
            receiver_idx=receiver_idx,
            ligand_idx=perturb_cfg["ligand_idx"],
            receptor_idx=perturb_cfg["receptor_idx"],
            ligand_scale=perturb_cfg["ligand_scale"],
            receptor_scale=perturb_cfg["receptor_scale"],
        )
        sender_perturbed_parts[perturb_name].append(
            _expression_df_from_numpy(exp_perturb, adata_cur, slice_index, "sender_melanoma", sender_idx)
        )
        receiver_perturbed_parts[perturb_name].append(
            _expression_df_from_numpy(exp_perturb, adata_cur, slice_index, "receiver_Tcell", receiver_idx)
        )

    slice_ko_summary_rows.append({
        "slice_index": slice_index,
        "n_selected_edges": int(edge_idx_selected.size),
        "n_sender_cells": int(sender_idx.size),
        "n_receiver_cells": int(receiver_idx.size),
        "mean_MI17_strength_selected_edges": float(np.nanmean(factor_cur[edge_idx_selected, MI17_KO_MI_INDEX])),
        "status": "used",
    })


def _concat_expression_parts(parts):
    parts = [df for df in parts if isinstance(df, pd.DataFrame) and df.shape[0] > 0]
    if len(parts) == 0:
        return pd.DataFrame(columns=gene_names_for_lrko)
    return pd.concat(parts, axis=0)


mi17_lrko_sender_original_expr = _concat_expression_parts(sender_original_parts)
mi17_lrko_receiver_original_expr = _concat_expression_parts(receiver_original_parts)
mi17_lrko_sender_perturbed_expr = {
    name: _concat_expression_parts(parts)
    for name, parts in sender_perturbed_parts.items()
}
mi17_lrko_receiver_perturbed_expr = {
    name: _concat_expression_parts(parts)
    for name, parts in receiver_perturbed_parts.items()
}

mi17_lrko_selected_cells_df = pd.DataFrame(selected_cell_metadata_rows).drop_duplicates()
mi17_lrko_selected_edges_df = pd.DataFrame(selected_edge_rows)
mi17_lrko_slice_summary_df = pd.DataFrame(slice_ko_summary_rows)

mi17_lrko_selected_cells_df.to_csv(mi17_lrko_outdir / f"{MI17_KO_MIOI}_selected_sender_receiver_cells.csv", index=False)
mi17_lrko_selected_edges_df.to_csv(mi17_lrko_outdir / f"{MI17_KO_MIOI}_selected_melanoma_to_Tcell_edges.csv", index=False)
mi17_lrko_slice_summary_df.to_csv(mi17_lrko_outdir / f"{MI17_KO_MIOI}_LRKO_slice_summary.csv", index=False)

print("MI17 LR-KO selected-edge summary:")
display(mi17_lrko_slice_summary_df)
print("Selected cells:", mi17_lrko_selected_cells_df.shape)
print("Selected edges:", mi17_lrko_selected_edges_df.shape)

In [ ]:
# ==============================================================
# Compute sender/receiver GO-program module-score changes
# --------------------------------------------------------------
# Module score:
#   Mean expression after per-gene standardization against the original
#   selected-cell reconstruction.
#
# Change score:
#   perturbed module score - original module score
#
# Also report log2FC of mean reconstructed expression per GO program.
# ==============================================================

def _valid_gene_indices_for_expr(expr_df, genes):
    gene_idx, genes_found, genes_missing = _resolve_gene_indices(expr_df.columns.astype(str), genes)
    return gene_idx, genes_found, genes_missing


def _compute_module_score_from_reference(expr_df, genes_idx, ref_mean=None, ref_std=None):
    genes_idx = np.asarray(genes_idx, dtype=int)
    genes_idx = genes_idx[(genes_idx >= 0) & (genes_idx < expr_df.shape[1])]

    if expr_df.shape[0] == 0 or genes_idx.size == 0:
        return pd.Series(np.nan, index=expr_df.index)

    X = expr_df.iloc[:, genes_idx].to_numpy(dtype=float)

    if ref_mean is None:
        ref_mean = np.nanmean(X, axis=0)
    if ref_std is None:
        ref_std = np.nanstd(X, axis=0, ddof=1)

    ref_std = np.asarray(ref_std, dtype=float)
    ref_std[(~np.isfinite(ref_std)) | (ref_std == 0)] = np.nan

    Z = (X - ref_mean) / ref_std
    Z = np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0)
    return pd.Series(np.nanmean(Z, axis=1), index=expr_df.index)


def _compute_mean_expr_log2fc(perturb_expr_df, original_expr_df, genes_idx, eps=1e-8):
    genes_idx = np.asarray(genes_idx, dtype=int)
    genes_idx = genes_idx[(genes_idx >= 0) & (genes_idx < original_expr_df.shape[1])]

    if perturb_expr_df.shape[0] == 0 or genes_idx.size == 0:
        return pd.Series(np.nan, index=perturb_expr_df.index)

    original_aligned = original_expr_df.reindex(perturb_expr_df.index)
    X0 = original_aligned.iloc[:, genes_idx].to_numpy(dtype=float)
    X1 = perturb_expr_df.iloc[:, genes_idx].to_numpy(dtype=float)

    mean0 = np.nanmean(X0, axis=1)
    mean1 = np.nanmean(X1, axis=1)

    return pd.Series(np.log2((mean1 + eps) / (mean0 + eps)), index=perturb_expr_df.index)


def _build_go_change_table_for_role(original_expr_df, perturbed_expr_dict, go_gene_dict, role_label):
    records = []

    if original_expr_df.shape[0] == 0:
        print(f"[Warning] No original expression rows for role={role_label}")
        return pd.DataFrame()

    for go_term, genes in go_gene_dict.items():
        gene_idx, genes_found, genes_missing = _valid_gene_indices_for_expr(original_expr_df, genes)

        if len(gene_idx) == 0:
            print(f"[Warning] {role_label} / {go_term}: no genes found in reconstructed expression matrix.")
            continue

        X_ref = original_expr_df.iloc[:, gene_idx].to_numpy(dtype=float)
        ref_mean = np.nanmean(X_ref, axis=0)
        ref_std = np.nanstd(X_ref, axis=0, ddof=1)

        original_score = _compute_module_score_from_reference(
            original_expr_df,
            gene_idx,
            ref_mean=ref_mean,
            ref_std=ref_std,
        )

        for perturb_name, perturb_expr_df in perturbed_expr_dict.items():
            if perturb_expr_df.shape[0] == 0:
                continue

            common_idx = perturb_expr_df.index.intersection(original_expr_df.index)
            if len(common_idx) == 0:
                continue

            perturb_use = perturb_expr_df.loc[common_idx]
            original_use = original_expr_df.loc[common_idx]
            original_score_use = original_score.loc[common_idx]

            perturb_score = _compute_module_score_from_reference(
                perturb_use,
                gene_idx,
                ref_mean=ref_mean,
                ref_std=ref_std,
            )
            log2fc_mean_expr = _compute_mean_expr_log2fc(
                perturb_use,
                original_use,
                gene_idx,
            )

            for cell_key in common_idx:
                records.append({
                    "cell_key": cell_key,
                    "cell_role": role_label,
                    "GO_program": go_term,
                    "Perturbation": perturb_name,
                    "n_GO_genes_in_data": int(len(gene_idx)),
                    "original_module_score": float(original_score_use.loc[cell_key]),
                    "perturbed_module_score": float(perturb_score.loc[cell_key]),
                    "delta_module_score": float(perturb_score.loc[cell_key] - original_score_use.loc[cell_key]),
                    "log2FC_mean_expression": float(log2fc_mean_expr.loc[cell_key]),
                })

    out = pd.DataFrame(records)
    if out.shape[0] > 0:
        out = out.merge(
            mi17_lrko_selected_cells_df[
                ["cell_key", "slice_index", "barcode", "cell_index", "celltype"]
            ],
            on="cell_key",
            how="left",
        )
    return out


sender_change_df = _build_go_change_table_for_role(
    original_expr_df=mi17_lrko_sender_original_expr,
    perturbed_expr_dict=mi17_lrko_sender_perturbed_expr,
    go_gene_dict=mi17_sender_go_gene_dict,
    role_label="sender_melanoma",
)

receiver_change_df = _build_go_change_table_for_role(
    original_expr_df=mi17_lrko_receiver_original_expr,
    perturbed_expr_dict=mi17_lrko_receiver_perturbed_expr,
    go_gene_dict=mi17_receiver_go_gene_dict,
    role_label="receiver_Tcell",
)

mi17_lrko_go_change_long = pd.concat(
    [sender_change_df, receiver_change_df],
    axis=0,
    ignore_index=True,
)

if mi17_lrko_go_change_long.shape[0] == 0:
    raise ValueError("No GO-program module-score changes were computed. Check selected cells and GO gene overlaps.")

change_long_path = mi17_lrko_outdir / f"{MI17_KO_MIOI}_LRKO_sender_receiver_GO_module_score_change_long.csv"
mi17_lrko_go_change_long.to_csv(change_long_path, index=False)

# -----------------------------
# Summary statistics
# -----------------------------
summary_rows = []

for (role, term, perturb), df_cur in mi17_lrko_go_change_long.groupby(
    ["cell_role", "GO_program", "Perturbation"],
    sort=False,
):
    delta = pd.to_numeric(df_cur["delta_module_score"], errors="coerce").dropna().to_numpy(dtype=float)
    lfc = pd.to_numeric(df_cur["log2FC_mean_expression"], errors="coerce").dropna().to_numpy(dtype=float)

    if len(delta) >= 2 and np.nanstd(delta) > 0:
        try:
            p_delta_vs_zero = wilcoxon(delta, alternative="two-sided").pvalue
        except Exception:
            p_delta_vs_zero = np.nan
    else:
        p_delta_vs_zero = np.nan

    summary_rows.append({
        "cell_role": role,
        "GO_program": term,
        "Perturbation": perturb,
        "n_cells": int(len(delta)),
        "mean_delta_module_score": float(np.nanmean(delta)) if len(delta) > 0 else np.nan,
        "median_delta_module_score": float(np.nanmedian(delta)) if len(delta) > 0 else np.nan,
        "mean_log2FC_mean_expression": float(np.nanmean(lfc)) if len(lfc) > 0 else np.nan,
        "median_log2FC_mean_expression": float(np.nanmedian(lfc)) if len(lfc) > 0 else np.nan,
        "wilcoxon_p_delta_vs_zero": p_delta_vs_zero,
    })

mi17_lrko_go_change_summary = pd.DataFrame(summary_rows)

# KO-TopMI17 vs KO-Random baseline comparison for each GO program
baseline_compare_rows = []
for (role, term), df_cur in mi17_lrko_go_change_long.groupby(["cell_role", "GO_program"], sort=False):
    top_vals = pd.to_numeric(
        df_cur.loc[df_cur["Perturbation"] == "KO-TopMI17", "delta_module_score"],
        errors="coerce",
    ).dropna()
    rand_vals = pd.to_numeric(
        df_cur.loc[df_cur["Perturbation"] == "KO-Random", "delta_module_score"],
        errors="coerce",
    ).dropna()

    if len(top_vals) > 0 and len(rand_vals) > 0:
        p_top_vs_random = mannwhitneyu(top_vals, rand_vals, alternative="two-sided").pvalue
    else:
        p_top_vs_random = np.nan

    baseline_compare_rows.append({
        "cell_role": role,
        "GO_program": term,
        "n_top": int(len(top_vals)),
        "n_random": int(len(rand_vals)),
        "mean_delta_KO_TopMI17": float(np.nanmean(top_vals)) if len(top_vals) > 0 else np.nan,
        "mean_delta_KO_Random": float(np.nanmean(rand_vals)) if len(rand_vals) > 0 else np.nan,
        "mean_delta_Top_minus_Random": (
            float(np.nanmean(top_vals) - np.nanmean(rand_vals))
            if len(top_vals) > 0 and len(rand_vals) > 0 else np.nan
        ),
        "mannwhitney_p_KO_TopMI17_vs_KO_Random": p_top_vs_random,
    })

mi17_lrko_top_vs_random_summary = pd.DataFrame(baseline_compare_rows)

summary_path = mi17_lrko_outdir / f"{MI17_KO_MIOI}_LRKO_GO_module_score_change_summary.csv"
top_vs_random_path = mi17_lrko_outdir / f"{MI17_KO_MIOI}_LRKO_GO_module_score_KOTopMI17_vs_KORandom_summary.csv"
mi17_lrko_go_change_summary.to_csv(summary_path, index=False)
mi17_lrko_top_vs_random_summary.to_csv(top_vs_random_path, index=False)

print(f"Saved long GO-program change table to: {change_long_path}")
print(f"Saved GO-program change summary to: {summary_path}")
print(f"Saved KO-TopMI17 vs KO-Random summary to: {top_vs_random_path}")

display(mi17_lrko_go_change_summary)
display(mi17_lrko_top_vs_random_summary)

In [ ]:
# ==============================================================
# Visualize MI-17 LR-KO effects on GO-program module scores
# ==============================================================

plt.close("all")
plt.style.use("default")
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})
sns.set_theme(style="white")

MI17_LRKO_PLOT_PERTURB_ORDER = [
    "KO-TopMI17",
    "50%-TopMI17",
    "KO-Random",
]

MI17_LRKO_PLOT_PALETTE = {
    "KO-TopMI17": "#8ca9ff",
    "50%-TopMI17": "#8dd4ff",
    "KO-Random": "#cccccc",
}


def _save_fig_both(fig, basepath_no_ext):
    basepath_no_ext = str(basepath_no_ext)
    fig.savefig(basepath_no_ext + ".pdf", bbox_inches="tight", facecolor="white")
    fig.savefig(basepath_no_ext + ".png", bbox_inches="tight", facecolor="white", dpi=300)




def _plot_lrko_go_delta_violin_boxplot(df, role_label, term_order, title, out_prefix):
    plot_df = df[
        (df["cell_role"] == role_label)
        & df["GO_program"].isin(term_order)
        & df["Perturbation"].isin(MI17_LRKO_PLOT_PERTURB_ORDER)
    ].copy()

    if plot_df.shape[0] == 0:
        print(f"[Warning] No data to plot for {role_label}")
        return

    plot_df["GO_program"] = pd.Categorical(
        plot_df["GO_program"],
        categories=term_order,
        ordered=True,
    )
    plot_df["Perturbation"] = pd.Categorical(
        plot_df["Perturbation"],
        categories=MI17_LRKO_PLOT_PERTURB_ORDER,
        ordered=True,
    )

    n_terms = len(term_order)

    # Height is 1.5x of the original 3.4, with enough room for all GO terms.
    fig_width = 8.5
    fig_height = max(5.1, 1.05 * n_terms)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height), facecolor="white")
    ax.set_facecolor("white")

    # ----------------------------------------------------------
    # Manual positions:
    #   y = GO program
    #   x = delta module score
    #
    # This avoids seaborn violinplot/boxplot using different
    # dodge offsets.
    # ----------------------------------------------------------
    n_groups = len(MI17_LRKO_PLOT_PERTURB_ORDER)
    group_width = 0.72
    offsets = np.linspace(
        -group_width / 2 + group_width / (2 * n_groups),
        group_width / 2 - group_width / (2 * n_groups),
        n_groups,
    )

    violin_width = 0.21
    box_width = 0.075

    for term_i, go_term in enumerate(term_order):
        base_y = term_i

        for perturb_i, perturb_name in enumerate(MI17_LRKO_PLOT_PERTURB_ORDER):
            y_pos = base_y + offsets[perturb_i]

            vals = pd.to_numeric(
                plot_df.loc[
                    (plot_df["GO_program"] == go_term)
                    & (plot_df["Perturbation"] == perturb_name),
                    "delta_module_score",
                ],
                errors="coerce",
            ).dropna().to_numpy(dtype=float)

            if vals.size == 0:
                continue

            # Violin plot
            violin_parts = ax.violinplot(
                vals,
                positions=[y_pos],
                vert=False,
                widths=violin_width,
                showmeans=False,
                showmedians=False,
                showextrema=False,
            )

            for body in violin_parts["bodies"]:
                body.set_facecolor(MI17_LRKO_PLOT_PALETTE[perturb_name])
                body.set_edgecolor("#939393")
                body.set_linewidth(0.8)
                body.set_alpha(1.0)

            # Boxplot centered at the same y_pos as the violin.
            # showcaps=False removes the two vertical cap lines at both whisker ends.
            bp = ax.boxplot(
                vals,
                positions=[y_pos],
                vert=False,
                widths=box_width,
                patch_artist=True,
                showfliers=False,
                showcaps=False,
                boxprops={
                    "facecolor": "black",
                    "edgecolor": "black",
                    "linewidth": 0.8,
                },
                whiskerprops={
                    "color": "black",
                    "linewidth": 0.8,
                },
                medianprops={
                    "color": "white",
                    "linewidth": 1.0,
                },
            )

            # Keep boxplot on top of violin.
            for key in ["boxes", "whiskers", "medians"]:
                for artist in bp[key]:
                    artist.set_zorder(5)

    ax.axvline(0, color="#4D4D4D", linewidth=0.8, linestyle="--")

    ax.set_yticks(np.arange(n_terms))
    ax.set_yticklabels(term_order)
    ax.invert_yaxis()

    ax.set_title(title)
    ax.set_xlabel("Δ GO module score\n(perturbed - original)")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=0)
    ax.tick_params(axis="y", rotation=0)

    sns.despine(ax=ax, top=True, right=True)

    legend_handles = [
        mpl.patches.Patch(
            facecolor=MI17_LRKO_PLOT_PALETTE[p],
            edgecolor="#939393",
            linewidth=0.8,
            label=p,
        )
        for p in MI17_LRKO_PLOT_PERTURB_ORDER
    ]

    ax.legend(
        handles=legend_handles,
        title="Perturbation",
        frameon=False,
        bbox_to_anchor=(1.01, 1.0),
        loc="upper left",
    )

    plt.tight_layout()

    _save_fig_both(fig, mi17_lrko_outdir / out_prefix)
    plt.show()
    plt.close(fig)

_plot_lrko_go_delta_violin_boxplot(
    df=mi17_lrko_go_change_long,
    role_label="sender_melanoma",
    term_order=MI17_SENDER_GO_TERMS,
    title=f"{MI17_KO_MIOI} top LR-pair KO effect on melanoma sender GO programs",
    out_prefix=f"{MI17_KO_MIOI}_LRKO_sender_melanoma_GO_delta_module_score_violin_boxplot",
)

_plot_lrko_go_delta_violin_boxplot(
    df=mi17_lrko_go_change_long,
    role_label="receiver_Tcell",
    term_order=MI17_RECEIVER_GO_TERMS,
    title=f"{MI17_KO_MIOI} top LR-pair KO effect on T-cell receiver GO programs",
    out_prefix=f"{MI17_KO_MIOI}_LRKO_receiver_Tcell_GO_delta_module_score_violin_boxplot",
)

print("Saved MI17 LR-KO GO-program violin+boxplots to:", mi17_lrko_outdir)


## 16. Cross-method CCC baseline benchmark: melanoma → T-cell MI-17 GO-program SMD

This notebook benchmarks communication representations from SpiderNet and three cell–cell communication (CCC) baselines on the Perturb-FISH melanoma dataset. It asks whether cells with high versus low melanoma → T-cell communication strength differ in sender- and receiver-side Gene Ontology (GO) programs associated with MI-17.

### Workflow

1. Load processed Perturb-FISH data and trained SpiderNet outputs.
2. Define the MI-17 sender and receiver GO programs and calculate global z-score module scores.
3. Obtain edge-level features from SpiderNet, NMF-LR, COMMOT, and ScCChain.
4. Restrict the evaluation to melanoma → T-cell edges from the 11 target perturbations, aggregate edge scores to cells, and form positive-score median-split groups.
5. Select one axis per method using perturbation-versus-control edge-score changes, then calculate pooled-standard-deviation SMDs and two-sided Mann–Whitney U-test P values for the GO programs.

### Inputs

- An installed SpiderNet package with the tutorial and benchmark dependencies.
- The processed Perturb-FISH bundle in `Results/PerturbFISH/ProcessedData`.
- Trained 23-dimensional SpiderNet outputs and `Factor_envir_list.pkl`.
- GO Biological Process 2021 gene sets obtained through GSEApy/Enrichr.
- Optional COMMOT results and ScCChain edge-program CSV files; the latter are generated outside Python by the paired Julia workflow.

### Outputs and paper mapping

The notebook writes cell/edge metadata, method-specific edge scores, GO module scores, axis-matching diagnostics, per-feature SMD tables, selected-axis summaries, and sender/receiver SMD barplots beneath the configured Perturb-FISH results directory. The manuscript discusses MI-17-associated programs and original-slice validation in Fig. PerturbFISH d–e, but the cross-method GO-program SMD benchmark produced here is an additional analysis and is not explicitly reported in the current manuscript.


### 16.0 Run order and external ScCChain dependency

1. Section 16.1 reuses the `WORKSPACE_ROOT` configured earlier in this notebook. Alternatively, set `SPIDERNET_ANALYSIS_ROOT` to override it; when neither is available, the section searches upward from the Jupyter working directory. Run Sections 16.1–16.8b to compute or load SpiderNet, NMF-LR, and COMMOT scores and export `ScCChain_input_manifest.csv`, `PerturbFISH_lr_db_for_scCChain.csv`, and the ScCChain h5ad inputs.
2. In a terminal, run the external ScCChain Julia workflow. `ScCChain_PerturbFISH_runner.jl` is not included in this tutorial folder, so replace the runner and output placeholders below with the paths for your installation and configured results directory:

```powershell
julia path/to/ScCChain_PerturbFISH_runner.jl `
  --manifest "<benchmark-output>/ScCChain_input_manifest.csv" `
  --lr-db-csv "<benchmark-output>/PerturbFISH_lr_db_for_scCChain.csv" `
  --result-root "<ScCChain-output>" `
  --n-programs 23
```

The exact command for the configured export directory is printed by Section 16.8b. Before regenerating ScCChain scores, verify `SCCCHAIN_OUTDIR` in Section 16.9: the current analysis preserves the separate legacy results layout used by the archived ScCChain output.

3. Return to the notebook and run Sections 16.9–16.11. These sections align ScCChain outputs, calculate the SMD tables, and save the sender/receiver barplots.

If you do not want to include ScCChain, you can skip the Julia step and continue; the notebook will print a warning and produce the comparison with available methods.


Section 16.10 restricts communication-strength aggregation and High/Low grouping to melanoma→T-cell edges whose melanoma sender is annotated with one of the 11 target perturbation genes (`CHUK`, `IRAK1`, `TRAM1`, `LBP`, `IRAK4`, `PELI1`, `TAB2`, `MAP2K2`, `MAP2K6`, `IRF7`, `MYD88`).


In [ ]:
# ==============================================================
# 16.1 Imports and paths
# ============================================================== 

import os
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from sklearn.decomposition import NMF

try:
    import gseapy as gp
except Exception as e:
    gp = None
    print("[Warning] gseapy is not available:", repr(e))

from SpiderNet.config import PathConfig, TrainingConfig
from SpiderNet.io import load_processed_data
from SpiderNet.analysis import sanity_check_processed, load_part0_outputs

# ------------------------------
# Analysis paths
# ------------------------------
from run_benchmarks import WORKSPACE_ROOT, DATA_ROOT, UPSTREAM_ROOT, PROCESSED_ROOT, RESULTS_DIR, OUTPUT_DIR
ANALYSIS_ROOT = WORKSPACE_ROOT
OUTPUT_ROOT = UPSTREAM_ROOT
PROCESSED_DATA_DIR = PROCESSED_ROOT

SPECIES = "human"
VERSION = "V1"
DIM_ENVIR = 23
MAX_EPOCH = 20000

paths = PathConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    version=VERSION,
    species=SPECIES,
)

train_cfg = TrainingConfig(
    version=VERSION,
    dim_envir=DIM_ENVIR,
    max_epoch=MAX_EPOCH,
    n_jobs=10,
)

base_run_dir = OUTPUT_DIR
base_model_dir = RESULTS_DIR / "Model"
base_run_dirs = {"run_dir": base_run_dir, "model_dir": base_model_dir}

BENCHMARK_OUTDIR = base_run_dir / "Baseline_CCC_MI17_GO_SMD_melanoma_to_Tcell"
BENCHMARK_OUTDIR.mkdir(parents=True, exist_ok=True)

# Method output roots
NMF_LR_OUTDIR = BENCHMARK_OUTDIR / "NMF_LR"
COMMOT_OUTDIR = BENCHMARK_OUTDIR / "COMMOT"
SCCCHAIN_INPUT_DIR = BENCHMARK_OUTDIR / "ScCChain_input_h5ad"
SCCCHAIN_OUTDIR = BENCHMARK_OUTDIR / "ScCChain"
for p in [NMF_LR_OUTDIR, COMMOT_OUTDIR, SCCCHAIN_INPUT_DIR, SCCCHAIN_OUTDIR]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------
# Analysis parameters
# ------------------------------
MIOI = "MI17"
MIOI_INDEX = int(MIOI.replace("MI", "").replace("-", "")) - 1

SENDER_CELLTYPE_KEYWORDS = ["melanoma", "cancer", "tumor", "malignant"]
RECEIVER_CELLTYPE_KEYWORDS = ["t cell", "t_cell", "t-cell", "tcell", " t "]

# Only use melanoma→T-cell edges whose melanoma sender cells are annotated
# within these perturbation genes. Downstream sender-side and receiver-side
# communication-strength aggregation and High/Low grouping both use this
# restricted edge set.
PERTURB_GENES_OF_INTEREST = [
    "CHUK", "IRAK1", "TRAM1", "LBP", "IRAK4", "PELI1",
    "TAB2", "MAP2K2", "MAP2K6", "IRF7", "MYD88",
]
# PERTURB_GENES_OF_INTEREST = [
#     "IRAK1", "TRAM1", "LBP", "PELI1",
#     "IRF7",
# ]
PERTURB_GENE_OBS_COL = None  # set manually if auto-detection picks the wrong adata.obs column

# Keywords used to identify unperturbed/control melanoma cells in the same perturbation
# annotation column. If no explicit control edges are detected, the axis-matching
# step can optionally fall back to melanoma→T-cell edges from non-target-perturbation
# melanoma cells. Set the fallback to False if you want a hard error instead.
CONTROL_PERTURB_KEYWORDS = [
    "control", "ctrl", "vehicle", "mock", "empty", "negative",
    "non-target", "non_target", "nontarget", "nontargeting",
    "non-targeting", "ntc", "scramble", "scrambled", "unperturbed",
]
AXIS_MATCH_CONTROL_FALLBACK_TO_NON_TARGET_MELANOMA = True

MELANOMA_PERTURB_EDGE_FILTER_COL = "is_melanoma_to_Tcell_targetPerturbSender"
CONTROL_MELANOMA_TO_TCELL_EDGE_COL = "is_control_melanoma_to_Tcell"

# Aggregation from melanoma→T-cell edges to cells for grouping.
# 'max' matches the idea of selecting cells with strong communication exposure.
# EDGE_TO_CELL_AGG = "max"
EDGE_TO_CELL_AGG = "sum"
HIGH_LOW_SPLIT = "median"  # median split among cells with finite and positive communication strength

# Baseline dimensions.
NMF_LR_N_COMPONENTS = DIM_ENVIR
SCCCHAIN_N_PROGRAMS = DIM_ENVIR
RANDOM_SEED = 0

print("Processed data:", PROCESSED_DATA_DIR)
print("SpiderNet run dir:", base_run_dir)
print("Benchmark output:", BENCHMARK_OUTDIR)


In [ ]:
# ==============================================================
# 16.2 Load processed Perturb-FISH data and SpiderNet outputs
# ============================================================== 

required_processed_files = [
    "adata_list.pkl",
    "LR_list.pkl",
    "genenames_train.pkl",
]
missing_processed_files = [
    file_name for file_name in required_processed_files
    if not (PROCESSED_DATA_DIR / file_name).is_file()
]
has_spidernet_data = any(
    (PROCESSED_DATA_DIR / file_name).is_file()
    for file_name in ("SpiderNet_data_pyg_list.pkl", "SpiderNet_data_pyg_list.pt")
)
if missing_processed_files or not has_spidernet_data:
    missing_description = missing_processed_files.copy()
    if not has_spidernet_data:
        missing_description.append("SpiderNet_data_pyg_list.pkl or SpiderNet_data_pyg_list.pt")
    raise FileNotFoundError(
        f"Incomplete processed-data directory: {PROCESSED_DATA_DIR}. "
        f"Missing: {missing_description}"
    )

processed = load_processed_data(PROCESSED_DATA_DIR)
if processed.adata_list is None or processed.spidernet_data is None:
    raise RuntimeError(
        "load_processed_data returned an incomplete bundle despite the input-file checks. "
        "Verify that the installed SpiderNet version can read this processed-data format."
    )
check_df, mismatch = sanity_check_processed(processed)
if mismatch.any():
    print("[Warning] Potential processed-data mismatch detected:")
    display(check_df.loc[mismatch])
else:
    print("Sanity check passed.")
    display(check_df)

part0_outputs = load_part0_outputs(RESULTS_DIR)
loading_receiver_use_df = part0_outputs["loading_receiver_use_df"]
loading_sender_use_df = part0_outputs["loading_sender_use_df"]
loading_LR_use = part0_outputs["loading_LR_use"]
Factor_envir_use = part0_outputs["Factor_envir_use"]

factor_envir_list_path = RESULTS_DIR / "Factor_envir_list.pkl"
if not factor_envir_list_path.exists():
    raise FileNotFoundError(f"Missing SpiderNet factor list: {factor_envir_list_path}")
with open(factor_envir_list_path, "rb") as f:
    spidernet_factor_list = pickle.load(f)

print("n batches:", len(processed.adata_list))
print("n LR pairs:", len(processed.lr_list))
print("n genes:", len(processed.genenames_train))
print("Factor_envir_use:", Factor_envir_use.shape)
print("SpiderNet factor list shapes:", [np.asarray(x).shape for x in spidernet_factor_list[:3]], "...")


In [ ]:
# ==============================================================
# 16.3 Helper functions
# ============================================================== 

def _as_numpy(x):
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _edge_index_to_2col(edge_index):
    edge_index = _as_numpy(edge_index)
    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got {edge_index.shape}")
    if edge_index.shape[0] == 2 and edge_index.shape[1] != 2:
        edge_index = edge_index.T
    if edge_index.shape[1] != 2:
        raise ValueError(f"edge_index must have two columns, got {edge_index.shape}")
    return edge_index.astype(int, copy=False)


def _get_data_field(data_cur, key):
    if isinstance(data_cur, dict):
        return data_cur[key]
    return getattr(data_cur, key)


def _get_edge_index(data_cur):
    return _edge_index_to_2col(_get_data_field(data_cur, "edge_index"))


def _align_edge_index_to_factor_rows(edge_index_cur, n_factor_rows, slice_index=None):
    """Collapse LR-expanded edge_index back to cell-cell edge level when needed."""
    edge_index_cur = _edge_index_to_2col(edge_index_cur)
    n_edge_rows = edge_index_cur.shape[0]
    n_factor_rows = int(n_factor_rows)
    prefix = f"slice {slice_index}: " if slice_index is not None else ""

    if n_edge_rows == n_factor_rows:
        return edge_index_cur

    if n_edge_rows % n_factor_rows != 0:
        # Fallback by ordered unique pairs.
        dedup = pd.DataFrame(edge_index_cur, columns=["sender", "receiver"]).drop_duplicates(keep="first").to_numpy(dtype=int)
        if dedup.shape[0] == n_factor_rows:
            print(f"{prefix}edge_index rows={n_edge_rows}; using ordered unique pairs to match factor rows={n_factor_rows}.")
            return dedup
        raise ValueError(f"{prefix}Cannot align edge_index rows={n_edge_rows} to factor rows={n_factor_rows}.")

    repeat_factor = n_edge_rows // n_factor_rows

    # Block/tile layout: [all edges for LR1, all edges for LR2, ...]
    first_block = edge_index_cur[:n_factor_rows, :]
    block_layout_ok = True
    for k in range(1, min(repeat_factor, 5)):
        if not np.array_equal(edge_index_cur[k*n_factor_rows:(k+1)*n_factor_rows, :], first_block):
            block_layout_ok = False
            break
    if block_layout_ok:
        print(f"{prefix}detected block LR-expanded edge_index with repeat_factor={repeat_factor}; using first block.")
        return first_block

    # Consecutive repeat layout: [edge1 repeated K times, edge2 repeated K times, ...]
    try:
        grouped = edge_index_cur.reshape(n_factor_rows, repeat_factor, 2)
        if np.all(grouped == grouped[:, :1, :]):
            print(f"{prefix}detected consecutive-repeat LR-expanded edge_index with repeat_factor={repeat_factor}; using first repeat.")
            return grouped[:, 0, :]
    except Exception:
        pass

    # Ordered unique fallback.
    dedup = pd.DataFrame(edge_index_cur, columns=["sender", "receiver"]).drop_duplicates(keep="first").to_numpy(dtype=int)
    if dedup.shape[0] == n_factor_rows:
        print(f"{prefix}using ordered unique pairs to match factor rows={n_factor_rows}.")
        return dedup

    raise ValueError(
        f"{prefix}Failed to align edge_index rows={n_edge_rows} to factor rows={n_factor_rows}; "
        f"repeat_factor={repeat_factor}, unique={dedup.shape[0]}."
    )


def _infer_celltype_column(adata_list):
    candidates = ["celltype2", "celltype", "cell_type", "cell_class", "CellType", "annotation"]
    for col in candidates:
        if all(col in adata.obs.columns for adata in adata_list):
            return col
    raise KeyError("Could not infer cell type column. Please set CELLTYPE_COL manually.")


CELLTYPE_COL = _infer_celltype_column(processed.adata_list)
print("Using cell type column:", CELLTYPE_COL)


def _match_keywords(values, keywords):
    s = pd.Series(values).astype(str).str.lower()
    out = np.zeros(len(s), dtype=bool)
    for kw in keywords:
        out |= s.str.contains(re.escape(kw.lower()), regex=True, na=False).to_numpy()
    return out


def _safe_colmax_normalize(X):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        m = np.nanmax(X) if X.size else np.nan
        if not np.isfinite(m) or m <= 0:
            return np.zeros_like(X, dtype=np.float32)
        return np.nan_to_num(X / m, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    if X.shape[0] == 0:
        return X.astype(np.float32)
    colmax = np.nanmax(X, axis=0)
    colmax[(~np.isfinite(colmax)) | (colmax <= 0)] = 1.0
    out = X / colmax[None, :]
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def _get_dense_X(adata):
    X = adata.X
    if sp.issparse(X):
        X = X.toarray()
    return np.asarray(X, dtype=np.float32)


def _pooled_smd(high_vals, low_vals):
    high_vals = pd.to_numeric(pd.Series(high_vals), errors="coerce").dropna().to_numpy(dtype=float)
    low_vals = pd.to_numeric(pd.Series(low_vals), errors="coerce").dropna().to_numpy(dtype=float)
    if len(high_vals) == 0 or len(low_vals) == 0:
        return np.nan
    if len(high_vals) > 1 and len(low_vals) > 1:
        pooled_var = ((len(high_vals)-1)*np.var(high_vals, ddof=1) + (len(low_vals)-1)*np.var(low_vals, ddof=1)) / (len(high_vals)+len(low_vals)-2)
        pooled_sd = np.sqrt(pooled_var)
    else:
        pooled_sd = np.nan
    if not np.isfinite(pooled_sd) or pooled_sd == 0:
        return np.nan
    return float((np.mean(high_vals) - np.mean(low_vals)) / pooled_sd)


def _format_lr_pair_name(lr):
    try:
        lig, rec = lr[0], lr[1]
    except Exception:
        return str(lr)
    def _side(x):
        if isinstance(x, (list, tuple, set, np.ndarray)):
            return "+".join(map(str, x))
        return str(x)
    return f"{_side(lig)}|{_side(rec)}"


def _lr_pair_to_ligand_receptor(lr):
    lig, rec = lr[0], lr[1]
    def _first_gene(x):
        if isinstance(x, (list, tuple, set, np.ndarray)):
            return str(list(x)[0])
        return str(x)
    return _first_gene(lig), _first_gene(rec)


In [ ]:
# ==============================================================
# 16.4 GO-program gene sets from the Perturb-FISH MI-17 panel
# ============================================================== 

MI17_SENDER_GO_TERMS = [
    "Cellular response to cytokine stimulus",
    "Response to interleukin-1",
    "Cellular response to reactive oxygen species",
    "TGF-beta receptor signaling pathway",
    "Epithelial to mesenchymal transition",
    "Extracellular matrix organization",
]

MI17_RECEIVER_GO_TERMS = [
    "Negative regulation of apoptotic process",
    "Response to endoplasmic reticulum stress",
    "Cellular response to reactive oxygen species",
    "Integrin-mediated signaling pathway",
    "Heterotypic cell-cell adhesion",
    "Regulation of SMAD protein phosphorylation",
]



In [ ]:
GO_LIBRARY_NAME = "GO_Biological_Process_2021"


def _normalize_go_term_name(x):
    x = str(x)
    x = re.sub(r"\s*\(GO:\d+\)\s*$", "", x)
    x = x.replace("–", "-").replace("—", "-")
    x = re.sub(r"\s+", " ", x).strip().lower()
    return x


def _load_go_library():
    if gp is None:
        raise ImportError("gseapy is required to download GO gene sets.")

    organism_candidates = []
    if str(SPECIES).lower().startswith("human"):
        organism_candidates.extend(["Human", "human", "Mouse", "mouse"])
    else:
        organism_candidates.extend(["Mouse", "mouse", "Human", "human"])

    last_error = None
    for organism in organism_candidates:
        try:
            lib = gp.get_library(name=GO_LIBRARY_NAME, organism=organism)
            print(f"Loaded {GO_LIBRARY_NAME} for organism={organism}; n_terms={len(lib)}")
            return lib, organism
        except Exception as e:
            print(f"[Warning] Failed to load GO library for organism={organism}: {e}")
            last_error = e
    raise RuntimeError("Could not load GO gene sets via gseapy/Enrichr.") from last_error


def _find_go_library_key(term, go_library):
    target = _normalize_go_term_name(term)
    norm_to_key = {}
    for key in go_library:
        norm_to_key.setdefault(_normalize_go_term_name(key), key)
    if target in norm_to_key:
        return norm_to_key[target]

    aliases = {
        "tgf-beta receptor signaling pathway": [
            "transforming growth factor beta receptor signaling pathway",
            "tgf beta receptor signaling pathway",
        ],
        "integrin-mediated signaling pathway": ["integrin mediated signaling pathway"],
        "heterotypic cell-cell adhesion": [
            "heterotypic cell cell adhesion",
            "heterotypic cell-cell adhesion via plasma membrane cell adhesion molecules",
        ],
    }
    for alias in aliases.get(target, []):
        alias_norm = _normalize_go_term_name(alias)
        if alias_norm in norm_to_key:
            return norm_to_key[alias_norm]

    # substring fallback
    for norm, key in norm_to_key.items():
        if target in norm or norm in target:
            return key
    return None


def _build_go_gene_dict(terms, go_library, gene_universe):
    gene_universe = pd.Index([str(g) for g in gene_universe])
    upper_to_actual = {}
    for g in gene_universe:
        upper_to_actual.setdefault(g.upper(), g)

    out = {}
    rows = []
    for term in terms:
        key = _find_go_library_key(term, go_library)
        if key is None:
            print(f"[Warning] GO term not found in library: {term}")
            genes_raw = []
        else:
            genes_raw = [str(g) for g in go_library[key]]

        genes_in = []
        for g in genes_raw:
            actual = upper_to_actual.get(g.upper())
            if actual is not None and actual not in genes_in:
                genes_in.append(actual)
        out[term] = genes_in
        rows.append({
            "GO_program": term,
            "GO_library_key": key,
            "n_genes_library": len(genes_raw),
            "n_genes_in_data": len(genes_in),
            "genes_in_data": ";".join(genes_in),
        })
    return out, pd.DataFrame(rows)


go_library, go_organism_used = _load_go_library()
gene_universe = processed.adata_list[0].var_names.astype(str)

sender_go_gene_dict, sender_go_gene_summary = _build_go_gene_dict(MI17_SENDER_GO_TERMS, go_library, gene_universe)
receiver_go_gene_dict, receiver_go_gene_summary = _build_go_gene_dict(MI17_RECEIVER_GO_TERMS, go_library, gene_universe)

sender_go_gene_summary.to_csv(BENCHMARK_OUTDIR / f"{MIOI}_sender_GO_gene_sets.csv", index=False)
receiver_go_gene_summary.to_csv(BENCHMARK_OUTDIR / f"{MIOI}_receiver_GO_gene_sets.csv", index=False)

print("Sender GO gene sets:")
display(sender_go_gene_summary)
print("Receiver GO gene sets:")
display(receiver_go_gene_summary)


In [ ]:
# ==============================================================
# 16.5 Build common cell metadata, edge metadata, and GO module scores
# ============================================================== 

# PerturbFISH target-gene annotation helpers.
# We restrict later communication-strength aggregation to melanoma→T-cell edges
# whose melanoma sender cell is annotated as one of PERTURB_GENES_OF_INTEREST.

PERTURB_GENE_SET_UPPER = {str(g).upper() for g in PERTURB_GENES_OF_INTEREST}
CONTROL_PERTURB_KEYWORDS_UPPER = [str(x).upper() for x in CONTROL_PERTURB_KEYWORDS]


def _is_control_unperturbed_annotation(x):
    if pd.isna(x):
        return False
    s = str(x).strip()
    if len(s) == 0 or s.lower() in {"nan", "none", "null", "na"}:
        return False
    s_upper = s.upper()

    # Exact or substring keyword matching for control/non-targeting annotations.
    for kw in CONTROL_PERTURB_KEYWORDS_UPPER:
        if kw in s_upper:
            return True

    # A few compact negative-control tokens are common in perturbation datasets.
    tokens = re.split(r"[^A-Za-z0-9]+", s_upper)
    if any(tok in {"NT", "NTC", "CTRL", "CONTROL"} for tok in tokens):
        return True
    return False


def _normalize_perturb_gene_token(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if len(s) == 0 or s.lower() in {"nan", "none", "null", "na"}:
        return np.nan

    s_upper = s.upper()

    # Exact match first.
    if s_upper in PERTURB_GENE_SET_UPPER:
        return s_upper

    # Split common multi-token annotations, e.g. "IRAK1-1", "KO_IRAK1", "IRAK1;sg1".
    tokens = re.split(r"[^A-Za-z0-9]+", s_upper)
    for token in tokens:
        if token in PERTURB_GENE_SET_UPPER:
            return token

    # Last-resort substring match with boundaries where possible.
    for gene in PERTURB_GENES_OF_INTEREST:
        gene_u = str(gene).upper()
        if re.search(rf"(^|[^A-Z0-9]){re.escape(gene_u)}([^A-Z0-9]|$)", s_upper):
            return gene_u

    return np.nan


def _score_perturb_gene_column(values):
    mapped = pd.Series(values).map(_normalize_perturb_gene_token)
    n_match = int(mapped.notna().sum())
    n_unique_match = int(mapped.dropna().nunique())
    return n_match, n_unique_match, mapped


def _infer_perturb_gene_column_for_adata(adata_cur, batch_index=None):
    """Infer the adata.obs column containing perturb-gene annotations.

    The score prioritizes columns that match many cells to the 11 requested
    perturb genes and contain several distinct matched genes.
    """
    if PERTURB_GENE_OBS_COL is not None:
        if PERTURB_GENE_OBS_COL not in adata_cur.obs.columns:
            raise KeyError(
                f"PERTURB_GENE_OBS_COL={PERTURB_GENE_OBS_COL!r} is not in adata.obs columns. "
                f"Available columns: {list(adata_cur.obs.columns)}"
            )
        mapped = pd.Series(adata_cur.obs[PERTURB_GENE_OBS_COL]).map(_normalize_perturb_gene_token)
        return PERTURB_GENE_OBS_COL, mapped

    preferred_name_tokens = [
        "perturb", "target", "gene", "guide", "grna", "sgrna", "ko", "crispr",
        "condition", "annotation", "sample", "sample_name",
    ]

    candidates = []
    for col in adata_cur.obs.columns:
        vals = adata_cur.obs[col]
        if vals.dtype.kind in "ifc" and vals.nunique(dropna=True) > 50:
            continue
        n_match, n_unique_match, mapped = _score_perturb_gene_column(vals)
        if n_match == 0:
            continue
        col_lower = str(col).lower()
        name_bonus = sum(tok in col_lower for tok in preferred_name_tokens)
        # Prefer many matched cells, multiple genes, and biologically named columns.
        score = (n_match, n_unique_match, name_bonus)
        candidates.append((score, col, mapped))

    if len(candidates) == 0:
        prefix = f"batch {batch_index}: " if batch_index is not None else ""
        raise ValueError(
            f"{prefix}Could not infer a perturb-gene annotation column matching "
            f"PERTURB_GENES_OF_INTEREST={PERTURB_GENES_OF_INTEREST}. "
            f"Please set PERTURB_GENE_OBS_COL manually in Section 16.1. "
            f"Available obs columns: {list(adata_cur.obs.columns)}"
        )

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)
    best_score, best_col, best_mapped = candidates[0]
    print(
        f"batch {batch_index}: perturb-gene column = {best_col!r}; "
        f"matched_cells={best_score[0]}, matched_genes={best_score[1]}"
    )
    return best_col, best_mapped


cell_meta_parts = []
edge_meta_parts = []
expr_parts = []
perturb_column_rows = []

global_cell_offset = 0
edge_global_offset = 0

for batch_index, (adata_cur, data_cur, factor_cur) in enumerate(
    zip(processed.adata_list, processed.spidernet_data, spidernet_factor_list)
):
    factor_cur = np.asarray(factor_cur, dtype=float)
    edge_index_raw = _get_edge_index(data_cur)
    edge_index_cur = _align_edge_index_to_factor_rows(
        edge_index_raw,
        n_factor_rows=factor_cur.shape[0],
        slice_index=batch_index,
    )

    n_cells = adata_cur.n_obs
    celltype_values = adata_cur.obs[CELLTYPE_COL].astype(str).to_numpy()
    sample_name = str(adata_cur.obs["sample_name"].iloc[0]) if "sample_name" in adata_cur.obs.columns else f"batch{batch_index}"

    perturb_col, perturb_gene_mapped = _infer_perturb_gene_column_for_adata(adata_cur, batch_index=batch_index)
    perturb_annotation_raw = adata_cur.obs[perturb_col].astype("object").where(adata_cur.obs[perturb_col].notna(), "").astype(str).to_numpy(dtype=object)
    perturb_gene_values = pd.Series(perturb_gene_mapped).astype("object").where(pd.Series(perturb_gene_mapped).notna(), "").to_numpy(dtype=object)
    is_target_perturb_gene = pd.Series(perturb_gene_mapped).notna().to_numpy()
    is_control_unperturbed = pd.Series(perturb_annotation_raw).map(_is_control_unperturbed_annotation).to_numpy(dtype=bool)
    perturb_column_rows.append({
        "batch_index": batch_index,
        "sample_name": sample_name,
        "perturb_gene_obs_col": perturb_col,
        "n_cells": int(n_cells),
        "n_cells_in_target_perturb_genes": int(is_target_perturb_gene.sum()),
        "n_cells_in_control_unperturbed": int(is_control_unperturbed.sum()),
        "target_perturb_genes_detected": ";".join(sorted(pd.Series(perturb_gene_mapped).dropna().unique().tolist())),
    })

    is_sender_melanoma = _match_keywords(celltype_values, SENDER_CELLTYPE_KEYWORDS)
    is_receiver_tcell = _match_keywords(celltype_values, RECEIVER_CELLTYPE_KEYWORDS)
    is_target_melanoma = is_sender_melanoma & is_target_perturb_gene

    cell_meta_cur = pd.DataFrame({
        "batch_index": batch_index,
        "local_cell_index": np.arange(n_cells, dtype=int),
        "global_cell_index": np.arange(global_cell_offset, global_cell_offset + n_cells, dtype=int),
        "barcode": adata_cur.obs_names.astype(str).to_numpy(),
        "sample_name": sample_name,
        "celltype": celltype_values,
        "perturb_annotation_raw": perturb_annotation_raw,
        "perturb_gene_annotation": perturb_gene_values,
        "is_target_perturb_gene": is_target_perturb_gene,
        "is_control_unperturbed": is_control_unperturbed,
        "is_sender_melanoma": is_sender_melanoma,
        "is_receiver_Tcell": is_receiver_tcell,
        "is_target_perturb_melanoma": is_target_melanoma,
    })
    cell_meta_parts.append(cell_meta_cur)

    sender_local = edge_index_cur[:, 0].astype(int)
    receiver_local = edge_index_cur[:, 1].astype(int)
    sender_is_melanoma = is_sender_melanoma[sender_local]
    receiver_is_tcell = is_receiver_tcell[receiver_local]
    sender_is_target_perturb = is_target_perturb_gene[sender_local]
    receiver_is_target_perturb = is_target_perturb_gene[receiver_local]
    sender_is_control_unperturbed = is_control_unperturbed[sender_local]
    receiver_is_control_unperturbed = is_control_unperturbed[receiver_local]

    edge_meta_cur = pd.DataFrame({
        "batch_index": batch_index,
        "local_edge_index": np.arange(edge_index_cur.shape[0], dtype=int),
        "global_edge_index": np.arange(edge_global_offset, edge_global_offset + edge_index_cur.shape[0], dtype=int),
        "sender_local": sender_local,
        "receiver_local": receiver_local,
        "sender_global": sender_local + global_cell_offset,
        "receiver_global": receiver_local + global_cell_offset,
        "sender_celltype": celltype_values[sender_local],
        "receiver_celltype": celltype_values[receiver_local],
        "sender_perturb_annotation_raw": perturb_annotation_raw[sender_local],
        "receiver_perturb_annotation_raw": perturb_annotation_raw[receiver_local],
        "sender_perturb_gene": perturb_gene_values[sender_local],
        "receiver_perturb_gene": perturb_gene_values[receiver_local],
        "sender_is_target_perturb_gene": sender_is_target_perturb,
        "receiver_is_target_perturb_gene": receiver_is_target_perturb,
        "sender_is_control_unperturbed": sender_is_control_unperturbed,
        "receiver_is_control_unperturbed": receiver_is_control_unperturbed,
        "sender_is_target_perturb_melanoma": sender_is_melanoma & sender_is_target_perturb,
        "receiver_is_target_perturb_melanoma": _match_keywords(celltype_values[receiver_local], SENDER_CELLTYPE_KEYWORDS) & receiver_is_target_perturb,
        "sender_is_control_melanoma": sender_is_melanoma & sender_is_control_unperturbed,
        "is_melanoma_to_Tcell": sender_is_melanoma & receiver_is_tcell,
        MELANOMA_PERTURB_EDGE_FILTER_COL: sender_is_melanoma & receiver_is_tcell & sender_is_target_perturb,
        CONTROL_MELANOMA_TO_TCELL_EDGE_COL: sender_is_melanoma & receiver_is_tcell & sender_is_control_unperturbed,
    })
    edge_meta_parts.append(edge_meta_cur)

    expr_parts.append(_get_dense_X(adata_cur))

    global_cell_offset += n_cells
    edge_global_offset += edge_index_cur.shape[0]

cell_meta_df = pd.concat(cell_meta_parts, axis=0, ignore_index=True)
edge_meta_df = pd.concat(edge_meta_parts, axis=0, ignore_index=True)
X_all = np.vstack(expr_parts).astype(np.float32, copy=False)
gene_names = pd.Index(processed.adata_list[0].var_names.astype(str))
perturb_gene_column_df = pd.DataFrame(perturb_column_rows)

print("Cells:", cell_meta_df.shape)
print("Edges:", edge_meta_df.shape)
print("Melanoma→T edges:", int(edge_meta_df["is_melanoma_to_Tcell"].sum()))
print(
    "Melanoma→T edges with sender melanoma annotated in target perturb genes:",
    int(edge_meta_df[MELANOMA_PERTURB_EDGE_FILTER_COL].sum()),
)
if CONTROL_MELANOMA_TO_TCELL_EDGE_COL in edge_meta_df.columns:
    print(
        "Control/unperturbed melanoma→T edges:",
        int(edge_meta_df[CONTROL_MELANOMA_TO_TCELL_EDGE_COL].sum()),
    )
print("Target perturb genes:", PERTURB_GENES_OF_INTEREST)
display(perturb_gene_column_df)

cell_meta_df.to_csv(BENCHMARK_OUTDIR / "PerturbFISH_cell_metadata_for_CCC_SMD.csv", index=False)
edge_meta_df.to_csv(BENCHMARK_OUTDIR / "PerturbFISH_edge_metadata_for_CCC_SMD.csv", index=False)
perturb_gene_column_df.to_csv(BENCHMARK_OUTDIR / "PerturbFISH_perturb_gene_column_detection_summary.csv", index=False)
pd.DataFrame({"perturb_gene": PERTURB_GENES_OF_INTEREST}).to_csv(
    BENCHMARK_OUTDIR / "PerturbFISH_target_perturb_genes_for_edge_filter.csv",
    index=False,
)


def _compute_global_z_module_scores(X, gene_names, go_gene_dict, prefix):
    gene_to_idx = {str(g): i for i, g in enumerate(gene_names)}
    records = {}
    summary_rows = []
    for term, genes in go_gene_dict.items():
        idx = [gene_to_idx[g] for g in genes if g in gene_to_idx]
        if len(idx) == 0:
            records[term] = np.full(X.shape[0], np.nan, dtype=np.float32)
            summary_rows.append({"GO_program": term, "n_genes_used": 0})
            continue
        X_sub = X[:, idx].astype(float, copy=False)
        mu = np.nanmean(X_sub, axis=0)
        sd = np.nanstd(X_sub, axis=0, ddof=1)
        sd[(~np.isfinite(sd)) | (sd == 0)] = np.nan
        Z = (X_sub - mu[None, :]) / sd[None, :]
        Z = np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0)
        records[term] = np.nanmean(Z, axis=1).astype(np.float32)
        summary_rows.append({"GO_program": term, "n_genes_used": len(idx), "genes_used": ";".join([gene_names[i] for i in idx])})
    out = pd.DataFrame(records)
    out.insert(0, "global_cell_index", cell_meta_df["global_cell_index"].to_numpy())
    pd.DataFrame(summary_rows).to_csv(BENCHMARK_OUTDIR / f"{prefix}_module_score_gene_summary.csv", index=False)
    out.to_csv(BENCHMARK_OUTDIR / f"{prefix}_GO_module_scores.csv", index=False)
    return out

sender_module_scores = _compute_global_z_module_scores(X_all, gene_names, sender_go_gene_dict, "sender")
receiver_module_scores = _compute_global_z_module_scores(X_all, gene_names, receiver_go_gene_dict, "receiver")

print("Sender module scores:", sender_module_scores.shape)
print("Receiver module scores:", receiver_module_scores.shape)


In [ ]:
# ==============================================================
# 16.6 SpiderNet and NMF-LR edge scores
# ============================================================== 

def _collect_spidernet_scores():
    parts = []
    for batch_index, factor_cur in enumerate(spidernet_factor_list):
        factor_cur = np.asarray(factor_cur, dtype=float)
        cols = [f"MI{i+1}" for i in range(factor_cur.shape[1])]
        df = pd.DataFrame(factor_cur, columns=cols)
        df.insert(0, "batch_index", batch_index)
        df.insert(1, "local_edge_index", np.arange(factor_cur.shape[0], dtype=int))
        parts.append(df)
    score_df = pd.concat(parts, axis=0, ignore_index=True)
    score_df = score_df.merge(edge_meta_df[["batch_index", "local_edge_index", "global_edge_index"]], on=["batch_index", "local_edge_index"], how="left")
    return score_df

spidernet_edge_scores = _collect_spidernet_scores()
spidernet_edge_scores.to_csv(BENCHMARK_OUTDIR / "SpiderNet_edge_scores.csv", index=False)
print("SpiderNet edge scores:", spidernet_edge_scores.shape)


def _extract_cellpair_lr_matrix(data_cur, n_cell_edges, n_lr, slice_index=None):
    if isinstance(data_cur, dict):
        keys = list(data_cur.keys())
    else:
        keys = dir(data_cur)
    candidate_key = None
    for key in ["cellpair_LRpair_neigh", "cellpair_lr", "cellpair_LRpair", "cellpair_LR"]:
        if (isinstance(data_cur, dict) and key in data_cur) or (not isinstance(data_cur, dict) and hasattr(data_cur, key)):
            candidate_key = key
            break
    if candidate_key is None:
        raise KeyError(f"Cannot find LR coexpression matrix in data object. Available keys include: {keys[:20]}")

    X = _as_numpy(_get_data_field(data_cur, candidate_key))
    X = np.asarray(X)
    if X.ndim == 1:
        X = X[:, None]

    if X.shape[0] == n_cell_edges and X.shape[1] == n_lr:
        return X.astype(np.float32, copy=False)

    # Common flattened edge×LR vector layouts.
    if X.shape[1] == 1 and X.shape[0] == n_cell_edges * n_lr:
        vec = X[:, 0]
        # Try block layout: [all edges for LR1, all edges for LR2, ...]
        block = vec.reshape(n_lr, n_cell_edges).T
        # Try repeat layout: [LR1..LRk for edge1, LR1..LRk for edge2, ...]
        repeat = vec.reshape(n_cell_edges, n_lr)
        # Prefer repeat if rows have more nonzero diversity; otherwise block is also valid.
        # Users can inspect saved output if needed.
        print(f"slice {slice_index}: LR matrix was flattened; using edge-major reshape to ({n_cell_edges}, {n_lr}).")
        return repeat.astype(np.float32, copy=False)

    if X.shape[0] == n_cell_edges and X.shape[1] != n_lr:
        raise ValueError(f"slice {slice_index}: LR matrix rows match edges but columns={X.shape[1]} != n_lr={n_lr}")

    raise ValueError(f"slice {slice_index}: unsupported LR matrix shape {X.shape}; expected ({n_cell_edges}, {n_lr}).")


def _collect_lr_coexpression_matrix():
    mats = []
    for batch_index, data_cur in enumerate(processed.spidernet_data):
        n_edges = int(edge_meta_df.loc[edge_meta_df["batch_index"] == batch_index].shape[0])
        X_lr = _extract_cellpair_lr_matrix(data_cur, n_cell_edges=n_edges, n_lr=len(processed.lr_list), slice_index=batch_index)
        mats.append(X_lr)
    return np.vstack(mats).astype(np.float32, copy=False)

from run_benchmarks import cache_path
nmf_lr_scores_path = cache_path("Baseline_CCC_MI17_GO_SMD_melanoma_to_Tcell/NMF_LR/NMF_LR_edge_scores.csv")
nmf_lr_model_path = NMF_LR_OUTDIR / "NMF_LR_model.pkl"
nmf_lr_loading_path = NMF_LR_OUTDIR / "NMF_LR_loading.csv"

if nmf_lr_scores_path.exists():
    print("Loading existing NMF-LR scores:", nmf_lr_scores_path)
    nmf_lr_edge_scores = pd.read_csv(nmf_lr_scores_path)
else:
    lr_all = _collect_lr_coexpression_matrix()
    print("Fitting NMF-LR on LR coexpression matrix:", lr_all.shape)
    nmf_model = NMF(
        n_components=NMF_LR_N_COMPONENTS,
        init="nndsvda",
        random_state=RANDOM_SEED,
        max_iter=1000,
    )
    W = nmf_model.fit_transform(np.asarray(lr_all, dtype=float))
    W = _safe_colmax_normalize(W)
    nmf_cols = [f"NMF-LR-{i+1}" for i in range(W.shape[1])]
    nmf_lr_edge_scores = pd.DataFrame(W, columns=nmf_cols)
    nmf_lr_edge_scores.insert(0, "global_edge_index", edge_meta_df["global_edge_index"].to_numpy())
    nmf_lr_edge_scores.to_csv(nmf_lr_scores_path, index=False)

    lr_names = [_format_lr_pair_name(lr) for lr in processed.lr_list]
    pd.DataFrame(nmf_model.components_, columns=lr_names, index=nmf_cols).to_csv(nmf_lr_loading_path)
    with open(nmf_lr_model_path, "wb") as f:
        pickle.dump(nmf_model, f)
    print("Saved NMF-LR scores to:", nmf_lr_scores_path)

print("NMF-LR edge scores:", nmf_lr_edge_scores.shape)


In [ ]:
# ==============================================================
# 16.7 COMMOT edge scores
# --------------------------------------------------------------
# This cell runs COMMOT if outputs are absent. COMMOT can be slow.
# ============================================================== 

def _build_commot_lr_table_for_adata(adata_cur):
    genes = set(map(str, adata_cur.var_names))
    rows = []
    for idx, lr in enumerate(processed.lr_list):
        ligand, receptor = _lr_pair_to_ligand_receptor(lr)
        if ligand not in genes or receptor not in genes:
            continue
        # Use LR-pair-specific pathway labels so each LR pair can be used as an independent feature.
        rows.append([ligand, receptor, f"LR_{idx:04d}_{ligand}_{receptor}"])
    if len(rows) == 0:
        raise ValueError("No LR pairs are present in the current adata genes for COMMOT.")
    return pd.DataFrame(rows, columns=[0, 1, 2])


def _run_commot_one_batch(adata_cur, edge_index_cur, sample_name, outdir):
    try:
        import commot as ct
    except Exception as e:
        raise ImportError("commot is required to run COMMOT baseline. Install/import commot or provide precomputed outputs.") from e

    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    out_csv = outdir / f"{sample_name}_COMMOT_edge_scores.csv"
    if out_csv.exists():
        return pd.read_csv(out_csv)

    adata_commot = adata_cur.copy()
    if "spatial" not in adata_commot.obsm:
        # Try common spatial columns.
        for cols in [("x", "y"), ("X", "Y"), ("spatial_x", "spatial_y")]:
            if all(c in adata_commot.obs.columns for c in cols):
                adata_commot.obsm["spatial"] = adata_commot.obs[list(cols)].to_numpy(dtype=float)
                break
    if "spatial" not in adata_commot.obsm:
        raise KeyError("COMMOT requires adata.obsm['spatial'] or spatial coordinate columns.")

    df_ligrec = _build_commot_lr_table_for_adata(adata_commot)
    coords = np.asarray(adata_commot.obsm["spatial"], dtype=float)
    # Use the median distance of the existing kNN edge list as the COMMOT distance threshold.
    d = np.sqrt(np.sum((coords[edge_index_cur[:, 0]] - coords[edge_index_cur[:, 1]]) ** 2, axis=1))
    dis_thr = float(np.nanmedian(d[np.isfinite(d)]))
    print(f"Running COMMOT for {sample_name}; LR pairs={df_ligrec.shape[0]}, dis_thr={dis_thr:.4f}")

    ct.tl.spatial_communication(
        adata_commot,
        database_name="cellchat",
        df_ligrec=df_ligrec,
        dis_thr=dis_thr,
        heteromeric=True,
        pathway_sum=True,
        cot_nitermax=2000,
    )

    score_cols = []
    score_mat_parts = []
    for pathway in df_ligrec[2].astype(str).unique().tolist():
        key = f"commot-cellchat-{pathway}"
        if key not in adata_commot.obsp:
            continue
        mat = adata_commot.obsp[key]
        vals = mat[edge_index_cur[:, 0], edge_index_cur[:, 1]]
        vals = np.asarray(vals).reshape(-1)
        score_cols.append(pathway)
        score_mat_parts.append(vals.astype(np.float32))

    if len(score_mat_parts) == 0:
        raise RuntimeError(f"COMMOT finished for {sample_name}, but no edge-score matrices were found in adata.obsp.")

    X = np.vstack(score_mat_parts).T
    X = _safe_colmax_normalize(X)
    out = pd.DataFrame(X, columns=score_cols)
    out.insert(0, "local_edge_index", np.arange(edge_index_cur.shape[0], dtype=int))
    out.to_csv(out_csv, index=False)
    return out


from run_benchmarks import cache_path
commot_scores_path = cache_path("Baseline_CCC_MI17_GO_SMD_melanoma_to_Tcell/COMMOT/COMMOT_edge_scores.csv")
if commot_scores_path.exists():
    print("Loading existing COMMOT scores:", commot_scores_path)
    commot_edge_scores = pd.read_csv(commot_scores_path)
else:
    commot_parts = []
    for batch_index, adata_cur in enumerate(processed.adata_list):
        edge_cur = edge_meta_df.loc[edge_meta_df["batch_index"] == batch_index, ["sender_local", "receiver_local", "local_edge_index", "global_edge_index"]].copy()
        edge_index_cur = edge_cur[["sender_local", "receiver_local"]].to_numpy(dtype=int)
        sample_name = str(edge_meta_df.loc[edge_meta_df["batch_index"] == batch_index, "batch_index"].iloc[0])
        if "sample_name" in adata_cur.obs.columns:
            sample_name = str(adata_cur.obs["sample_name"].iloc[0])
        sample_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", sample_name)
        try:
            df_cur = _run_commot_one_batch(adata_cur, edge_index_cur, sample_name, COMMOT_OUTDIR / sample_name)
            df_cur = df_cur.merge(edge_cur[["local_edge_index", "global_edge_index"]], on="local_edge_index", how="left")
            df_cur.insert(0, "batch_index", batch_index)
            commot_parts.append(df_cur)
        except Exception as e:
            print(f"[Warning] COMMOT failed for batch {batch_index}: {repr(e)}")

    if len(commot_parts) == 0:
        print("[Warning] No COMMOT outputs were generated. COMMOT will be skipped.")
        commot_edge_scores = pd.DataFrame(columns=["global_edge_index"])
    else:
        # Align feature columns across batches; missing feature values are filled with zero.
        commot_edge_scores = pd.concat(commot_parts, axis=0, ignore_index=True).fillna(0.0)
        commot_edge_scores.to_csv(commot_scores_path, index=False)
        print("Saved COMMOT scores to:", commot_scores_path)

print("COMMOT edge scores:", commot_edge_scores.shape)


In [ ]:
# ==============================================================
# 16.8b Refresh ScCChain inputs and LR database
# --------------------------------------------------------------
# This retained compatibility export overwrites the h5ad inputs and supports
# compound LR entries before ScCChain_PerturbFISH_runner.jl is run.
# ==============================================================

def _sanitize_name_for_sccchain(x):
    x = str(x)
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x if x else "sample"


def _flatten_lr_side_for_sccchain(x):
    if isinstance(x, str):
        return x
    if isinstance(x, (list, tuple, np.ndarray, pd.Series)):
        vals = []
        for y in x:
            if isinstance(y, str):
                vals.append(y)
            elif isinstance(y, (list, tuple, np.ndarray, pd.Series)):
                vals.extend([str(z) for z in y])
            else:
                vals.append(str(y))
        vals = [v for v in vals if len(str(v).strip()) > 0]
        return ";".join(vals)
    return str(x)


def _lr_pair_to_ligand_receptor_for_sccchain(lr):
    if isinstance(lr, (list, tuple, np.ndarray, pd.Series)) and len(lr) >= 2:
        ligand = _flatten_lr_side_for_sccchain(lr[0])
        receptor = _flatten_lr_side_for_sccchain(lr[1])
        return ligand, receptor

    lr_str = str(lr)
    for sep in ["—", "-", "_", "|", ":", "~"]:
        if sep in lr_str:
            parts = lr_str.split(sep)
            if len(parts) >= 2:
                return parts[0].strip(), parts[1].strip()

    raise ValueError(f"Cannot parse LR pair: {lr}")


# Make sure output dirs exist
BENCHMARK_OUTDIR.mkdir(parents=True, exist_ok=True)
SCCCHAIN_INPUT_DIR.mkdir(parents=True, exist_ok=True)
SCCCHAIN_OUTDIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1. Export h5ad files
# -----------------------------
manifest_rows = []

for batch_index, adata_cur in enumerate(processed.adata_list):
    sample_name = f"batch{batch_index}"

    if "sample_name" in adata_cur.obs.columns:
        sample_name = str(adata_cur.obs["sample_name"].iloc[0])
    elif "sample" in adata_cur.obs.columns:
        sample_name = str(adata_cur.obs["sample"].iloc[0])

    sample_name = _sanitize_name_for_sccchain(sample_name)
    h5ad_path = SCCCHAIN_INPUT_DIR / f"{sample_name}.h5ad"

    adata_export = adata_cur.copy()
    adata_export.var["Gene_name"] = adata_export.var_names.astype(str)

    if "spatial" not in adata_export.obsm:
        for cols in [
            ("x", "y"),
            ("X", "Y"),
            ("spatial_x", "spatial_y"),
            ("array_row", "array_col"),
        ]:
            if all(c in adata_export.obs.columns for c in cols):
                adata_export.obsm["spatial"] = adata_export.obs[list(cols)].to_numpy(dtype=float)
                break

    if "spatial" not in adata_export.obsm:
        raise KeyError(
            f"Batch {batch_index} does not have adata.obsm['spatial'] "
            "or recognizable spatial coordinate columns."
        )

    adata_export.write_h5ad(h5ad_path)

    manifest_rows.append({
        "batch_index": int(batch_index),
        "sample_name": sample_name,
        "h5ad_path": str(h5ad_path),
    })

scchain_manifest_df = pd.DataFrame(manifest_rows)
scchain_manifest_path = BENCHMARK_OUTDIR / "ScCChain_input_manifest.csv"
scchain_manifest_df.to_csv(scchain_manifest_path, index=False)

# -----------------------------
# 2. Export LR database
# -----------------------------
lr_rows = []

for i, lr in enumerate(processed.lr_list):
    ligand, receptor = _lr_pair_to_ligand_receptor_for_sccchain(lr)
    lr_rows.append({
        "lr_index": int(i),
        "ligand": ligand,
        "receptor": receptor,
        "pathway": f"LR_{i:04d}",
    })

lr_db_path = BENCHMARK_OUTDIR / "PerturbFISH_lr_db_for_scCChain.csv"
pd.DataFrame(lr_rows).to_csv(lr_db_path, index=False)

# -----------------------------
# 3. Check files and print command
# -----------------------------
print("Saved ScCChain manifest:")
print(scchain_manifest_path)
print("Exists:", scchain_manifest_path.exists())

print("\nSaved ScCChain LR DB:")
print(lr_db_path)
print("Exists:", lr_db_path.exists())

print("\nManifest preview:")
display(scchain_manifest_df)

print("\nRun this in PowerShell:")
print(
    f'julia .\\ScCChain_PerturbFISH_runner.jl `\n'
    f'  --manifest "{scchain_manifest_path}" `\n'
    f'  --lr-db-csv "{lr_db_path}" `\n'
    f'  --result-root "{SCCCHAIN_OUTDIR}" `\n'
    f'  --n-programs {SCCCHAIN_N_PROGRAMS}'
)

In [ ]:
# ==============================================================
# 16.9 Load ScCChain edge-program outputs
# ============================================================== 
# Existing ScCChain results use this legacy layout rather than base_run_dir.
from run_benchmarks import SCCCHAIN_RESULTS
SCCCHAIN_OUTDIR = SCCCHAIN_RESULTS


print("Using SCCCHAIN_OUTDIR:", SCCCHAIN_OUTDIR)
print("Exists:", SCCCHAIN_OUTDIR.exists())
print("Sample1 exists:", (SCCCHAIN_OUTDIR / "Sample1").exists())
print("Sample1 files:", list((SCCCHAIN_OUTDIR / "Sample1").glob("*_ScCChain_edge_program_scores.csv")))

def _find_sccchain_csv(sample_result_root):
    sample_result_root = Path(sample_result_root)
    files = sorted(sample_result_root.glob("*_ScCChain_edge_program_scores.csv"))
    if len(files) == 0:
        return None
    return files[0]


def _load_sccchain_scores_for_batch(batch_index, sample_name, edge_index_cur):
    sample_dir = SCCCHAIN_OUTDIR / sample_name
    csv_path = _find_sccchain_csv(sample_dir)
    if csv_path is None:
        print(f"[Warning] Missing ScCChain output for batch {batch_index}, sample={sample_name}: {sample_dir}")
        return None
    sc_df = pd.read_csv(csv_path)
    if sc_df.shape[1] < 3:
        raise ValueError(f"Unexpected ScCChain output format: {csv_path}")
    sender_col, receiver_col = sc_df.columns[:2]
    program_cols = sc_df.columns[2:].tolist()
    sc_df = sc_df.rename(columns={sender_col: "sender_index", receiver_col: "receiver_index"}).copy()
    sc_df["sender_index"] = pd.to_numeric(sc_df["sender_index"], errors="coerce")
    sc_df["receiver_index"] = pd.to_numeric(sc_df["receiver_index"], errors="coerce")
    sc_df = sc_df.dropna(subset=["sender_index", "receiver_index"]).copy()
    sc_df["sender_index"] = sc_df["sender_index"].astype(int)
    sc_df["receiver_index"] = sc_df["receiver_index"].astype(int)

    # Julia ScCChain often returns 1-based indices. Convert if it looks 1-based.
    n_cells = processed.adata_list[batch_index].n_obs
    if sc_df[["sender_index", "receiver_index"]].min().min() >= 1 and sc_df[["sender_index", "receiver_index"]].max().max() <= n_cells:
        sc_df["sender_index"] -= 1
        sc_df["receiver_index"] -= 1

    ref_pairs = pd.DataFrame(edge_index_cur, columns=["sender_index", "receiver_index"])
    ref_index = pd.MultiIndex.from_frame(ref_pairs[["sender_index", "receiver_index"]])
    sc_indexed = sc_df.drop_duplicates(["sender_index", "receiver_index"], keep="first").set_index(["sender_index", "receiver_index"])[program_cols]
    X = sc_indexed.reindex(ref_index).fillna(0.0).to_numpy(dtype=np.float32, copy=False)
    X = _safe_colmax_normalize(X)
    out_cols = [f"ScCChain::{sample_name}::{c}" for c in program_cols]
    out = pd.DataFrame(X, columns=out_cols)
    out.insert(0, "local_edge_index", np.arange(edge_index_cur.shape[0], dtype=int))
    out.insert(0, "batch_index", batch_index)
    out = out.merge(edge_meta_df[["batch_index", "local_edge_index", "global_edge_index"]], on=["batch_index", "local_edge_index"], how="left")
    return out

sccchain_scores_path = SCCCHAIN_OUTDIR / "ScCChain_edge_scores_aligned.csv"
if sccchain_scores_path.exists():
    print("Loading existing aligned ScCChain scores:", sccchain_scores_path)
    sccchain_edge_scores = pd.read_csv(sccchain_scores_path)
else:
    scc_parts = []
    if 'scchain_manifest_df' not in globals():
        scchain_manifest_path = BENCHMARK_OUTDIR / "ScCChain_input_manifest.csv"
        scchain_manifest_df = pd.read_csv(scchain_manifest_path)
    for _, row in scchain_manifest_df.iterrows():
        batch_index = int(row["batch_index"])
        sample_name = str(row["sample_name"])
        edge_cur = edge_meta_df.loc[edge_meta_df["batch_index"] == batch_index, ["sender_local", "receiver_local"]].to_numpy(dtype=int)
        df_cur = _load_sccchain_scores_for_batch(batch_index, sample_name, edge_cur)
        if df_cur is not None:
            scc_parts.append(df_cur)
    if len(scc_parts) == 0:
        print("[Warning] No ScCChain outputs loaded. ScCChain will be skipped.")
        sccchain_edge_scores = pd.DataFrame(columns=["global_edge_index"])
    else:
        sccchain_edge_scores = pd.concat(scc_parts, axis=0, ignore_index=True).fillna(0.0)
        sccchain_edge_scores.to_csv(BENCHMARK_OUTDIR / "ScCChain" / "ScCChain_edge_scores_aligned.csv", index=False)
        print("Saved aligned ScCChain scores:", sccchain_scores_path)

print("ScCChain edge scores:", sccchain_edge_scores.shape)


In [ ]:
# ==============================================================
# 16.10 Compute SMD from perturbation-filtered melanoma→T-cell communication groups
# --------------------------------------------------------------
# Communication-strength aggregation and High/Low grouping for both sender
# melanoma cells and receiver T cells use only melanoma→T-cell edges whose
# sender is annotated with a gene in PERTURB_GENES_OF_INTEREST.
# ============================================================== 

def _edge_score_long_from_wide(method_name, edge_score_df, feature_cols=None):
    if edge_score_df is None or edge_score_df.shape[0] == 0:
        return pd.DataFrame(columns=["Method", "Feature", "global_edge_index", "score"])
    if feature_cols is None:
        feature_cols = [c for c in edge_score_df.columns if c not in {"batch_index", "local_edge_index", "global_edge_index"}]
    feature_cols = [c for c in feature_cols if c in edge_score_df.columns]
    if len(feature_cols) == 0:
        return pd.DataFrame(columns=["Method", "Feature", "global_edge_index", "score"])
    long_df = edge_score_df[["global_edge_index"] + feature_cols].melt(
        id_vars="global_edge_index",
        value_vars=feature_cols,
        var_name="Feature",
        value_name="score",
    )
    long_df["Method"] = method_name
    return long_df[["Method", "Feature", "global_edge_index", "score"]]


# Ensure the filtered edge column exists, even if users rerun this cell without rerunning Cell 5.
if MELANOMA_PERTURB_EDGE_FILTER_COL not in edge_meta_df.columns:
    required_cell_cols = {"global_cell_index", "is_sender_melanoma", "is_target_perturb_gene"}
    missing_cell_cols = required_cell_cols.difference(cell_meta_df.columns)
    if len(missing_cell_cols) > 0:
        raise KeyError(
            f"Missing {missing_cell_cols} in cell_meta_df. Please rerun Section 16.5 so perturb-gene annotations are added."
        )
    lookup_cols = ["is_sender_melanoma", "is_target_perturb_gene", "perturb_gene_annotation"]
    if "is_control_unperturbed" in cell_meta_df.columns:
        lookup_cols.append("is_control_unperturbed")
    if "perturb_annotation_raw" in cell_meta_df.columns:
        lookup_cols.append("perturb_annotation_raw")
    sender_lookup = cell_meta_df.set_index("global_cell_index")[lookup_cols]
    sender_info = sender_lookup.reindex(edge_meta_df["sender_global"].astype(int).values)
    edge_meta_df[MELANOMA_PERTURB_EDGE_FILTER_COL] = (
        edge_meta_df["is_melanoma_to_Tcell"].to_numpy(dtype=bool)
        & sender_info["is_sender_melanoma"].fillna(False).to_numpy(dtype=bool)
        & sender_info["is_target_perturb_gene"].fillna(False).to_numpy(dtype=bool)
    )
    edge_meta_df["sender_perturb_gene"] = sender_info["perturb_gene_annotation"].fillna("").to_numpy()
    if "is_control_unperturbed" in sender_info.columns:
        edge_meta_df["sender_is_control_unperturbed"] = sender_info["is_control_unperturbed"].fillna(False).to_numpy(dtype=bool)
        edge_meta_df[CONTROL_MELANOMA_TO_TCELL_EDGE_COL] = (
            edge_meta_df["is_melanoma_to_Tcell"].to_numpy(dtype=bool)
            & sender_info["is_sender_melanoma"].fillna(False).to_numpy(dtype=bool)
            & sender_info["is_control_unperturbed"].fillna(False).to_numpy(dtype=bool)
        )
    if "perturb_annotation_raw" in sender_info.columns:
        edge_meta_df["sender_perturb_annotation_raw"] = sender_info["perturb_annotation_raw"].fillna("").to_numpy()

# SpiderNet: fix to MI17 for the main comparison.
spidernet_feature_col = f"MI{MIOI_INDEX + 1}"
method_long_parts = [
    _edge_score_long_from_wide("SpiderNet", spidernet_edge_scores, feature_cols=[spidernet_feature_col]),
    _edge_score_long_from_wide("NMF-LR", nmf_lr_edge_scores),
]

if commot_edge_scores is not None and commot_edge_scores.shape[0] > 0:
    method_long_parts.append(_edge_score_long_from_wide("COMMOT", commot_edge_scores))
if sccchain_edge_scores is not None and sccchain_edge_scores.shape[0] > 0:
    method_long_parts.append(_edge_score_long_from_wide("ScCChain", sccchain_edge_scores))

edge_score_long_all = pd.concat(method_long_parts, axis=0, ignore_index=True)
edge_score_long_all["score"] = pd.to_numeric(edge_score_long_all["score"], errors="coerce").fillna(0.0)

edge_annotation_cols = [
    "global_edge_index", "sender_global", "receiver_global", "is_melanoma_to_Tcell",
    MELANOMA_PERTURB_EDGE_FILTER_COL, "sender_celltype", "receiver_celltype", "batch_index",
]
for optional_col in [
    "sender_perturb_gene", "receiver_perturb_gene",
    "sender_perturb_annotation_raw", "receiver_perturb_annotation_raw",
    "sender_is_target_perturb_gene", "receiver_is_target_perturb_gene",
    "sender_is_control_unperturbed", "receiver_is_control_unperturbed",
    CONTROL_MELANOMA_TO_TCELL_EDGE_COL,
    "sender_is_target_perturb_melanoma",
]:
    if optional_col in edge_meta_df.columns:
        edge_annotation_cols.append(optional_col)
edge_annotation_cols = list(dict.fromkeys(edge_annotation_cols))

edge_score_long_all = edge_score_long_all.merge(
    edge_meta_df[edge_annotation_cols],
    on="global_edge_index",
    how="left",
)

edge_score_long_all.to_csv(
    BENCHMARK_OUTDIR / "melanoma_to_Tcell_edge_scores_all_methods_long_unfiltered.csv",
    index=False,
)

# Restrict to melanoma→T-cell edges whose melanoma sender is one of the 11 target perturb genes.
edge_score_long = edge_score_long_all[edge_score_long_all[MELANOMA_PERTURB_EDGE_FILTER_COL].fillna(False)].copy()

print("All melanoma→T edge score rows:", edge_score_long_all[edge_score_long_all["is_melanoma_to_Tcell"].fillna(False)].shape)
print(
    "Target-perturb-gene melanoma→T edge score rows used for aggregation/grouping:",
    edge_score_long.shape,
)
print("Target perturb genes used:", PERTURB_GENES_OF_INTEREST)

if edge_score_long.shape[0] == 0:
    raise ValueError(
        "No target-perturb-gene melanoma→T-cell edges were found. Check PERTURB_GENE_OBS_COL, "
        "PERTURB_GENES_OF_INTEREST, and cell-type matching keywords."
    )

edge_score_long.to_csv(
    BENCHMARK_OUTDIR / "targetPerturb_melanoma_to_Tcell_edge_scores_all_methods_long.csv",
    index=False,
)

# Useful diagnostic: how many filtered edges per perturb gene.
if "sender_perturb_gene" in edge_score_long.columns:
    edge_filter_summary = (
        edge_score_long[["global_edge_index", "sender_perturb_gene", "batch_index"]]
        .drop_duplicates()
        .groupby(["sender_perturb_gene", "batch_index"], as_index=False)
        .size()
        .rename(columns={"size": "n_edges"})
    )
    edge_filter_summary.to_csv(
        BENCHMARK_OUTDIR / "targetPerturb_melanoma_to_Tcell_edge_filter_summary_by_gene_batch.csv",
        index=False,
    )
    display(edge_filter_summary)


def _aggregate_edge_scores_to_cells(edge_score_long, role):
    if role == "sender":
        idx_col = "sender_global"
    elif role == "receiver":
        idx_col = "receiver_global"
    else:
        raise ValueError(role)

    if EDGE_TO_CELL_AGG == "max":
        agg = edge_score_long.groupby(["Method", "Feature", idx_col], as_index=False)["score"].max()
    elif EDGE_TO_CELL_AGG == "mean":
        agg = edge_score_long.groupby(["Method", "Feature", idx_col], as_index=False)["score"].mean()
    elif EDGE_TO_CELL_AGG == "sum":
        agg = edge_score_long.groupby(["Method", "Feature", idx_col], as_index=False)["score"].sum()
    else:
        raise ValueError(f"Unsupported EDGE_TO_CELL_AGG: {EDGE_TO_CELL_AGG}")
    agg = agg.rename(columns={idx_col: "global_cell_index", "score": "communication_strength"})
    agg["Role"] = role
    agg["edge_filter"] = MELANOMA_PERTURB_EDGE_FILTER_COL
    agg["edge_to_cell_agg"] = EDGE_TO_CELL_AGG
    return agg

# Both sides are grouped from the same target-perturb-gene-filtered melanoma→T-cell edge set.
sender_cell_comm = _aggregate_edge_scores_to_cells(edge_score_long, role="sender")
receiver_cell_comm = _aggregate_edge_scores_to_cells(edge_score_long, role="receiver")
cell_comm_long = pd.concat([sender_cell_comm, receiver_cell_comm], axis=0, ignore_index=True)
cell_comm_long.to_csv(
    BENCHMARK_OUTDIR / "cell_level_targetPerturb_melanoma_to_Tcell_communication_strength_long.csv",
    index=False,
)


def _make_high_low_groups(cell_comm_df):
    out_parts = []
    for (method, feature, role), df_cur in cell_comm_df.groupby(["Method", "Feature", "Role"], sort=False):
        df_cur = df_cur.copy()
        vals = pd.to_numeric(df_cur["communication_strength"], errors="coerce")
        valid = np.isfinite(vals) & (vals > 0)
        df_cur = df_cur.loc[valid].copy()
        if df_cur.shape[0] < 4 or df_cur["communication_strength"].nunique() < 2:
            continue
        threshold = float(np.nanmedian(df_cur["communication_strength"]))
        df_cur["CommGroup"] = np.where(df_cur["communication_strength"] > threshold, "High", "Low")
        # If too many ties at median cause empty High/Low, use qcut fallback.
        if df_cur["CommGroup"].nunique() < 2:
            try:
                df_cur["CommGroup"] = pd.qcut(df_cur["communication_strength"], q=2, labels=["Low", "High"], duplicates="drop").astype(str)
            except Exception:
                continue
        if df_cur["CommGroup"].nunique() >= 2:
            out_parts.append(df_cur)
    if len(out_parts) == 0:
        return pd.DataFrame()
    return pd.concat(out_parts, axis=0, ignore_index=True)

grouped_cell_comm = _make_high_low_groups(cell_comm_long)
grouped_cell_comm.to_csv(
    BENCHMARK_OUTDIR / "cell_level_targetPerturb_high_low_communication_groups_long.csv",
    index=False,
)
print("Grouped cells:", grouped_cell_comm.shape)


def _compute_smd_table_for_role(grouped_cell_comm, module_scores_df, go_terms, role):
    records = []
    role_df = grouped_cell_comm[grouped_cell_comm["Role"] == role].copy()
    module_scores_indexed = module_scores_df.set_index("global_cell_index")

    for (method, feature), df_cur in role_df.groupby(["Method", "Feature"], sort=False):
        cell_idx = df_cur["global_cell_index"].astype(int).to_numpy()
        group_map = df_cur.set_index("global_cell_index")["CommGroup"].to_dict()
        for term in go_terms:
            score_series = module_scores_indexed[term].reindex(cell_idx)
            tmp = pd.DataFrame({
                "global_cell_index": cell_idx,
                "module_score": score_series.to_numpy(dtype=float),
            })
            tmp["CommGroup"] = tmp["global_cell_index"].map(group_map)
            high_vals = tmp.loc[tmp["CommGroup"] == "High", "module_score"]
            low_vals = tmp.loc[tmp["CommGroup"] == "Low", "module_score"]
            smd = _pooled_smd(high_vals, low_vals)
            if len(high_vals.dropna()) > 0 and len(low_vals.dropna()) > 0:
                try:
                    p = mannwhitneyu(high_vals.dropna(), low_vals.dropna(), alternative="two-sided").pvalue
                except Exception:
                    p = np.nan
            else:
                p = np.nan
            records.append({
                "Role": role,
                "Method": method,
                "Feature": feature,
                "GO_program": term,
                "edge_filter": MELANOMA_PERTURB_EDGE_FILTER_COL,
                "edge_to_cell_agg": EDGE_TO_CELL_AGG,
                "n_high": int(high_vals.dropna().shape[0]),
                "n_low": int(low_vals.dropna().shape[0]),
                "mean_high": float(np.nanmean(high_vals)) if len(high_vals.dropna()) > 0 else np.nan,
                "mean_low": float(np.nanmean(low_vals)) if len(low_vals.dropna()) > 0 else np.nan,
                "SMD_high_minus_low": smd,
                "pvalue": p,
            })
    return pd.DataFrame(records)

sender_smd_by_feature = _compute_smd_table_for_role(grouped_cell_comm, sender_module_scores, MI17_SENDER_GO_TERMS, role="sender")
receiver_smd_by_feature = _compute_smd_table_for_role(grouped_cell_comm, receiver_module_scores, MI17_RECEIVER_GO_TERMS, role="receiver")
smd_by_feature = pd.concat([sender_smd_by_feature, receiver_smd_by_feature], axis=0, ignore_index=True)
smd_by_feature.to_csv(
    BENCHMARK_OUTDIR / "GO_program_SMD_by_method_feature_long_targetPerturbEdges.csv",
    index=False,
)
# Keep the original filename as a compatibility output, now containing target-perturb-gene-filtered results.
smd_by_feature.to_csv(BENCHMARK_OUTDIR / "GO_program_SMD_by_method_feature_long.csv", index=False)

# Perturbation-vs-control axis matching for cross-method comparison.
# --------------------------------------------------------------
# The selected baseline axis is no longer chosen using GO-program SMD.
# Instead, for each method/feature, we compare melanoma→T-cell edge scores in:
#   perturb-gene-specific melanoma sender edges  vs.  control/unperturbed melanoma sender edges.
# For each of the 11 perturb genes, we compute:
#   mean(score | sender melanoma perturbed by gene) - mean(score | control melanoma sender)
# Then each method selects the ONE axis with the largest mean change across all
# perturb genes. This produces a GO-independent matched axis, analogous to using
# the perturbation-induced SpiderNet MI change in the in-silico perturbation analysis.


def _get_axis_matching_edge_table(edge_score_long_all):
    required_cols = {"Method", "Feature", "score", "is_melanoma_to_Tcell", "sender_perturb_gene"}
    missing = required_cols.difference(edge_score_long_all.columns)
    if len(missing) > 0:
        raise KeyError(f"Missing columns for axis matching: {missing}")

    m2t = edge_score_long_all[edge_score_long_all["is_melanoma_to_Tcell"].fillna(False)].copy()
    if m2t.shape[0] == 0:
        raise ValueError("No melanoma→T-cell edge scores are available for axis matching.")

    # Preferred control: explicit control/unperturbed melanoma sender edges.
    if CONTROL_MELANOMA_TO_TCELL_EDGE_COL in m2t.columns:
        control_mask = m2t[CONTROL_MELANOMA_TO_TCELL_EDGE_COL].fillna(False).to_numpy(dtype=bool)
        control_definition = CONTROL_MELANOMA_TO_TCELL_EDGE_COL
    elif "sender_is_control_unperturbed" in m2t.columns:
        control_mask = m2t["sender_is_control_unperturbed"].fillna(False).to_numpy(dtype=bool)
        control_definition = "sender_is_control_unperturbed"
    else:
        control_mask = np.zeros(m2t.shape[0], dtype=bool)
        control_definition = "none_detected"

    # Robust fallback: use melanoma→T-cell edges whose melanoma sender is not one
    # of the target perturb genes. This is useful when the dataset annotation does
    # not explicitly label controls, but should be checked carefully.
    if control_mask.sum() == 0 and AXIS_MATCH_CONTROL_FALLBACK_TO_NON_TARGET_MELANOMA:
        if "sender_is_target_perturb_gene" in m2t.columns:
            control_mask = ~m2t["sender_is_target_perturb_gene"].fillna(False).to_numpy(dtype=bool)
        else:
            sender_gene_u = m2t["sender_perturb_gene"].astype(str).str.upper()
            control_mask = ~sender_gene_u.isin([str(g).upper() for g in PERTURB_GENES_OF_INTEREST]).to_numpy(dtype=bool)
        control_definition = "fallback_non_target_melanoma_to_Tcell_edges"
        print(
            "[Warning] No explicit control/unperturbed melanoma→T-cell edges were detected. "
            "Falling back to non-target-perturbation melanoma→T-cell edges as control."
        )

    if control_mask.sum() == 0:
        raise ValueError(
            "No control/unperturbed melanoma→T-cell edges were found for axis matching. "
            "Check CONTROL_PERTURB_KEYWORDS, PERTURB_GENE_OBS_COL, or set "
            "AXIS_MATCH_CONTROL_FALLBACK_TO_NON_TARGET_MELANOMA=True."
        )

    m2t["axis_match_is_control_edge"] = control_mask
    m2t["axis_match_control_definition"] = control_definition
    m2t["sender_perturb_gene_upper"] = m2t["sender_perturb_gene"].astype(str).str.upper()
    return m2t, control_definition


def _compute_perturb_vs_control_axis_changes(edge_score_long_all):
    axis_edge_df, control_definition = _get_axis_matching_edge_table(edge_score_long_all)

    control_df = axis_edge_df[axis_edge_df["axis_match_is_control_edge"]].copy()
    control_summary = (
        control_df
        .groupby(["Method", "Feature"], as_index=False, sort=False)
        .agg(
            control_mean_score=("score", "mean"),
            n_control_edge_score_rows=("score", "size"),
            n_control_unique_edges=("global_edge_index", "nunique"),
        )
    )

    rows = []
    target_gene_order = [str(g).upper() for g in PERTURB_GENES_OF_INTEREST]
    for gene_u in target_gene_order:
        gene_df = axis_edge_df[axis_edge_df["sender_perturb_gene_upper"] == gene_u].copy()
        gene_summary = (
            gene_df
            .groupby(["Method", "Feature"], as_index=False, sort=False)
            .agg(
                perturb_mean_score=("score", "mean"),
                n_perturb_edge_score_rows=("score", "size"),
                n_perturb_unique_edges=("global_edge_index", "nunique"),
            )
        )
        merged = gene_summary.merge(control_summary, on=["Method", "Feature"], how="left")
        merged["perturb_gene"] = gene_u
        merged["mean_score_change_perturb_minus_control"] = (
            merged["perturb_mean_score"] - merged["control_mean_score"]
        )
        merged["control_definition"] = control_definition
        rows.append(merged)

    if len(rows) == 0:
        axis_gene_change = pd.DataFrame()
    else:
        axis_gene_change = pd.concat(rows, axis=0, ignore_index=True)

    if axis_gene_change.shape[0] == 0:
        raise ValueError("No perturb-vs-control axis-change rows were generated.")

    axis_gene_change["mean_score_change_perturb_minus_control"] = pd.to_numeric(
        axis_gene_change["mean_score_change_perturb_minus_control"],
        errors="coerce",
    )

    axis_feature_summary = (
        axis_gene_change
        .dropna(subset=["mean_score_change_perturb_minus_control"])
        .groupby(["Method", "Feature"], as_index=False, sort=False)
        .agg(
            mean_change_across_perturb_genes=("mean_score_change_perturb_minus_control", "mean"),
            median_change_across_perturb_genes=("mean_score_change_perturb_minus_control", "median"),
            min_change_across_perturb_genes=("mean_score_change_perturb_minus_control", "min"),
            max_change_across_perturb_genes=("mean_score_change_perturb_minus_control", "max"),
            n_perturb_genes_with_valid_change=("perturb_gene", "nunique"),
            mean_perturb_score_across_genes=("perturb_mean_score", "mean"),
            mean_control_score_across_genes=("control_mean_score", "mean"),
            mean_n_perturb_unique_edges=("n_perturb_unique_edges", "mean"),
            mean_n_control_unique_edges=("n_control_unique_edges", "mean"),
        )
    )

    selected_rows = []
    for method, df_cur in axis_feature_summary.groupby("Method", sort=False):
        df_cur = df_cur.copy()
        if df_cur.shape[0] == 0:
            continue
        # Prefer axes evaluated for more perturb genes, then select the largest
        # mean perturb-vs-control increase across genes.
        df_cur = df_cur.sort_values(
            ["n_perturb_genes_with_valid_change", "mean_change_across_perturb_genes"],
            ascending=[False, False],
        )
        selected_rows.append(df_cur.iloc[0].to_dict())

    axis_match_selection = pd.DataFrame(selected_rows)
    if axis_match_selection.shape[0] > 0:
        axis_match_selection = axis_match_selection.rename(
            columns={"Feature": "Selected_match_Feature"}
        )
        axis_match_selection["axis_selection_rule"] = (
            "max mean perturb-vs-control melanoma→T-cell edge score change across perturb genes"
        )
        axis_match_selection["control_definition"] = control_definition

    return axis_gene_change, axis_feature_summary, axis_match_selection


axis_match_gene_change, axis_match_feature_summary, axis_match_selection = _compute_perturb_vs_control_axis_changes(
    edge_score_long_all=edge_score_long_all,
)

axis_match_gene_change.to_csv(
    BENCHMARK_OUTDIR / "axis_matching_perturb_vs_control_gene_level_changes.csv",
    index=False,
)
axis_match_feature_summary.to_csv(
    BENCHMARK_OUTDIR / "axis_matching_perturb_vs_control_feature_summary.csv",
    index=False,
)
axis_match_selection.to_csv(
    BENCHMARK_OUTDIR / "axis_matching_selected_axis_by_method.csv",
    index=False,
)

# Use the GO-independent matched axis for downstream SMD benchmark.
smd_valid = smd_by_feature.copy()
smd_valid["SMD_high_minus_low"] = pd.to_numeric(
    smd_valid["SMD_high_minus_low"],
    errors="coerce",
)

if axis_match_selection.shape[0] == 0:
    smd_max_summary = pd.DataFrame(columns=smd_valid.columns.tolist())
else:
    selected_key_df = axis_match_selection[["Method", "Selected_match_Feature"]].rename(
        columns={"Selected_match_Feature": "Feature"}
    )
    smd_max_summary = smd_valid.merge(
        selected_key_df.assign(_selected_axis=True),
        on=["Method", "Feature"],
        how="inner",
    ).copy()
    smd_max_summary = smd_max_summary.drop(columns=["_selected_axis"])

    smd_max_summary = smd_max_summary.merge(
        axis_match_selection,
        left_on=["Method", "Feature"],
        right_on=["Method", "Selected_match_Feature"],
        how="left",
        suffixes=("", "_axis_match"),
    )

# Save with explicit axis-matching names.
smd_max_summary.to_csv(
    BENCHMARK_OUTDIR / "GO_program_SMD_axisMatch_perturb_vs_control_summary_targetPerturbEdges.csv",
    index=False,
)

# Keep previous filenames as compatibility outputs, now containing perturb-vs-control axis-matched results.
smd_max_summary.to_csv(
    BENCHMARK_OUTDIR / "GO_program_SMD_shared_axis_summary_targetPerturbEdges.csv",
    index=False,
)
smd_max_summary.to_csv(
    BENCHMARK_OUTDIR / "GO_program_SMD_max_feature_summary_targetPerturbEdges.csv",
    index=False,
)
smd_max_summary.to_csv(
    BENCHMARK_OUTDIR / "GO_program_SMD_max_feature_summary.csv",
    index=False,
)

print("SMD by feature:", smd_by_feature.shape)
print("Axis-matching gene-level changes:", axis_match_gene_change.shape)
print("Axis-matching candidate summary:", axis_match_feature_summary.shape)
print("Selected matched axes:", axis_match_selection.shape)
print("Axis-matched SMD summary:", smd_max_summary.shape)
print("Selected matched axes by Method:")
display(axis_match_selection)
display(smd_max_summary.head())


In [ ]:
# ==============================================================
# 16.11 Plot sender and receiver SMD barplots
# ==============================================================

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})
sns.set_theme(style="white")

METHOD_ORDER = ["SpiderNet", "NMF-LR", "COMMOT", "ScCChain"]

# HGSOC_Malignantsubtype_analysis_V2-style fill/edge palette
METHOD_COLOR_DICT = {
    "SpiderNet": {"edge": "#9F3B38", "fill": "#E1B6A7"},
    "NMF-LR": {"edge": "#82CCE2", "fill": "#D4ECF1"},
    "COMMOT": {"edge": "#519384", "fill": "#B9CEC7"},
    "ScCChain": {"edge": "#636491", "fill": "#A6A2B9"},
}

DEFAULT_METHOD_COLOR = {"edge": "#4D4D4D", "fill": "#BDBDBD"}


def _plot_smd_barplot(summary_df, role, term_order, out_prefix, title):
    plot_df = summary_df[
        (summary_df["Role"] == role)
        & (summary_df["GO_program"].isin(term_order))
    ].copy()

    if plot_df.shape[0] == 0:
        print(f"[Warning] No SMD data for role={role}")
        return

    plot_df["GO_program"] = pd.Categorical(
        plot_df["GO_program"],
        categories=term_order,
        ordered=True,
    )

    method_order = [m for m in METHOD_ORDER if m in plot_df["Method"].unique()]
    plot_df["Method"] = pd.Categorical(
        plot_df["Method"],
        categories=method_order,
        ordered=True,
    )

    plot_df = plot_df.sort_values(["GO_program", "Method"]).copy()
    plot_df["SMD_high_minus_low"] = pd.to_numeric(
        plot_df["SMD_high_minus_low"],
        errors="coerce",
    )

    n_terms = len(term_order)
    n_methods = len(method_order)

    # Horizontal grouped barplot:
    #   y = GO program
    #   x = SMD
    fig_width = 6.4
    fig_height = max(4.2, 0.60 * n_terms + 1.2)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height), facecolor="white")
    ax.set_facecolor("white")

    y_base = np.arange(n_terms)

    group_height = 0.78
    inner_gap = 0.045
    bar_h = (group_height - inner_gap * max(n_methods - 1, 0)) / max(n_methods, 1)

    offsets = (
        -group_height / 2
        + (np.arange(n_methods) + 0.5) * bar_h
        + np.arange(n_methods) * inner_gap
    )

    for j, method in enumerate(method_order):
        method_df = (
            plot_df[plot_df["Method"] == method]
            .set_index("GO_program")
            .reindex(term_order)
        )

        vals = method_df["SMD_high_minus_low"].to_numpy(dtype=float)

        color_cfg = METHOD_COLOR_DICT.get(method, DEFAULT_METHOD_COLOR)

        ax.barh(
            y_base + offsets[j],
            vals,
            height=bar_h,
            facecolor=color_cfg["fill"],
            edgecolor=color_cfg["edge"],
            linewidth=1.0,
            label=method,
        )

    ax.axvline(0, color="#4D4D4D", linewidth=0.8, linestyle="--")

    ax.set_yticks(y_base)
    ax.set_yticklabels(term_order)
    ax.invert_yaxis()

    ax.set_title(title)
    ax.set_xlabel(
        "SMD of GO module score\n"
        "High vs Low melanoma→T-cell communication"
    )
    ax.set_ylabel("")

    ax.tick_params(axis="x", rotation=0)
    ax.tick_params(axis="y", rotation=0)

    sns.despine(ax=ax, top=True, right=True)

    legend_handles = [
        mpl.patches.Patch(
            facecolor=METHOD_COLOR_DICT.get(m, DEFAULT_METHOD_COLOR)["fill"],
            edgecolor=METHOD_COLOR_DICT.get(m, DEFAULT_METHOD_COLOR)["edge"],
            linewidth=1.0,
            label=m,
        )
        for m in method_order
    ]

    ax.legend(
        handles=legend_handles,
        title="Method",
        frameon=False,
        bbox_to_anchor=(1.01, 1.0),
        loc="upper left",
    )

    plt.tight_layout()

    out_pdf = BENCHMARK_OUTDIR / f"{out_prefix}.pdf"
    out_png = BENCHMARK_OUTDIR / f"{out_prefix}.png"

    fig.savefig(out_pdf, bbox_inches="tight", facecolor="white")
    fig.savefig(out_png, bbox_inches="tight", facecolor="white", dpi=300)

    plt.show()
    plt.close(fig)

    print("Saved:", out_pdf)


_plot_smd_barplot(
    smd_max_summary,
    role="sender",
    term_order=MI17_SENDER_GO_TERMS,
    out_prefix=f"{MIOI}_sender_GO_program_SMD_axisMatch_perturb_vs_control_barplot",
    title=f"{MIOI} melanoma sender GO programs: perturb-vs-control matched-axis SMD",
)

_plot_smd_barplot(
    smd_max_summary,
    role="receiver",
    term_order=MI17_RECEIVER_GO_TERMS,
    out_prefix=f"{MIOI}_receiver_GO_program_SMD_axisMatch_perturb_vs_control_barplot",
    title=f"{MIOI} T-cell receiver GO programs: perturb-vs-control matched-axis SMD",
)